# Name That LLM — StyleDistance V7 + Prompt Assignment V8

V7 evidence is computed unchanged. V8 then recovers prompt groups from test text (TF-IDF, average-linkage cosine clustering, groups capped at 18) and assigns models one-to-one within each group by minimum-cost matching, because every training prompt was answered once by each of the 18 models.

Prompt-group development: 40.4509% → 52.4897%; related-prompt check: 40.8588% → 52.3791%. Both reuse inspected data, the selection was adaptive, and the gain assumes the hidden test shares the per-prompt structure. Predictions are transductive.

CPU; internet disabled.

In [ ]:
import os
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','LOKY_MAX_CPU_COUNT'):
    os.environ[key]='2'
for key in ('HF_HUB_OFFLINE','TRANSFORMERS_OFFLINE','HF_HUB_DISABLE_TELEMETRY','HF_HUB_DISABLE_IMPLICIT_TOKEN'):
    os.environ[key]='1'
os.environ['TOKENIZERS_PARALLELISM']='false'


In [ ]:
import json
from pathlib import Path
import sys
sources = json.loads('{"__init__.py": "\\"\\"\\"Reproducible baselines for Name That LLM.\\"\\"\\"\\n", "baseline.py": "\\"\\"\\"Shared training and submission code, also embedded in the Kaggle notebook.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport logging\\nimport platform\\nimport time\\nimport warnings\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport scipy\\nimport sklearn\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score\\nfrom sklearn.model_selection import GroupKFold\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import Normalizer\\nfrom sklearn.svm import LinearSVC\\n\\nLOGGER = logging.getLogger(\\"name_that_llm\\")\\nLABELS = tuple(sorted((\\n    \\"gpt-5.5\\", \\"gpt-5.6-luna\\", \\"gpt-5.6-sol\\", \\"gpt-5.6-terra\\",\\n    \\"gpt-6-astra\\", \\"gemini-3.1-pro-preview\\", \\"gemini-3.6-flash\\",\\n    \\"gemini-3.7-flash\\", \\"gemini-3.8-flash\\", \\"claude-fable-5\\",\\n    \\"claude-opus-4-6\\", \\"claude-opus-4-8\\", \\"claude-opus-5\\",\\n    \\"claude-sonnet-5\\", \\"qwen3.8-max\\", \\"glm-5.3\\",\\n    \\"glm-5.3-flash\\", \\"kimi-k3\\",\\n)))\\n\\n\\n@dataclass(frozen=True)\\nclass BaselineConfig:\\n    \\"\\"\\"Fixed before inspecting any validation results.\\"\\"\\"\\n\\n    variant: str = \\"char\\"\\n    seed: int = 42\\n    n_splits: int = 5\\n    c: float = 1.0\\n    min_df: int = 2\\n    char_max_features: int = 250_000\\n    word_max_features: int = 100_000\\n    max_iter: int = 10_000\\n    tolerance: float = 1e-4\\n\\n\\ndef validate_frame(frame: pd.DataFrame, *, training: bool) -> None:\\n    required = [\\"id\\", \\"text\\"] + ([\\"prompt_id\\", \\"model\\"] if training else [])\\n    missing = set(required) - set(frame.columns)\\n    if missing:\\n        raise ValueError(f\\"Missing columns: {sorted(missing)}\\")\\n    if frame.empty:\\n        raise ValueError(\\"Input must contain at least one passage.\\")\\n    for column in required:\\n        if frame[column].isna().any() or not frame[column].map(\\n            lambda value: isinstance(value, str) and bool(value.strip())\\n        ).all():\\n            raise ValueError(f\\"Column {column!r} contains missing, empty or non-string values.\\")\\n    if frame[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Passage IDs must be unique.\\")\\n    if training and set(frame[\\"model\\"]) != set(LABELS):\\n        raise ValueError(\\"Training labels must match all 18 competition labels.\\")\\n\\n\\ndef load_data(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:\\n    # Explicit string types retain leading zeros and the exact CSV-decoded text.\\n    train = pd.read_csv(data_dir / \\"train.csv\\", dtype=str, keep_default_na=False)\\n    test = pd.read_csv(data_dir / \\"test.csv\\", dtype=str, keep_default_na=False)\\n    validate_frame(train, training=True)\\n    validate_frame(test, training=False)\\n    return train, test\\n\\n\\ndef build_pipeline(config: BaselineConfig) -> Pipeline:\\n    if config.variant not in {\\"char\\", \\"char_word\\"}:\\n        raise ValueError(f\\"Unknown baseline variant: {config.variant}\\")\\n    common = dict(min_df=config.min_df, sublinear_tf=True, dtype=np.float32)\\n    features = [(\\"char\\", TfidfVectorizer(\\n        analyzer=\\"char\\", ngram_range=(3, 5), lowercase=False,\\n        strip_accents=None, max_features=config.char_max_features, **common,\\n    ))]\\n    if config.variant == \\"char_word\\":\\n        features.append((\\"word\\", TfidfVectorizer(\\n            analyzer=\\"word\\", ngram_range=(1, 2), lowercase=True,\\n            strip_accents=None, stop_words=None,\\n            max_features=config.word_max_features, **common,\\n        )))\\n    return Pipeline([\\n        (\\"features\\", FeatureUnion(features)),\\n        (\\"normalize\\", Normalizer()),\\n        (\\"classifier\\", LinearSVC(\\n            C=config.c, dual=True, random_state=config.seed,\\n            max_iter=config.max_iter, tol=config.tolerance,\\n        )),\\n    ])\\n\\n\\ndef make_folds(train: pd.DataFrame, config: BaselineConfig) -> list[tuple[np.ndarray, np.ndarray]]:\\n    validate_frame(train, training=True)\\n    splitter = GroupKFold(\\n        n_splits=config.n_splits, shuffle=True, random_state=config.seed,\\n    )\\n    folds = list(splitter.split(train[\\"text\\"], train[\\"model\\"], groups=train[\\"prompt_id\\"]))\\n    counts = np.zeros(len(train), dtype=int)\\n    for fitting, held_out in folds:\\n        if set(train.iloc[fitting][\\"prompt_id\\"]) & set(train.iloc[held_out][\\"prompt_id\\"]):\\n            raise ValueError(\\"A prompt appears in both sides of a fold.\\")\\n        for indexes in (fitting, held_out):\\n            if set(train.iloc[indexes][\\"model\\"]) != set(LABELS):\\n                raise ValueError(\\"Every fold must contain all 18 labels on both sides.\\")\\n        counts[held_out] += 1\\n    if not np.all(counts == 1):\\n        raise ValueError(\\"Each passage must be held out exactly once.\\")\\n    return folds\\n\\n\\ndef fit_checked(pipeline: Pipeline, text: pd.Series, labels: pd.Series) -> Pipeline:\\n    with warnings.catch_warnings(record=True) as caught:\\n        warnings.simplefilter(\\"always\\")\\n        pipeline.fit(text, labels)\\n    for warning in caught:\\n        LOGGER.warning(\\"%s: %s\\", warning.category.__name__, warning.message)\\n    if any(issubclass(w.category, ConvergenceWarning) for w in caught):\\n        raise RuntimeError(\\"LinearSVC did not converge; inspect the run log before proceeding.\\")\\n    return pipeline\\n\\n\\ndef validate_submission(\\n    test: pd.DataFrame, submission: pd.DataFrame,\\n) -> None:\\n    validate_frame(test, training=False)\\n    if list(submission.columns) != [\\"id\\", \\"model\\"]:\\n        raise ValueError(\\"Submission columns must be exactly id, model.\\")\\n    if len(submission) != len(test):\\n        raise ValueError(\\"Submission row count differs from the runtime test set.\\")\\n    if submission.isna().any().any() or submission[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Submission contains missing values or duplicate IDs.\\")\\n    if submission[\\"id\\"].tolist() != test[\\"id\\"].tolist():\\n        raise ValueError(\\"Submission IDs must match runtime test IDs in order.\\")\\n    if not set(submission[\\"model\\"]).issubset(LABELS):\\n        raise ValueError(\\"Submission contains an unknown model label.\\")\\n\\n\\ndef predict_submission(pipeline: Pipeline, test: pd.DataFrame) -> pd.DataFrame:\\n    validate_frame(test, training=False)\\n    predictions = pipeline.predict(test[\\"text\\"])\\n    submission = pd.DataFrame({\\"id\\": test[\\"id\\"].to_numpy(), \\"model\\": predictions})\\n    validate_submission(test, submission)\\n    return submission\\n\\n\\ndef file_sha256(path: Path) -> str:\\n    return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef runtime_versions() -> dict[str, str]:\\n    return {\\n        \\"python\\": platform.python_version(), \\"numpy\\": np.__version__,\\n        \\"pandas\\": pd.__version__, \\"scipy\\": scipy.__version__,\\n        \\"scikit_learn\\": sklearn.__version__,\\n    }\\n\\n\\nclass BaselineExperiment:\\n    \\"\\"\\"Compare two predeclared candidates on a single fixed prompt split.\\"\\"\\"\\n\\n    def __init__(self, data_dir: Path, output_dir: Path):\\n        self.data_dir = data_dir\\n        self.output_dir = output_dir\\n\\n    def run(self) -> dict:\\n        self.output_dir.mkdir(parents=True, exist_ok=True)\\n        train, test = load_data(self.data_dir)\\n        configs = [BaselineConfig(variant=v) for v in (\\"char\\", \\"char_word\\")]\\n        folds = make_folds(train, configs[0])\\n        assignment = train[[\\"id\\", \\"prompt_id\\"]].copy()\\n        assignment[\\"fold\\"] = -1\\n        for fold, (_, held_out) in enumerate(folds):\\n            assignment.loc[assignment.index[held_out], \\"fold\\"] = fold\\n        assignment.to_csv(self.output_dir / \\"folds.csv\\", index=False)\\n        summary = {\\n            \\"selection_metric\\": \\"pooled_oof_balanced_accuracy\\",\\n            \\"tie_break\\": \\"char (simpler candidate)\\",\\n            \\"versions\\": runtime_versions(),\\n            \\"train_sha256\\": file_sha256(self.data_dir / \\"train.csv\\"),\\n            \\"train_rows\\": len(train), \\"prompt_count\\": train[\\"prompt_id\\"].nunique(),\\n            \\"labels\\": list(LABELS), \\"candidates\\": {},\\n            \\"limitations\\": [\\n                \\"Prompt groups can still share related topics.\\",\\n                \\"Features were motivated by a label-visible review of all training data.\\",\\n                \\"Selecting the better of two candidates can make its CV estimate optimistic.\\",\\n                \\"Visible test predictions are a format check, not a scored evaluation.\\",\\n            ],\\n        }\\n        for config in configs:\\n            oof = np.empty(len(train), dtype=object)\\n            fold_metrics = []\\n            candidate_dir = self.output_dir / config.variant\\n            candidate_dir.mkdir(exist_ok=True)\\n            for fold, (fitting, held_out) in enumerate(folds):\\n                LOGGER.info(\\"Start %s fold=%d train=%d valid=%d\\",\\n                            config.variant, fold, len(fitting), len(held_out))\\n                started = time.perf_counter()\\n                pipeline = fit_checked(\\n                    build_pipeline(config), train.iloc[fitting][\\"text\\"],\\n                    train.iloc[fitting][\\"model\\"],\\n                )\\n                oof[held_out] = pipeline.predict(train.iloc[held_out][\\"text\\"])\\n                score = balanced_accuracy_score(train.iloc[held_out][\\"model\\"], oof[held_out])\\n                feature_sizes = {\\n                    name: len(vectorizer.vocabulary_)\\n                    for name, vectorizer in pipeline.named_steps[\\"features\\"].transformer_list\\n                }\\n                result = {\\n                    \\"fold\\": fold, \\"balanced_accuracy\\": float(score),\\n                    \\"train_rows\\": len(fitting), \\"validation_rows\\": len(held_out),\\n                    \\"seconds\\": time.perf_counter() - started,\\n                    \\"feature_counts\\": feature_sizes,\\n                    \\"solver_iterations\\": int(pipeline.named_steps[\\"classifier\\"].n_iter_),\\n                }\\n                fold_metrics.append(result)\\n                LOGGER.info(\\"Done %s %s\\", config.variant, json.dumps(result))\\n            predictions = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].copy()\\n            predictions[\\"fold\\"] = assignment[\\"fold\\"]\\n            predictions[\\"prediction\\"] = oof\\n            predictions.to_csv(candidate_dir / \\"oof_predictions.csv\\", index=False)\\n            matrix = confusion_matrix(train[\\"model\\"], oof, labels=LABELS)\\n            pd.DataFrame(matrix, index=LABELS, columns=LABELS).rename_axis(\\"true_model\\").to_csv(\\n                candidate_dir / \\"confusion_matrix.csv\\"\\n            )\\n            recalls = recall_score(train[\\"model\\"], oof, labels=LABELS, average=None)\\n            per_class = pd.DataFrame({\\n                \\"model\\": LABELS, \\"recall\\": recalls,\\n                \\"support\\": matrix.sum(axis=1),\\n                \\"correct\\": matrix.diagonal(),\\n            })\\n            per_class.to_csv(candidate_dir / \\"per_class_recall.csv\\", index=False)\\n            scores = [result[\\"balanced_accuracy\\"] for result in fold_metrics]\\n            summary[\\"candidates\\"][config.variant] = {\\n                \\"config\\": asdict(config),\\n                \\"pooled_oof_balanced_accuracy\\": float(balanced_accuracy_score(train[\\"model\\"], oof)),\\n                \\"fold_mean\\": float(np.mean(scores)),\\n                \\"fold_standard_deviation\\": float(np.std(scores, ddof=1)),\\n                \\"folds\\": fold_metrics,\\n                \\"per_class\\": per_class.to_dict(orient=\\"records\\"),\\n            }\\n        selected = max(\\n            summary[\\"candidates\\"],\\n            key=lambda name: summary[\\"candidates\\"][name][\\"pooled_oof_balanced_accuracy\\"],\\n        )\\n        summary[\\"selected_variant\\"] = selected\\n        # Write the comparison before the separate full-data training step.\\n        self._write_summary(summary)\\n        LOGGER.info(\\"Selected %s; refitting on all %d training rows\\", selected, len(train))\\n        config = BaselineConfig(**summary[\\"candidates\\"][selected][\\"config\\"])\\n        started = time.perf_counter()\\n        pipeline = fit_checked(build_pipeline(config), train[\\"text\\"], train[\\"model\\"])\\n        submission = predict_submission(pipeline, test)\\n        submission_path = self.output_dir / \\"submission.parquet\\"\\n        submission.to_parquet(submission_path, index=False)\\n        validate_submission(test, pd.read_parquet(submission_path))\\n        summary[\\"full_fit_and_preview_seconds\\"] = time.perf_counter() - started\\n        summary[\\"visible_test_rows\\"] = len(test)\\n        summary[\\"preview_submission_sha256\\"] = file_sha256(submission_path)\\n        self._write_summary(summary)\\n        LOGGER.info(\\"Preview submission verified: rows=%d; this is not a leaderboard score.\\", len(test))\\n        return summary\\n\\n    def _write_summary(self, summary: dict) -> None:\\n        (self.output_dir / \\"metrics.json\\").write_text(json.dumps(summary, indent=2) + \\"\\\\n\\")\\n", "boosting_v3.py": "\\"\\"\\"Three fixed CPU LightGBM recipes using the existing style measurements.\\n\\nDART motivation: https://arxiv.org/html/1505.01866v1.\\nOfficial implementation: https://github.com/lightgbm-org/LightGBM.\\nThis bounded experiment is not a reproduction or a state-of-the-art claim.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom lightgbm import LGBMClassifier\\nfrom sklearn.pipeline import Pipeline\\n\\nfrom name_that_llm.stylometry import StyleFeatures\\n\\n\\nBOOSTING_CANDIDATES = {\\n    \\"lightgbm7\\": {\\n        \\"boosting_type\\": \\"gbdt\\",\\n        \\"n_estimators\\": 400,\\n        \\"learning_rate\\": .035,\\n        \\"num_leaves\\": 7,\\n        \\"max_depth\\": 4,\\n        \\"min_child_samples\\": 25,\\n        \\"reg_lambda\\": 3.,\\n        \\"colsample_bytree\\": .8,\\n    },\\n    \\"lightgbm15\\": {\\n        \\"boosting_type\\": \\"gbdt\\",\\n        \\"n_estimators\\": 300,\\n        \\"learning_rate\\": .04,\\n        \\"num_leaves\\": 15,\\n        \\"max_depth\\": 5,\\n        \\"min_child_samples\\": 30,\\n        \\"reg_lambda\\": 10.,\\n        \\"colsample_bytree\\": .7,\\n    },\\n    \\"dart7\\": {\\n        \\"boosting_type\\": \\"dart\\",\\n        \\"n_estimators\\": 300,\\n        \\"learning_rate\\": .08,\\n        \\"num_leaves\\": 7,\\n        \\"max_depth\\": 4,\\n        \\"min_child_samples\\": 25,\\n        \\"reg_lambda\\": 3.,\\n        \\"colsample_bytree\\": .8,\\n        \\"drop_rate\\": .03,\\n        \\"skip_drop\\": .5,\\n        \\"max_drop\\": 30,\\n    },\\n}\\n\\nBOOSTING_COMMON = {\\n    \\"objective\\": \\"multiclass\\",\\n    \\"n_jobs\\": 2,\\n    \\"random_state\\": 42,\\n    \\"deterministic\\": True,\\n    \\"force_col_wise\\": True,\\n    \\"verbosity\\": -1,\\n    \\"subsample\\": 1.,\\n    \\"subsample_freq\\": 0,\\n    \\"max_bin\\": 127,\\n    \\"reg_alpha\\": 0.,\\n}\\n\\n\\ndef build_boosting_pipeline(name):\\n    \\"\\"\\"Return a fresh pipeline whose only learned component sees fit rows.\\"\\"\\"\\n    if name not in BOOSTING_CANDIDATES:\\n        raise ValueError(f\\"Unknown boosting recipe: {name}\\")\\n    return Pipeline([\\n        (\\"style\\", StyleFeatures()),\\n        (\\"classifier\\", LGBMClassifier(\\n            **BOOSTING_COMMON, **BOOSTING_CANDIDATES[name],\\n        )),\\n    ])\\n", "classifier_v5.py": "\\"\\"\\"Frozen LUAR linear probe and fixed evidence fusion for experiment v5.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nfrom pathlib import Path\\nimport time\\nimport warnings\\n\\nimport numpy as np\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.linear_model import LogisticRegression\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom name_that_llm.baseline import LABELS, file_sha256\\nfrom name_that_llm.ensemble_v2 import normalize_evidence\\nfrom name_that_llm.stacking_v4 import array_hash, validate_fold_assignment\\n\\nLOGGER = logging.getLogger(\\"embeddings_v5\\")\\nCLASSIFIER_PARAMETERS = {\\"C\\": 1.0, \\"class_weight\\": \\"balanced\\", \\"solver\\": \\"lbfgs\\",\\n                         \\"max_iter\\": 3000, \\"tol\\": 1e-4}\\nRECIPES = {\\"luar\\": {\\"luar_weight\\": 1.0, \\"v4_weight\\": 0.0},\\n           \\"luar_v4_half\\": {\\"luar_weight\\": .5, \\"v4_weight\\": .5}}\\n\\n\\ndef validate_embeddings(values, rows=None):\\n    values = np.asarray(values)\\n    if values.ndim != 2 or values.shape[1] != 512 or not len(values):\\n        raise ValueError(\\"Expected nonempty row-by-512 LUAR embeddings\\")\\n    if rows is not None and len(values) != rows:\\n        raise ValueError(\\"Embedding row count differs\\")\\n    if not np.issubdtype(values.dtype, np.floating) or not np.isfinite(values).all():\\n        raise ValueError(\\"Embeddings must be finite floating-point values\\")\\n    if not np.allclose(np.linalg.norm(values, axis=1), 1, atol=2e-6, rtol=2e-5):\\n        raise ValueError(\\"Expected unit-norm embeddings\\")\\n    return values\\n\\n\\ndef fit_classifier(embeddings, y):\\n    values = validate_embeddings(embeddings, len(y))\\n    labels = np.asarray(y)\\n    if labels.ndim != 1 or set(labels) != set(LABELS):\\n        raise ValueError(\\"Training requires all 18 competition labels\\")\\n    model = LogisticRegression(**CLASSIFIER_PARAMETERS)\\n    with threadpool_limits(limits=2), warnings.catch_warnings(record=True) as caught:\\n        warnings.simplefilter(\\"always\\")\\n        model.fit(values, labels)\\n    for warning in caught:\\n        LOGGER.warning(\\"%s: %s\\", warning.category.__name__, warning.message)\\n    if any(issubclass(w.category, ConvergenceWarning) for w in caught):\\n        raise RuntimeError(\\"LUAR classifier did not converge\\")\\n    if tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"LUAR classifier class order differs\\")\\n    return model\\n\\n\\ndef incumbent_evidence(stack_probs, base_evidence):\\n    stack = normalize_evidence(stack_probs, probabilities=True)\\n    base = np.asarray(base_evidence)\\n    if base.shape != (len(stack), 4, len(LABELS)) or not np.isfinite(base).all():\\n        raise ValueError(\\"Invalid incumbent base evidence\\")\\n    return .5 * stack + .5 * base[:, [0, 1, 3]].mean(axis=1)\\n\\n\\ndef combine_evidence(luar_probs, v4_scores, recipe):\\n    if recipe not in RECIPES.values():\\n        raise ValueError(\\"Recipe differs from the two frozen candidates\\")\\n    luar = normalize_evidence(luar_probs, probabilities=True)\\n    v4 = normalize_evidence(v4_scores)\\n    if luar.shape != v4.shape:\\n        raise ValueError(\\"Candidate and incumbent rows differ\\")\\n    return recipe[\\"luar_weight\\"] * luar + recipe[\\"v4_weight\\"] * v4\\n\\n\\ndef evaluate_oof(embeddings, y, folds, groups):\\n    values = validate_embeddings(embeddings, len(y))\\n    labels = np.asarray(y)\\n    assignment = validate_fold_assignment(folds, labels, groups)\\n    groups = np.asarray(groups)\\n    result = np.full((len(values), len(LABELS)), np.nan)\\n    audit = []\\n    for fold in range(5):\\n        fit = np.flatnonzero(assignment != fold)\\n        valid = np.flatnonzero(assignment == fold)\\n        if set(groups[fit]) & set(groups[valid]):\\n            raise ValueError(\\"Training groups overlap held-out groups\\")\\n        started = time.perf_counter()\\n        model = fit_classifier(values[fit], labels[fit])\\n        with threadpool_limits(limits=2):\\n            result[valid] = model.predict_proba(values[valid])\\n        record = {\\"held_out_fold\\": fold, \\"training_rows\\": len(fit), \\"held_out_rows\\": len(valid),\\n                  \\"training_indexes_sha256\\": array_hash(fit), \\"held_out_indexes_sha256\\": array_hash(valid),\\n                  \\"training_groups\\": sorted(set(groups[fit])), \\"held_out_groups\\": sorted(set(groups[valid])),\\n                  \\"classes\\": model.classes_.tolist(), \\"iterations\\": model.n_iter_.tolist(),\\n                  \\"seconds\\": time.perf_counter() - started}\\n        audit.append(record)\\n        LOGGER.info(\\"Completed fold=%d fit_rows=%d valid_rows=%d seconds=%.3f\\", fold, len(fit), len(valid), record[\\"seconds\\"])\\n    normalize_evidence(result, probabilities=True)\\n    return result, audit\\n\\n\\ndef source_hashes():\\n    from name_that_llm.stacking_v4 import source_hashes as incumbent_hashes\\n    hashes = incumbent_hashes()\\n    root = Path(__file__).resolve().parent\\n    for name in (\\"classifier_v5.py\\", \\"embeddings_v5.py\\"):\\n        hashes[name] = file_sha256(root / name)\\n    return hashes\\n", "classifier_v6.py": "\\"\\"\\"Two preregistered probes of the unchanged LUAR representation for v6.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom copy import deepcopy\\nimport logging\\nfrom pathlib import Path\\nimport time\\nimport warnings\\n\\nimport numpy as np\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import balanced_accuracy_score\\nfrom sklearn.svm import SVC\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom name_that_llm.baseline import LABELS, file_sha256\\nfrom name_that_llm.classifier_v5 import validate_embeddings\\nfrom name_that_llm.ensemble_v2 import normalize_evidence\\nfrom name_that_llm.stacking_v4 import array_hash, validate_fold_assignment\\n\\nLOGGER = logging.getLogger(\\"embeddings_v6\\")\\nMODEL_DEFINITIONS = {\\n    \\"lr64\\": {\\"kind\\": \\"logistic\\", \\"parameters\\": {\\n        \\"C\\": 64.0, \\"class_weight\\": \\"balanced\\", \\"solver\\": \\"lbfgs\\", \\"max_iter\\": 3000, \\"tol\\": 1e-4}},\\n    \\"rbf10\\": {\\"kind\\": \\"svc\\", \\"parameters\\": {\\n        \\"C\\": 10.0, \\"kernel\\": \\"rbf\\", \\"gamma\\": \\"scale\\", \\"class_weight\\": \\"balanced\\",\\n        \\"probability\\": False, \\"decision_function_shape\\": \\"ovr\\", \\"cache_size\\": 512,\\n        \\"tol\\": .001, \\"max_iter\\": -1}},\\n}\\nRECIPES = {\\n    \\"lr64\\": {\\"model\\": \\"lr64\\", \\"embedding_weight\\": 1.0, \\"v4_weight\\": 0.0},\\n    \\"lr64_v4_half\\": {\\"model\\": \\"lr64\\", \\"embedding_weight\\": .5, \\"v4_weight\\": .5},\\n    \\"rbf10\\": {\\"model\\": \\"rbf10\\", \\"embedding_weight\\": 1.0, \\"v4_weight\\": 0.0},\\n    \\"rbf10_v4_half\\": {\\"model\\": \\"rbf10\\", \\"embedding_weight\\": .5, \\"v4_weight\\": .5},\\n}\\n\\n\\ndef fit_classifier(embeddings, y, model_name):\\n    if model_name not in MODEL_DEFINITIONS:\\n        raise ValueError(\\"Unknown frozen classifier\\")\\n    values = validate_embeddings(embeddings, len(y))\\n    labels = np.asarray(y)\\n    if labels.ndim != 1 or set(labels) != set(LABELS):\\n        raise ValueError(\\"Training requires all 18 competition labels\\")\\n    definition = MODEL_DEFINITIONS[model_name]\\n    factory = LogisticRegression if definition[\\"kind\\"] == \\"logistic\\" else SVC\\n    model = factory(**deepcopy(definition[\\"parameters\\"]))\\n    with threadpool_limits(limits=2), warnings.catch_warnings(record=True) as caught:\\n        warnings.simplefilter(\\"always\\")\\n        model.fit(values, labels)\\n    for warning in caught:\\n        LOGGER.warning(\\"%s %s: %s\\", model_name, warning.category.__name__, warning.message)\\n    if any(issubclass(w.category, ConvergenceWarning) for w in caught) or getattr(model, \\"fit_status_\\", 0) != 0:\\n        raise RuntimeError(f\\"Frozen classifier {model_name} did not converge\\")\\n    if tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"Classifier class order differs\\")\\n    return model\\n\\n\\ndef model_evidence(model, embeddings, model_name):\\n    \\"\\"\\"Use LR log probabilities or SVC OVR scores, without fitting calibration.\\"\\"\\"\\n    if model_name not in MODEL_DEFINITIONS or tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"Unknown model or incorrect class order\\")\\n    values = validate_embeddings(embeddings)\\n    with threadpool_limits(limits=2):\\n        if MODEL_DEFINITIONS[model_name][\\"kind\\"] == \\"logistic\\":\\n            return normalize_evidence(model.predict_proba(values), probabilities=True)\\n        return normalize_evidence(model.decision_function(values))\\n\\n\\ndef combine_evidence(embedding_scores, v4_scores, recipe):\\n    if recipe not in RECIPES.values():\\n        raise ValueError(\\"Recipe differs from the four frozen candidates\\")\\n    scores = np.asarray(embedding_scores, dtype=np.float64)\\n    if scores.ndim != 2 or scores.shape[1] != len(LABELS) or not np.isfinite(scores).all():\\n        raise ValueError(\\"Invalid normalized embedding evidence\\")\\n    # The probe evidence is already normalized by model_evidence. Re-normalize\\n    # only the completed incumbent, whose internal fusion has a different scale.\\n    v4 = normalize_evidence(v4_scores)\\n    if scores.shape != v4.shape:\\n        raise ValueError(\\"Embedding and incumbent evidence rows differ\\")\\n    return recipe[\\"embedding_weight\\"] * scores + recipe[\\"v4_weight\\"] * v4\\n\\n\\ndef choose_recipe(results):\\n    \\"\\"\\"Highest score, with exact ties resolved in frozen declaration order.\\"\\"\\"\\n    names = [name for name in RECIPES if name in results]\\n    if not names or set(names) != set(results):\\n        raise ValueError(\\"Unknown or empty recipe results\\")\\n    for name in names:\\n        score = results[name][\\"balanced_accuracy\\"]\\n        if not np.isfinite(score) or not 0 <= score <= 1:\\n            raise ValueError(\\"Invalid selection score\\")\\n    return max(names, key=lambda name: results[name][\\"balanced_accuracy\\"])\\n\\n\\ndef evaluate_oof(embeddings, y, folds, groups, model_name):\\n    values = validate_embeddings(embeddings, len(y))\\n    labels = np.asarray(y)\\n    assignment = validate_fold_assignment(folds, labels, groups)\\n    groups = np.asarray(groups)\\n    result = np.full((len(values), len(LABELS)), np.nan, dtype=np.float64)\\n    audit = []\\n    for fold in range(5):\\n        fit = np.flatnonzero(assignment != fold)\\n        valid = np.flatnonzero(assignment == fold)\\n        if set(groups[fit]) & set(groups[valid]):\\n            raise ValueError(\\"Training groups overlap held-out groups\\")\\n        started = time.perf_counter()\\n        model = fit_classifier(values[fit], labels[fit], model_name)\\n        train_scores = model_evidence(model, values[fit], model_name)\\n        held_out_scores = model_evidence(model, values[valid], model_name)\\n        result[valid] = held_out_scores\\n        record = {\\n            \\"model\\": model_name, \\"definition\\": deepcopy(MODEL_DEFINITIONS[model_name]),\\n            \\"held_out_fold\\": fold, \\"training_rows\\": len(fit), \\"held_out_rows\\": len(valid),\\n            \\"training_indexes_sha256\\": array_hash(fit), \\"held_out_indexes_sha256\\": array_hash(valid),\\n            \\"training_embeddings_sha256\\": array_hash(values[fit]),\\n            \\"held_out_embeddings_sha256\\": array_hash(values[valid]),\\n            \\"held_out_scores_sha256\\": array_hash(held_out_scores),\\n            \\"training_groups\\": sorted(set(groups[fit])), \\"held_out_groups\\": sorted(set(groups[valid])),\\n            \\"classes\\": model.classes_.tolist(), \\"iterations\\": np.asarray(model.n_iter_).tolist(),\\n            \\"training_balanced_accuracy\\": float(balanced_accuracy_score(labels[fit], np.asarray(LABELS)[train_scores.argmax(axis=1)])),\\n            \\"held_out_balanced_accuracy\\": float(balanced_accuracy_score(labels[valid], np.asarray(LABELS)[held_out_scores.argmax(axis=1)])),\\n            \\"prediction_rule\\": \\"argmax of normalized evidence, in LABELS order\\",\\n            \\"seconds\\": time.perf_counter() - started,\\n        }\\n        if MODEL_DEFINITIONS[model_name][\\"kind\\"] == \\"svc\\":\\n            record.update({\\"support_vectors\\": int(len(model.support_)), \\"support_vectors_per_class\\": model.n_support_.tolist(),\\n                           \\"fitted_gamma\\": float(model._gamma), \\"fit_status\\": int(model.fit_status_),\\n                           \\"probability_calibration\\": False})\\n        audit.append(record)\\n        LOGGER.info(\\"Completed model=%s fold=%d train_BA=%.6f valid_BA=%.6f seconds=%.3f\\", model_name, fold,\\n                    record[\\"training_balanced_accuracy\\"], record[\\"held_out_balanced_accuracy\\"], record[\\"seconds\\"])\\n    if not np.isfinite(result).all():\\n        raise ValueError(\\"Incomplete or nonfinite OOF evidence\\")\\n    return result, audit\\n\\n\\ndef source_hashes():\\n    from name_that_llm.classifier_v5 import source_hashes as predecessor_hashes\\n    hashes = predecessor_hashes()\\n    source = Path(__file__).resolve()\\n    hashes[source.name] = file_sha256(source)\\n    scripts = source.parents[2] / \\"scripts\\"\\n    for name in (\\"run_embeddings_v5.py\\", \\"select_v4.py\\", \\"run_nested_v4.py\\", \\"extract_embeddings_v5.py\\"):\\n        hashes[f\\"scripts/{name}\\"] = file_sha256(scripts / name)\\n    return hashes\\n", "classifier_v7.py": "\\"\\"\\"One frozen linear probe of the separately released StyleDistance encoder.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport logging\\nfrom pathlib import Path\\nimport time\\nimport warnings\\n\\nimport numpy as np\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import balanced_accuracy_score\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom name_that_llm.baseline import LABELS, file_sha256\\nfrom name_that_llm.ensemble_v2 import normalize_evidence\\nfrom name_that_llm.stacking_v4 import array_hash, validate_fold_assignment\\n\\nLOGGER = logging.getLogger(\\"styledistance_v7\\")\\nCLASSIFIER_PARAMETERS = {\\"C\\": 64., \\"class_weight\\": \\"balanced\\", \\"solver\\": \\"lbfgs\\", \\"max_iter\\": 3000, \\"tol\\": 1e-4}\\nRECIPES = {\\"styledistance\\": {\\"embedding_weight\\": 1., \\"v4_weight\\": 0.},\\n           \\"styledistance_v4_half\\": {\\"embedding_weight\\": .5, \\"v4_weight\\": .5}}\\n\\n\\ndef validate_embeddings(values, rows=None):\\n    values = np.asarray(values)\\n    if values.ndim != 2 or values.shape[1] != 768 or not len(values):\\n        raise ValueError(\\"Expected nonempty row-by-768 StyleDistance embeddings\\")\\n    if rows is not None and len(values) != rows:\\n        raise ValueError(\\"Embedding row count differs\\")\\n    if not np.issubdtype(values.dtype, np.floating) or not np.isfinite(values).all():\\n        raise ValueError(\\"Embeddings must be finite floating-point values\\")\\n    if not np.allclose(np.linalg.norm(values, axis=1), 1, atol=2e-6, rtol=2e-5):\\n        raise ValueError(\\"Expected unit-norm embeddings\\")\\n    return values\\n\\n\\ndef fit_classifier(embeddings, y):\\n    values = validate_embeddings(embeddings, len(y))\\n    labels = np.asarray(y)\\n    if labels.ndim != 1 or set(labels) != set(LABELS):\\n        raise ValueError(\\"All 18 competition labels are required\\")\\n    model = LogisticRegression(**CLASSIFIER_PARAMETERS)\\n    with threadpool_limits(limits=2), warnings.catch_warnings(record=True) as caught:\\n        warnings.simplefilter(\\"always\\")\\n        model.fit(values, labels)\\n    for warning in caught:\\n        LOGGER.warning(\\"%s: %s\\", warning.category.__name__, warning.message)\\n    if any(issubclass(w.category, ConvergenceWarning) for w in caught):\\n        raise RuntimeError(\\"StyleDistance linear classifier did not converge\\")\\n    if tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"Classifier class order differs\\")\\n    return model\\n\\n\\ndef model_evidence(model, embeddings):\\n    if tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"Classifier class order differs\\")\\n    with threadpool_limits(limits=2):\\n        return normalize_evidence(model.predict_proba(validate_embeddings(embeddings)), probabilities=True)\\n\\n\\ndef combine_evidence(embedding_scores, v4_scores, recipe):\\n    if recipe not in RECIPES.values():\\n        raise ValueError(\\"Recipe differs from the two frozen candidates\\")\\n    scores = np.asarray(embedding_scores)\\n    incumbent = normalize_evidence(v4_scores)\\n    if scores.shape != incumbent.shape or not np.isfinite(scores).all():\\n        raise ValueError(\\"Invalid candidate evidence shape or values\\")\\n    return recipe[\\"embedding_weight\\"] * scores + recipe[\\"v4_weight\\"] * incumbent\\n\\n\\ndef choose_recipe(results):\\n    names = [name for name in RECIPES if name in results]\\n    if not names or set(names) != set(results):\\n        raise ValueError(\\"Unknown or empty recipe results\\")\\n    if any(not np.isfinite(results[n][\\"balanced_accuracy\\"]) or not 0 <= results[n][\\"balanced_accuracy\\"] <= 1 for n in names):\\n        raise ValueError(\\"Invalid candidate score\\")\\n    return max(names, key=lambda n: results[n][\\"balanced_accuracy\\"])\\n\\n\\ndef evaluate_oof(embeddings, y, folds, groups):\\n    values = validate_embeddings(embeddings, len(y))\\n    labels, groups = np.asarray(y), np.asarray(groups)\\n    assignment = validate_fold_assignment(folds, labels, groups)\\n    evidence = np.full((len(values), len(LABELS)), np.nan)\\n    audit = []\\n    for fold in range(5):\\n        fitting, valid = np.flatnonzero(assignment != fold), np.flatnonzero(assignment == fold)\\n        if set(groups[fitting]) & set(groups[valid]):\\n            raise ValueError(\\"Training and held-out groups overlap\\")\\n        started = time.perf_counter()\\n        model = fit_classifier(values[fitting], labels[fitting])\\n        evidence[valid] = model_evidence(model, values[valid])\\n        train_predictions = np.asarray(LABELS)[model_evidence(model, values[fitting]).argmax(axis=1)]\\n        record = {\\"held_out_fold\\": fold, \\"training_rows\\": len(fitting), \\"held_out_rows\\": len(valid),\\n                  \\"training_indexes_sha256\\": array_hash(fitting), \\"held_out_indexes_sha256\\": array_hash(valid),\\n                  \\"training_embeddings_sha256\\": array_hash(values[fitting]), \\"held_out_embeddings_sha256\\": array_hash(values[valid]),\\n                  \\"held_out_scores_sha256\\": array_hash(evidence[valid]),\\n                  \\"training_groups\\": sorted(set(groups[fitting])), \\"held_out_groups\\": sorted(set(groups[valid])),\\n                  \\"classes\\": model.classes_.tolist(), \\"iterations\\": model.n_iter_.tolist(),\\n                  \\"training_balanced_accuracy\\": float(balanced_accuracy_score(labels[fitting], train_predictions)),\\n                  \\"seconds\\": time.perf_counter() - started}\\n        audit.append(record)\\n        LOGGER.info(\\"Completed fold=%d train_BA=%.6f seconds=%.3f\\", fold, record[\\"training_balanced_accuracy\\"], record[\\"seconds\\"])\\n    if not np.isfinite(evidence).all():\\n        raise ValueError(\\"Incomplete OOF evidence\\")\\n    return evidence, audit\\n\\n\\ndef source_hashes():\\n    from name_that_llm.classifier_v5 import source_hashes as imported_hashes\\n    hashes = imported_hashes()\\n    root = Path(__file__).resolve().parent\\n    for name in (\\"encoder_v7.py\\", \\"classifier_v7.py\\"):\\n        hashes[name] = file_sha256(root / name)\\n    for name in (\\"run_embeddings_v5.py\\", \\"select_v4.py\\", \\"run_nested_v4.py\\", \\"extract_embeddings_v7.py\\"):\\n        hashes[f\\"scripts/{name}\\"] = file_sha256(root.parents[1] / \\"scripts\\" / name)\\n    return hashes\\n", "embeddings_v5.py": "\\"\\"\\"Audited, local-only frozen LUAR-CRUD representations for experiment v5.\\n\\nEach row is one episode containing every consecutive 126-token content window.\\nThe upstream episode attention has no document-padding mask: only equal-length\\nepisodes may share an inference batch. Competition labels never enter this API.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections import defaultdict\\nfrom dataclasses import asdict, dataclass\\nimport hashlib\\nimport importlib.metadata\\nimport importlib.util\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport platform\\nimport sys\\nimport time\\nimport types\\nfrom typing import Callable, Sequence\\n\\nimport numpy as np\\n\\n\\nMODEL_ID = \\"rrivera1849/LUAR-CRUD\\"\\nMODEL_REVISION = \\"8b36ce7f289ee91900a7dff0eaccf5cdca745ab1\\"\\nREVIEWED_SOURCE_SHA256 = {\\n    \\"model.py\\": \\"70d320a277fd26447bae9a57a8b1c4550097f9430666c3c3b106ff736e26c3b5\\",\\n    \\"config.py\\": \\"1425b832fbb977f3109b5e3429666a5d0d07be643b3853be53d942162def4b94\\",\\n    \\"config.json\\": \\"df2e852aabe2b4361ed9cca5b8112c3a7c336346d1928168b79839c0380d5927\\",\\n    \\"tokenizer.json\\": \\"7b62d797c95d1563d3467f48daad0eca3e150f2d587e7e687b557c9ef1b25473\\",\\n    \\"tokenizer_config.json\\": \\"4008a762b09edcbdd9b7583372e806ae65bf640ea419e34a6ba2b770e0e91a6b\\",\\n    \\"special_tokens_map.json\\": \\"8c785abebea9ae3257b61681b4e6fd8365ceafde980c21970d001e834cf10835\\",\\n}\\nREQUIRED_MODEL_FILES = (*REVIEWED_SOURCE_SHA256, \\"model.safetensors\\", \\"vocab.json\\", \\"merges.txt\\", \\"README.md\\")\\n\\n\\ndef _json_native(value):\\n    \\"\\"\\"Transformers 5 loading diagnostics include sets; preserve them as lists.\\"\\"\\"\\n    if isinstance(value, dict):\\n        return {key: _json_native(item) for key, item in value.items()}\\n    if isinstance(value, set):\\n        return [_json_native(item) for item in sorted(value, key=repr)]\\n    if isinstance(value, (tuple, list)):\\n        return [_json_native(item) for item in value]\\n    if isinstance(value, np.generic):\\n        return value.item()\\n    return value\\n\\n\\ndef sha256_file(path: str | Path) -> str:\\n    digest = hashlib.sha256()\\n    with Path(path).open(\\"rb\\") as handle:\\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef sha256_text(text: str) -> str:\\n    if not isinstance(text, str):\\n        raise TypeError(\\"Completion text must be an unchanged string\\")\\n    return hashlib.sha256(text.encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef build_row_manifest(ids: Sequence[str], texts: Sequence[str]) -> list[dict]:\\n    if len(ids) != len(texts):\\n        raise ValueError(\\"IDs and texts have different lengths\\")\\n    if len(set(ids)) != len(ids) or any(not isinstance(x, str) or not x for x in ids):\\n        raise ValueError(\\"Row IDs must be unique nonempty strings\\")\\n    return [{\\"row_index\\": i, \\"id\\": row_id, \\"text_sha256\\": sha256_text(text)}\\n            for i, (row_id, text) in enumerate(zip(ids, texts, strict=True))]\\n\\n\\ndef validate_model_manifest(model_dir: str | Path) -> dict:\\n    \\"\\"\\"Verify pinned identity and every model file before executing local source.\\"\\"\\"\\n    root = Path(model_dir).resolve(strict=True)\\n    manifest = json.loads((root / \\"manifest.json\\").read_text())\\n    if manifest.get(\\"model_id\\") != MODEL_ID or manifest.get(\\"revision\\") != MODEL_REVISION:\\n        raise ValueError(\\"Unexpected LUAR model identity or revision\\")\\n    entries = {entry[\\"path\\"]: entry for entry in manifest[\\"files\\"]}\\n    for name in REQUIRED_MODEL_FILES:\\n        if name not in entries:\\n            raise ValueError(f\\"Missing model manifest entry: {name}\\")\\n        path = root / name\\n        if path.is_symlink() or not path.is_file():\\n            raise ValueError(f\\"Expected a local regular model file: {name}\\")\\n        if path.stat().st_size != entries[name][\\"size_bytes\\"] or sha256_file(path) != entries[name][\\"sha256\\"]:\\n            raise ValueError(f\\"Model manifest mismatch: {name}\\")\\n    for name, expected in REVIEWED_SOURCE_SHA256.items():\\n        if entries[name][\\"sha256\\"] != expected:\\n            raise ValueError(f\\"Source differs from the reviewed revision: {name}\\")\\n    return manifest\\n\\n\\ndef load_embedding_cache(cache_dir: str | Path, split: str, ids: Sequence[str],\\n                         texts: Sequence[str], *, model_dir: str | Path,\\n                         plan_path: str | Path, source_path: str | Path | None = None) -> tuple[np.ndarray, dict]:\\n    \\"\\"\\"Reject stale, reordered, edited or mismatched cached vectors before fitting.\\"\\"\\"\\n    if split not in (\\"train\\", \\"test\\"):\\n        raise ValueError(\\"Expected train or test cache\\")\\n    root = Path(cache_dir)\\n    metadata = json.loads((root / f\\"{split}-embeddings.json\\").read_text())\\n    model_manifest = validate_model_manifest(model_dir)\\n    provenance = metadata[\\"provenance\\"]\\n    expected_rows = build_row_manifest(ids, texts)\\n    plan = json.loads(Path(plan_path).read_text())\\n    checks = {\\n        \\"status\\": metadata[\\"status\\"] == \\"complete\\",\\n        \\"split\\": metadata[\\"split\\"] == split,\\n        \\"rows\\": metadata[\\"rows\\"] == expected_rows,\\n        \\"shape\\": metadata[\\"shape\\"] == [len(ids), 512],\\n        \\"dtype\\": metadata[\\"dtype\\"] == \\"float32\\",\\n        \\"plan\\": metadata[\\"plan_sha256\\"] == sha256_file(plan_path),\\n        \\"model_identity\\": provenance[\\"model_id\\"] == MODEL_ID and provenance[\\"revision\\"] == MODEL_REVISION,\\n        \\"model_manifest\\": provenance[\\"manifest_sha256\\"] == sha256_file(Path(model_dir) / \\"manifest.json\\"),\\n        \\"model_files\\": provenance[\\"model_files\\"] == model_manifest[\\"files\\"],\\n        \\"settings\\": provenance[\\"runtime\\"][\\"settings\\"] == plan[\\"encoder\\"],\\n        \\"encoder_source\\": provenance[\\"encoder_source_sha256\\"] == sha256_file(__file__),\\n        \\"extraction_source\\": metadata[\\"script_sha256\\"] == sha256_file(Path(__file__).resolve().parents[2] / \\"scripts\\" / \\"extract_embeddings_v5.py\\"),\\n    }\\n    if source_path is not None:\\n        checks[\\"source_csv\\"] = metadata[\\"source_csv_sha256\\"] == sha256_file(source_path)\\n    if not all(checks.values()):\\n        raise ValueError(f\\"Embedding cache provenance mismatch: {[key for key, valid in checks.items() if not valid]}\\")\\n    artifact = metadata[\\"artifact\\"]\\n    path = root / f\\"{split}-embeddings.npz\\"\\n    if artifact[\\"path\\"] != path.name or path.stat().st_size != artifact[\\"size_bytes\\"] or sha256_file(path) != artifact[\\"sha256\\"]:\\n        raise ValueError(\\"Embedding cache artifact checksum mismatch\\")\\n    with np.load(path, allow_pickle=False) as cache:\\n        embeddings = cache[\\"embeddings\\"]\\n        if cache[\\"ids\\"].tolist() != list(ids) or cache[\\"text_sha256\\"].tolist() != [row[\\"text_sha256\\"] for row in expected_rows]:\\n            raise ValueError(\\"Embedding cache row/text mismatch\\")\\n        tokens, windows = cache[\\"content_tokens\\"], cache[\\"window_counts\\"]\\n        if tokens.shape != (len(ids),) or windows.shape != (len(ids),) or (tokens < 0).any():\\n            raise ValueError(\\"Embedding cache token counts are invalid\\")\\n        if not np.array_equal(windows, np.maximum(1, (tokens + 125) // 126)):\\n            raise ValueError(\\"Embedding cache window coverage is inconsistent\\")\\n        if tokens.tolist() != metadata[\\"report\\"][\\"content_tokens\\"] or windows.tolist() != metadata[\\"report\\"][\\"window_counts\\"]:\\n            raise ValueError(\\"Embedding cache count metadata mismatch\\")\\n        if embeddings.shape != (len(ids), 512) or embeddings.dtype != np.float32 or not np.isfinite(embeddings).all():\\n            raise ValueError(\\"Embedding cache vectors are invalid\\")\\n        if not np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=2e-6, rtol=0):\\n            raise ValueError(\\"Embedding cache vectors are not unit normalized\\")\\n    return embeddings, metadata\\n\\n\\n@dataclass(frozen=True)\\nclass EncoderSettings:\\n    content_tokens: int = 126\\n    sequence_length: int = 128\\n    max_batch_windows: int = 32\\n    document_batch_size: int = 16\\n    cpu_threads: int = 2\\n    device: str = \\"cpu\\"\\n\\n    def __post_init__(self):\\n        if self.content_tokens != 126 or self.sequence_length != 128:\\n            raise ValueError(\\"V5 fixes 126 content tokens plus two special tokens\\")\\n        if min(self.max_batch_windows, self.document_batch_size, self.cpu_threads) < 1:\\n            raise ValueError(\\"Batch bounds and CPU thread count must be positive\\")\\n        if self.device != \\"cpu\\":\\n            raise ValueError(\\"V5 uses the audited CPU float32 path\\")\\n\\n\\n@dataclass(frozen=True)\\nclass TokenizedEpisode:\\n    input_ids: np.ndarray\\n    attention_mask: np.ndarray\\n    content_tokens: int\\n\\n    @property\\n    def windows(self) -> int:\\n        return len(self.input_ids)\\n\\n\\nclass FrozenLuarEncoder:\\n    \\"\\"\\"Load the reviewed local custom classes and compute normalized vectors.\\"\\"\\"\\n\\n    def __init__(self, model_dir: str | Path, settings: EncoderSettings | None = None):\\n        self.settings = settings or EncoderSettings()\\n        self.model_dir = Path(model_dir).resolve(strict=True)\\n        self.manifest = validate_model_manifest(self.model_dir)\\n        # Set before importing HF. The executable also installs a socket guard.\\n        os.environ[\\"HF_HUB_OFFLINE\\"] = \\"1\\"\\n        os.environ[\\"TRANSFORMERS_OFFLINE\\"] = \\"1\\"\\n        os.environ[\\"HF_HUB_DISABLE_TELEMETRY\\"] = \\"1\\"\\n        os.environ[\\"TOKENIZERS_PARALLELISM\\"] = \\"false\\"\\n        import torch\\n        from transformers import AutoTokenizer\\n\\n        self.torch = torch\\n        torch.set_num_threads(self.settings.cpu_threads)\\n        torch.use_deterministic_algorithms(True)\\n        config_class, model_class = self._load_local_classes()\\n        config = config_class.from_pretrained(str(self.model_dir), local_files_only=True)\\n        self.model, loading_info = model_class.from_pretrained(\\n            str(self.model_dir), config=config, local_files_only=True,\\n            use_safetensors=True, output_loading_info=True,\\n        )\\n        loading_info = _json_native(loading_info)\\n        bad = {key: value for key, value in loading_info.items() if value}\\n        if bad:\\n            raise ValueError(f\\"Checkpoint did not load exactly: {bad}\\")\\n        self.model.to(device=\\"cpu\\", dtype=torch.float32).eval()\\n        self.model.requires_grad_(False)\\n        self.tokenizer = AutoTokenizer.from_pretrained(\\n            str(self.model_dir), local_files_only=True, trust_remote_code=False, use_fast=True,\\n        )\\n        if self.tokenizer.num_special_tokens_to_add(pair=False) != 2:\\n            raise ValueError(\\"Expected the reviewed RoBERTa two-token template\\")\\n        if (self.tokenizer.cls_token_id, self.tokenizer.sep_token_id, self.tokenizer.pad_token_id) != (0, 2, 1):\\n            raise ValueError(\\"Special token IDs differ from the reviewed RoBERTa template\\")\\n        self.report: dict = {}\\n        self.runtime = {\\n            \\"python\\": platform.python_version(), \\"platform\\": platform.platform(),\\n            **{name: importlib.metadata.version(name) for name in\\n               (\\"torch\\", \\"transformers\\", \\"einops\\", \\"huggingface_hub\\", \\"tokenizers\\", \\"safetensors\\", \\"numpy\\")},\\n            \\"parameter_count\\": sum(p.numel() for p in self.model.parameters()),\\n            \\"parameter_dtype\\": str(next(self.model.parameters()).dtype),\\n            \\"backbone_attention\\": self.model.transformer.config._attn_implementation,\\n            \\"torch_num_threads\\": torch.get_num_threads(),\\n            \\"settings\\": asdict(self.settings), \\"loading_info\\": loading_info,\\n            \\"loaded_custom_source\\": {\\n                name: {\\"path\\": str(self.model_dir / name), \\"sha256\\": sha256_file(self.model_dir / name)}\\n                for name in (\\"model.py\\", \\"config.py\\")\\n            },\\n        }\\n\\n    def _load_local_classes(self):\\n        # No dynamic HF modules cache, trust_remote_code or remote-code resolution.\\n        package_name = \\"_reviewed_luar_crud_\\" + hashlib.sha256(str(self.model_dir).encode()).hexdigest()[:16]\\n        package = types.ModuleType(package_name)\\n        package.__path__ = [str(self.model_dir)]\\n        sys.modules[package_name] = package\\n        modules = {}\\n        old_bytecode = sys.dont_write_bytecode\\n        try:\\n            sys.dont_write_bytecode = True\\n            for name in (\\"config\\", \\"model\\"):\\n                qualified = f\\"{package_name}.{name}\\"\\n                spec = importlib.util.spec_from_file_location(qualified, self.model_dir / f\\"{name}.py\\")\\n                if spec is None or spec.loader is None:\\n                    raise ImportError(f\\"Cannot load reviewed local module {name}\\")\\n                module = importlib.util.module_from_spec(spec)\\n                sys.modules[qualified] = module\\n                spec.loader.exec_module(module)\\n                modules[name] = module\\n        finally:\\n            sys.dont_write_bytecode = old_bytecode\\n        return modules[\\"config\\"].LUARConfig, modules[\\"model\\"].LUAR\\n\\n    def tokenize(self, text: str) -> TokenizedEpisode:\\n        if not isinstance(text, str):\\n            raise TypeError(\\"Completion text must be an unchanged string\\")\\n        content = self.tokenizer(text, add_special_tokens=False, truncation=False,\\n                                 return_attention_mask=False, verbose=False)[\\"input_ids\\"]\\n        windows = max(1, (len(content) + self.settings.content_tokens - 1) // self.settings.content_tokens)\\n        input_ids = np.full((windows, self.settings.sequence_length), self.tokenizer.pad_token_id, dtype=np.int64)\\n        attention_mask = np.zeros_like(input_ids)\\n        for index in range(windows):\\n            start = index * self.settings.content_tokens\\n            chunk = content[start:start + self.settings.content_tokens]\\n            # Exact single-sequence RobertaProcessing template from reviewed\\n            # tokenizer.json. Transformers 5 removed the old build_inputs helper.\\n            prepared = [self.tokenizer.cls_token_id, *chunk, self.tokenizer.sep_token_id]\\n            if len(prepared) != len(chunk) + 2 or len(prepared) > self.settings.sequence_length:\\n                raise ValueError(\\"Unexpected special token construction\\")\\n            input_ids[index, :len(prepared)] = prepared\\n            attention_mask[index, :len(prepared)] = 1\\n        return TokenizedEpisode(input_ids, attention_mask, len(content))\\n\\n    def encode(self, texts: Sequence[str], progress: Callable[[dict], None] | None = None) -> np.ndarray:\\n        started = time.perf_counter()\\n        episodes = [self.tokenize(text) for text in texts]\\n        tokenization_seconds = time.perf_counter() - started\\n        grouped: dict[int, list[int]] = defaultdict(list)\\n        for index, episode in enumerate(episodes):\\n            grouped[episode.windows].append(index)\\n        result = np.empty((len(episodes), 512), dtype=np.float32)\\n        completed = 0\\n        batch_records = []\\n        with self.torch.inference_mode():\\n            for windows, indices in sorted(grouped.items()):\\n                # One long row is never truncated. document_batch_size still\\n                # bounds transformer activation memory when windows > the cap.\\n                rows_per_batch = max(1, self.settings.max_batch_windows // windows)\\n                for start in range(0, len(indices), rows_per_batch):\\n                    selected = indices[start:start + rows_per_batch]\\n                    batch_started = time.perf_counter()\\n                    inputs = {\\n                        name: self.torch.from_numpy(np.stack([getattr(episodes[i], name) for i in selected]))\\n                        for name in (\\"input_ids\\", \\"attention_mask\\")\\n                    }\\n                    raw = self.model(**inputs, document_batch_size=self.settings.document_batch_size)\\n                    norms = self.torch.linalg.vector_norm(raw, ord=2, dim=1, keepdim=True)\\n                    if tuple(raw.shape) != (len(selected), 512) or not self.torch.isfinite(raw).all() or (norms <= 0).any():\\n                        raise ValueError(\\"Invalid LUAR output shape, value or norm\\")\\n                    result[selected] = (raw / norms).cpu().numpy()\\n                    completed += len(selected)\\n                    record = {\\"batch\\": len(batch_records), \\"rows\\": len(selected), \\"windows_per_row\\": windows,\\n                              \\"completed_rows\\": completed, \\"total_rows\\": len(episodes),\\n                              \\"elapsed_seconds\\": time.perf_counter() - batch_started}\\n                    batch_records.append(record)\\n                    if progress:\\n                        progress(record)\\n        self.report = {\\"rows\\": len(episodes), \\"content_tokens\\": [e.content_tokens for e in episodes],\\n                       \\"window_counts\\": [e.windows for e in episodes],\\n                       \\"total_windows\\": sum(e.windows for e in episodes),\\n                       \\"tokenization_seconds\\": tokenization_seconds,\\n                       \\"elapsed_seconds\\": time.perf_counter() - started, \\"batches\\": batch_records}\\n        return result\\n\\n    def provenance(self) -> dict:\\n        return {\\"model_id\\": MODEL_ID, \\"revision\\": MODEL_REVISION,\\n                \\"manifest_sha256\\": sha256_file(self.model_dir / \\"manifest.json\\"),\\n                \\"model_files\\": self.manifest[\\"files\\"], \\"runtime\\": self.runtime,\\n                \\"encoder_source_sha256\\": sha256_file(__file__)}\\n", "encoder_v7.py": "\\"\\"\\"Frozen local StyleDistance encoder: raw text, complete windows, masked mean pooling.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import asdict, dataclass\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport platform\\nimport time\\nfrom typing import Callable, Sequence\\n\\nimport numpy as np\\n\\nMODEL_ID = \\"StyleDistance/styledistance\\"\\nMODEL_REVISION = \\"b7df5f0b0480773c097ba3121d83ca32b71015ca\\"\\nREVIEWED_SOURCE_SHA256 = {\'config.json\': \'b7bcc672f41211775356b1bb7b614453b91edc8425f58eb4ba2c71d6d8b1875a\', \'modules.json\': \'8f4b264b80206c830bebbdcae377e137925650a433b689343a63bdc9b3145460\', \'1_Pooling/config.json\': \'1c8bc7bd750c5c20d8707f1c5c578f5d69bb3d1d5ebcf4b2fde5128de154ec1c\', \'sentence_bert_config.json\': \'ec8e29d6dcb61b611b7d3fdd2982c4524e6ad985959fa7194eacfb655a8d0d51\', \'config_sentence_transformers.json\': \'fe35b3fc2768f364c610e91305d45aa600eda73a2613635a47db835f9cc4fb74\', \'tokenizer.json\': \'6e6a06a662f8bce9823abd4c2cda07dcb7e07a84153a2ed190b8d901516b3162\', \'tokenizer_config.json\': \'56f306b23b9233c94016f1b0b8aea3bc120676b378f70933908c9e62ab2daa36\', \'special_tokens_map.json\': \'06e405a36dfe4b9604f484f6a1e619af1a7f7d09e34a8555eb0b77b66318067f\'}\\nREQUIRED_MODEL_FILES = (*REVIEWED_SOURCE_SHA256, \\"model.safetensors\\", \\"vocab.json\\", \\"merges.txt\\", \\"README.md\\")\\n\\n\\ndef _json_native(value):\\n    \\"\\"\\"Transformers 5 loading diagnostics include sets; preserve them as lists.\\"\\"\\"\\n    if isinstance(value, dict):\\n        return {key: _json_native(item) for key, item in value.items()}\\n    if isinstance(value, set):\\n        return [_json_native(item) for item in sorted(value, key=repr)]\\n    if isinstance(value, (tuple, list)):\\n        return [_json_native(item) for item in value]\\n    if isinstance(value, np.generic):\\n        return value.item()\\n    return value\\n\\n\\ndef sha256_file(path: str | Path) -> str:\\n    digest = hashlib.sha256()\\n    with Path(path).open(\\"rb\\") as handle:\\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef sha256_text(text: str) -> str:\\n    if not isinstance(text, str):\\n        raise TypeError(\\"Completion text must be an unchanged string\\")\\n    return hashlib.sha256(text.encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef build_row_manifest(ids: Sequence[str], texts: Sequence[str]) -> list[dict]:\\n    if len(ids) != len(texts):\\n        raise ValueError(\\"IDs and texts have different lengths\\")\\n    if len(set(ids)) != len(ids) or any(not isinstance(x, str) or not x for x in ids):\\n        raise ValueError(\\"Row IDs must be unique nonempty strings\\")\\n    return [{\\"row_index\\": i, \\"id\\": row_id, \\"text_sha256\\": sha256_text(text)}\\n            for i, (row_id, text) in enumerate(zip(ids, texts, strict=True))]\\n\\n\\ndef validate_model_manifest(model_dir: str | Path) -> dict:\\n    \\"\\"\\"Verify pinned identity and every model file before loading the local checkpoint.\\"\\"\\"\\n    root = Path(model_dir).resolve(strict=True)\\n    manifest = json.loads((root / \\"manifest.json\\").read_text())\\n    if manifest.get(\\"model_id\\") != MODEL_ID or manifest.get(\\"revision\\") != MODEL_REVISION:\\n        raise ValueError(\\"Unexpected StyleDistance model identity or revision\\")\\n    entries = {entry[\\"path\\"]: entry for entry in manifest[\\"files\\"]}\\n    if len(entries) != len(manifest[\\"files\\"]) or not set(REQUIRED_MODEL_FILES).issubset(entries):\\n        raise ValueError(\\"Missing or duplicate model manifest entries\\")\\n    for name in entries:\\n        if Path(name).is_absolute() or \\"..\\" in Path(name).parts:\\n            raise ValueError(\\"Unsafe model manifest path\\")\\n        path = root / name\\n        if path.is_symlink() or not path.is_file():\\n            raise ValueError(f\\"Expected a local regular model file: {name}\\")\\n        if path.stat().st_size != entries[name][\\"size_bytes\\"] or sha256_file(path) != entries[name][\\"sha256\\"]:\\n            raise ValueError(f\\"Model manifest mismatch: {name}\\")\\n    for name, expected in REVIEWED_SOURCE_SHA256.items():\\n        if entries[name][\\"sha256\\"] != expected:\\n            raise ValueError(f\\"Source differs from the reviewed revision: {name}\\")\\n    return manifest\\n\\n\\ndef load_embedding_cache(cache_dir: str | Path, split: str, ids: Sequence[str],\\n                         texts: Sequence[str], *, model_dir: str | Path,\\n                         plan_path: str | Path, source_path: str | Path | None = None) -> tuple[np.ndarray, dict]:\\n    \\"\\"\\"Reject stale, reordered, edited or mismatched cached vectors before fitting.\\"\\"\\"\\n    if split not in (\\"train\\", \\"test\\"):\\n        raise ValueError(\\"Expected train or test cache\\")\\n    root = Path(cache_dir)\\n    metadata = json.loads((root / f\\"{split}-embeddings.json\\").read_text())\\n    model_manifest = validate_model_manifest(model_dir)\\n    provenance = metadata[\\"provenance\\"]\\n    expected_rows = build_row_manifest(ids, texts)\\n    plan = json.loads(Path(plan_path).read_text())\\n    checks = {\\n        \\"status\\": metadata[\\"status\\"] == \\"complete\\",\\n        \\"split\\": metadata[\\"split\\"] == split,\\n        \\"rows\\": metadata[\\"rows\\"] == expected_rows,\\n        \\"shape\\": metadata[\\"shape\\"] == [len(ids), 768],\\n        \\"dtype\\": metadata[\\"dtype\\"] == \\"float32\\",\\n        \\"plan\\": metadata[\\"plan_sha256\\"] == sha256_file(plan_path),\\n        \\"model_identity\\": provenance[\\"model_id\\"] == MODEL_ID and provenance[\\"revision\\"] == MODEL_REVISION,\\n        \\"model_manifest\\": provenance[\\"manifest_sha256\\"] == sha256_file(Path(model_dir) / \\"manifest.json\\"),\\n        \\"model_files\\": provenance[\\"model_files\\"] == model_manifest[\\"files\\"],\\n        \\"settings\\": provenance[\\"runtime\\"][\\"settings\\"] == plan[\\"encoder\\"],\\n        \\"encoder_source\\": provenance[\\"encoder_source_sha256\\"] == sha256_file(__file__),\\n        \\"extraction_source\\": metadata[\\"script_sha256\\"] == sha256_file(Path(__file__).resolve().parents[2] / \\"scripts\\" / \\"extract_embeddings_v7.py\\"),\\n    }\\n    if source_path is not None:\\n        checks[\\"source_csv\\"] = metadata[\\"source_csv_sha256\\"] == sha256_file(source_path)\\n    if not all(checks.values()):\\n        raise ValueError(f\\"Embedding cache provenance mismatch: {[key for key, valid in checks.items() if not valid]}\\")\\n    artifact = metadata[\\"artifact\\"]\\n    path = root / f\\"{split}-embeddings.npz\\"\\n    if artifact[\\"path\\"] != path.name or path.stat().st_size != artifact[\\"size_bytes\\"] or sha256_file(path) != artifact[\\"sha256\\"]:\\n        raise ValueError(\\"Embedding cache artifact checksum mismatch\\")\\n    with np.load(path, allow_pickle=False) as cache:\\n        embeddings = cache[\\"embeddings\\"]\\n        if cache[\\"ids\\"].tolist() != list(ids) or cache[\\"text_sha256\\"].tolist() != [row[\\"text_sha256\\"] for row in expected_rows]:\\n            raise ValueError(\\"Embedding cache row/text mismatch\\")\\n        tokens, windows = cache[\\"content_tokens\\"], cache[\\"window_counts\\"]\\n        if tokens.shape != (len(ids),) or windows.shape != (len(ids),) or (tokens < 0).any():\\n            raise ValueError(\\"Embedding cache token counts are invalid\\")\\n        if not np.array_equal(windows, np.maximum(1, (tokens + 509) // 510)):\\n            raise ValueError(\\"Embedding cache window coverage is inconsistent\\")\\n        if tokens.tolist() != metadata[\\"report\\"][\\"content_tokens\\"] or windows.tolist() != metadata[\\"report\\"][\\"window_counts\\"]:\\n            raise ValueError(\\"Embedding cache count metadata mismatch\\")\\n        if embeddings.shape != (len(ids), 768) or embeddings.dtype != np.float32 or not np.isfinite(embeddings).all():\\n            raise ValueError(\\"Embedding cache vectors are invalid\\")\\n        if not np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=2e-6, rtol=0):\\n            raise ValueError(\\"Embedding cache vectors are not unit normalized\\")\\n    return embeddings, metadata\\n\\n\\n@dataclass(frozen=True)\\nclass EncoderSettings:\\n    content_tokens: int = 510\\n    sequence_length: int = 512\\n    max_batch_windows: int = 8\\n    cpu_threads: int = 2\\n    device: str = \\"cpu\\"\\n    dtype: str = \\"float32\\"\\n\\n    def __post_init__(self):\\n        if self.content_tokens != 510 or self.sequence_length != 512:\\n            raise ValueError(\\"V7 fixes 510 content tokens plus two special tokens\\")\\n        if min(self.max_batch_windows, self.cpu_threads) < 1:\\n            raise ValueError(\\"Batch bounds and thread count must be positive\\")\\n        if self.device != \\"cpu\\" or self.dtype != \\"float32\\":\\n            raise ValueError(\\"V7 fixes CPU float32 inference\\")\\n\\n\\n@dataclass(frozen=True)\\nclass TokenizedPassage:\\n    input_ids: list[list[int]]\\n    content_tokens: int\\n\\n    @property\\n    def windows(self) -> int:\\n        return len(self.input_ids)\\n\\n\\ndef pool_window_embeddings(hidden, attention_mask):\\n    \\"\\"\\"Released ST 2.7 mean pooling; mask includes the two special tokens.\\"\\"\\"\\n    mask = attention_mask.unsqueeze(-1).expand(hidden.size()).to(hidden.dtype)\\n    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)\\n\\n\\nclass FrozenStyleDistanceEncoder:\\n    \\"\\"\\"Standard RoBERTa loaded from verified files without remote custom code.\\"\\"\\"\\n\\n    def __init__(self, model_dir: str | Path, settings: EncoderSettings | None = None):\\n        self.settings = settings or EncoderSettings()\\n        self.model_dir = Path(model_dir).resolve(strict=True)\\n        self.manifest = validate_model_manifest(self.model_dir)\\n        for name in (\\"HF_HUB_OFFLINE\\", \\"TRANSFORMERS_OFFLINE\\", \\"HF_HUB_DISABLE_TELEMETRY\\"):\\n            os.environ[name] = \\"1\\"\\n        os.environ[\\"TOKENIZERS_PARALLELISM\\"] = \\"false\\"\\n        import torch\\n        from transformers import AutoModel, AutoTokenizer\\n\\n        self.torch = torch\\n        torch.set_num_threads(self.settings.cpu_threads)\\n        torch.use_deterministic_algorithms(True)\\n        self.model, loading_info = AutoModel.from_pretrained(\\n            str(self.model_dir), local_files_only=True, trust_remote_code=False,\\n            use_safetensors=True, dtype=torch.float32, output_loading_info=True,\\n        )\\n        loading_info = _json_native(loading_info)\\n        if any(loading_info.values()):\\n            raise ValueError(f\\"Checkpoint did not load exactly: {loading_info}\\")\\n        self.model.to(device=\\"cpu\\", dtype=torch.float32).eval()\\n        self.model.requires_grad_(False)\\n        if self.model.config.model_type != \\"roberta\\" or self.model.config.hidden_size != 768:\\n            raise ValueError(\\"Unexpected StyleDistance architecture\\")\\n        self.tokenizer = AutoTokenizer.from_pretrained(\\n            str(self.model_dir), local_files_only=True, trust_remote_code=False, use_fast=True,\\n        )\\n        if self.tokenizer.num_special_tokens_to_add(pair=False) != 2:\\n            raise ValueError(\\"Expected the reviewed two-token RoBERTa template\\")\\n        if (self.tokenizer.cls_token_id, self.tokenizer.sep_token_id, self.tokenizer.pad_token_id) != (0, 2, 1):\\n            raise ValueError(\\"Unexpected special-token IDs\\")\\n        self.report: dict = {}\\n        self.runtime = {\\n            \\"python\\": platform.python_version(), \\"platform\\": platform.platform(),\\n            **{name: importlib.metadata.version(name) for name in\\n               (\\"torch\\", \\"transformers\\", \\"huggingface_hub\\", \\"tokenizers\\", \\"safetensors\\", \\"numpy\\")},\\n            \\"parameter_count\\": sum(p.numel() for p in self.model.parameters()),\\n            \\"parameter_dtype\\": str(next(self.model.parameters()).dtype),\\n            \\"backbone_attention\\": self.model.config._attn_implementation,\\n            \\"torch_num_threads\\": torch.get_num_threads(),\\n            \\"settings\\": asdict(self.settings), \\"loading_info\\": loading_info,\\n            \\"custom_remote_code\\": False,\\n            \\"pooling_accumulation\\": \\"float32 window mean; float64 token-count-weighted passage sum; final float32 unit vector\\",\\n        }\\n\\n    def tokenize(self, text: str) -> TokenizedPassage:\\n        if not isinstance(text, str):\\n            raise TypeError(\\"Completion text must be an unchanged string\\")\\n        content = self.tokenizer(text, add_special_tokens=False, truncation=False,\\n                                 return_attention_mask=False, verbose=False)[\\"input_ids\\"]\\n        chunks = [content[i:i + self.settings.content_tokens]\\n                  for i in range(0, len(content), self.settings.content_tokens)] or [[]]\\n        return TokenizedPassage(\\n            [[self.tokenizer.cls_token_id, *chunk, self.tokenizer.sep_token_id] for chunk in chunks],\\n            len(content),\\n        )\\n\\n    def encode(self, texts: Sequence[str], progress: Callable[[dict], None] | None = None) -> np.ndarray:\\n        started = time.perf_counter()\\n        passages = [self.tokenize(text) for text in texts]\\n        tokenization_seconds = time.perf_counter() - started\\n        # Only token IDs are retained for all passages. Transformer states are\\n        # released each bounded window batch, including for arbitrarily long rows.\\n        windows = [(row, ids) for row, passage in enumerate(passages) for ids in passage.input_ids]\\n        sums = np.zeros((len(passages), 768), dtype=np.float64)\\n        weights = np.zeros(len(passages), dtype=np.int64)\\n        records = []\\n        with self.torch.inference_mode():\\n            for start in range(0, len(windows), self.settings.max_batch_windows):\\n                selected = windows[start:start + self.settings.max_batch_windows]\\n                batch_started = time.perf_counter()\\n                width = max(len(ids) for _, ids in selected)\\n                ids_array = np.full((len(selected), width), self.tokenizer.pad_token_id, dtype=np.int64)\\n                masks = np.zeros_like(ids_array)\\n                for index, (_, ids) in enumerate(selected):\\n                    ids_array[index, :len(ids)] = ids\\n                    masks[index, :len(ids)] = 1\\n                inputs = {\\"input_ids\\": self.torch.from_numpy(ids_array),\\n                          \\"attention_mask\\": self.torch.from_numpy(masks)}\\n                hidden = self.model(**inputs).last_hidden_state\\n                pooled = pool_window_embeddings(hidden, inputs[\\"attention_mask\\"]).cpu().numpy()\\n                if pooled.shape != (len(selected), 768) or not np.isfinite(pooled).all():\\n                    raise ValueError(\\"Invalid StyleDistance pooled output\\")\\n                for index, (row, ids) in enumerate(selected):\\n                    sums[row] += pooled[index].astype(np.float64) * len(ids)\\n                    weights[row] += len(ids)\\n                del hidden, pooled, inputs\\n                record = {\\"batch\\": len(records), \\"windows\\": len(selected), \\"padded_tokens\\": width,\\n                          \\"completed_windows\\": start + len(selected), \\"total_windows\\": len(windows),\\n                          \\"elapsed_seconds\\": time.perf_counter() - batch_started}\\n                records.append(record)\\n                if progress:\\n                    progress(record)\\n        means = sums / np.maximum(weights, 1)[:, None]\\n        norms = np.linalg.norm(means, axis=1, keepdims=True)\\n        if not np.isfinite(means).all() or (norms <= 0).any():\\n            raise ValueError(\\"Invalid StyleDistance passage mean or norm\\")\\n        result = (means / norms).astype(np.float32)\\n        self.report = {\\"rows\\": len(passages), \\"content_tokens\\": [p.content_tokens for p in passages],\\n                       \\"window_counts\\": [p.windows for p in passages], \\"total_windows\\": len(windows),\\n                       \\"tokenization_seconds\\": tokenization_seconds,\\n                       \\"elapsed_seconds\\": time.perf_counter() - started, \\"batches\\": records}\\n        return result\\n\\n    def provenance(self) -> dict:\\n        return {\\"model_id\\": MODEL_ID, \\"revision\\": MODEL_REVISION,\\n                \\"manifest_sha256\\": sha256_file(self.model_dir / \\"manifest.json\\"),\\n                \\"model_files\\": self.manifest[\\"files\\"], \\"runtime\\": self.runtime,\\n                \\"encoder_source_sha256\\": sha256_file(__file__)}\\n\\n\\n", "ensemble_v2.py": "\\"\\"\\"Fixed-weight combinations of text-only attribution evidence.\\"\\"\\"\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, ClassifierMixin\\nfrom sklearn.utils.validation import check_is_fitted\\n\\nfrom name_that_llm.baseline import LABELS, fit_checked\\nfrom name_that_llm.sparse_v2 import SPARSE_CANDIDATES, build_sparse_pipeline\\nfrom name_that_llm.stylometry import build_style_pipeline\\nfrom name_that_llm.generative import GenerativeConfig, build_generative\\n\\n\\ndef normalize_evidence(scores, *, probabilities=False):\\n    \\"\\"\\"Put different score types on a common per-passage scale, without fitting.\\"\\"\\"\\n    values = np.asarray(scores, dtype=np.float64)\\n    if values.ndim != 2 or values.shape[1] != len(LABELS) or not np.isfinite(values).all():\\n        raise ValueError(\\"Expected finite passage-by-label evidence.\\")\\n    if probabilities:\\n        if np.any(values < 0) or not np.allclose(values.sum(axis=1), 1.0, atol=1e-5):\\n            raise ValueError(\\"Invalid class probabilities.\\")\\n        values = np.log(np.clip(values, 1e-12, 1.0))\\n    centered = values - values.mean(axis=1, keepdims=True)\\n    return centered / np.maximum(centered.std(axis=1, keepdims=True), 1e-8)\\n\\n\\ndef make_component(name):\\n    family, variant = name.split(\\":\\", 1)\\n    if family == \\"sparse\\":\\n        return build_sparse_pipeline(next(c for c in SPARSE_CANDIDATES if c.name == variant))\\n    if family == \\"style\\":\\n        return build_style_pipeline(variant)\\n    if family == \\"generative\\":\\n        return build_generative(GenerativeConfig(variant=variant))\\n    raise ValueError(f\\"Unknown component: {name}\\")\\n\\n\\nclass EvidenceEnsemble(ClassifierMixin, BaseEstimator):\\n    def __init__(self, components, weights):\\n        self.components = components\\n        self.weights = weights\\n\\n    def fit(self, X, y):\\n        weights = np.asarray(self.weights, dtype=float)\\n        if len(weights) != len(self.components) or not len(weights):\\n            raise ValueError(\\"Each component needs a weight.\\")\\n        if not np.isfinite(weights).all() or (weights <= 0).any():\\n            raise ValueError(\\"Weights must be positive and finite.\\")\\n        self.weights_ = weights / weights.sum()\\n        self.models_ = []\\n        self.classes_ = np.asarray(LABELS)\\n        for name in self.components:\\n            model = fit_checked(make_component(name), X, y)\\n            if tuple(model.classes_) != LABELS:\\n                raise ValueError(\\"Component class order differs from competition labels.\\")\\n            self.models_.append(model)\\n        return self\\n\\n    def decision_function(self, X):\\n        check_is_fitted(self, [\\"models_\\", \\"weights_\\", \\"classes_\\"])\\n        combined = None\\n        for model, weight in zip(self.models_, self.weights_):\\n            is_probability = hasattr(model, \\"predict_proba\\")\\n            raw = model.predict_proba(X) if is_probability else model.decision_function(X)\\n            evidence = normalize_evidence(raw, probabilities=is_probability)\\n            combined = weight * evidence if combined is None else combined + weight * evidence\\n        return combined\\n\\n    def predict(self, X):\\n        return self.classes_[self.decision_function(X).argmax(axis=1)]\\n", "ensemble_v3.py": "\\"\\"\\"Fixed, passage-local ensembles for the separately recorded v3 experiment.\\"\\"\\"\\n\\nimport numpy as np\\nfrom scipy.special import softmax\\nfrom sklearn.base import BaseEstimator, ClassifierMixin\\nfrom sklearn.utils.validation import check_is_fitted\\n\\nfrom name_that_llm.baseline import LABELS\\nfrom name_that_llm.ensemble_v2 import make_component, normalize_evidence\\n\\n\\ndef make_v3_component(name):\\n    if name.startswith(\\"kernel:\\"):\\n        from name_that_llm.kernel_v3 import build_kernel_pipeline\\n        return build_kernel_pipeline(name.split(\\":\\", 1)[1])\\n    if name.startswith(\\"linear:\\"):\\n        from name_that_llm.linear_v3 import build_linear_pipeline\\n        return build_linear_pipeline(name.split(\\":\\", 1)[1])\\n    if name.startswith(\\"boosting:\\"):\\n        from name_that_llm.boosting_v3 import build_boosting_pipeline\\n        return build_boosting_pipeline(name.split(\\":\\", 1)[1])\\n    if name.startswith(\\"topic:\\"):\\n        from name_that_llm.topic_v3 import build_topic_pipeline\\n        return build_topic_pipeline(name.split(\\":\\", 1)[1])\\n    return make_component(name)\\n\\n\\ndef combine_evidence(evidence, weights, pooling=\\"evidence\\"):\\n    \\"\\"\\"Combine aligned evidence without learning from inference rows.\\"\\"\\"\\n    arrays = [np.asarray(values, dtype=float) for values in evidence]\\n    weights = np.asarray(weights, dtype=float)\\n    if not arrays or len(arrays) != len(weights):\\n        raise ValueError(\\"One weight is required per component.\\")\\n    if not np.isfinite(weights).all() or (weights <= 0).any():\\n        raise ValueError(\\"Weights must be positive and finite.\\")\\n    if pooling not in {\\"evidence\\", \\"probability\\"}:\\n        raise ValueError(\\"Unknown pooling method.\\")\\n    shape = arrays[0].shape\\n    if len(shape) != 2 or shape[1] != len(LABELS):\\n        raise ValueError(\\"Expected passage-by-class scores.\\")\\n    if any(a.shape != shape or not np.isfinite(a).all() for a in arrays):\\n        raise ValueError(\\"Component evidence must be finite and aligned.\\")\\n    if pooling == \\"probability\\":\\n        arrays = [softmax(a, axis=1) for a in arrays]\\n    return sum(a * w for a, w in zip(arrays, weights)) / weights.sum()\\n\\n\\nclass V3Ensemble(ClassifierMixin, BaseEstimator):\\n    def __init__(self, components, weights, pooling=\\"evidence\\"):\\n        self.components = components\\n        self.weights = weights\\n        self.pooling = pooling\\n\\n    def fit(self, X, y, groups=None):\\n        # Check the recipe before starting any expensive fits.\\n        combine_evidence(\\n            [np.zeros((1, len(LABELS))) for _ in self.components],\\n            self.weights, self.pooling,\\n        )\\n        self.classes_ = np.asarray(LABELS)\\n        self.models_ = []\\n        for name in self.components:\\n            model = make_v3_component(name)\\n            if getattr(model, \\"requires_groups\\", False):\\n                if groups is None:\\n                    raise ValueError(\\"This component needs training prompt groups.\\")\\n                model.fit(X, y, groups=groups)\\n            else:\\n                model.fit(X, y)\\n            if tuple(model.classes_) != LABELS:\\n                raise ValueError(\\"Component class order differs from the competition labels.\\")\\n            self.models_.append(model)\\n        return self\\n\\n    def decision_function(self, X):\\n        check_is_fitted(self, [\\"classes_\\", \\"models_\\"])\\n        evidence = []\\n        for model in self.models_:\\n            probability = hasattr(model, \\"predict_proba\\")\\n            raw = model.predict_proba(X) if probability else model.decision_function(X)\\n            evidence.append(normalize_evidence(raw, probabilities=probability))\\n        return combine_evidence(evidence, self.weights, self.pooling)\\n\\n    def predict(self, X):\\n        return self.classes_[self.decision_function(X).argmax(axis=1)]\\n", "generative.py": "\\"\\"\\"Small, predeclared count-based attribution models for improvement-v2.\\n\\nThe NB-SVM variant implements the log-count-ratio weighting in Wang and\\nManning (2012), with character features and one-versus-rest multiclass fitting.\\nIt does not use their optional coefficient interpolation. This is an adaptation,\\nnot a reproduction of their sentiment experiment.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nimport warnings\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, ClassifierMixin\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.feature_extraction.text import CountVectorizer\\nfrom sklearn.naive_bayes import ComplementNB, MultinomialNB\\nfrom sklearn.svm import LinearSVC\\nfrom sklearn.utils.validation import check_is_fitted\\n\\n\\nLOGGER = logging.getLogger(\\"name_that_llm.generative\\")\\n\\n\\n@dataclass(frozen=True)\\nclass GenerativeConfig:\\n    \\"\\"\\"Three candidates are fixed before any improvement-v2 validation scores.\\"\\"\\"\\n\\n    variant: str = \\"char_nbsvm\\"\\n    alpha: float = 1.0\\n    c: float = 1.0\\n    min_df: int = 2\\n    max_features: int = 250_000\\n    max_iter: int = 10_000\\n    tolerance: float = 1e-4\\n    seed: int = 42\\n\\n\\nGENERATIVE_CONFIGS = tuple(\\n    GenerativeConfig(variant=variant)\\n    for variant in (\\"char_mnb\\", \\"char_cnb\\", \\"char_nbsvm\\")\\n)\\n\\n\\nclass GenerativeAttribution(ClassifierMixin, BaseEstimator):\\n    \\"\\"\\"Text-to-score estimator; only ``fit`` constructs a vocabulary.\\n\\n    ``decision_function`` columns follow ``classes_`` (lexical label order).\\n    Scores are uncalibrated. NB scores are per-feature average log evidence;\\n    this preserves each row\'s predicted class while reducing length dependence.\\n    \\"\\"\\"\\n\\n    def __init__(self, config: GenerativeConfig | None = None):\\n        self.config = config\\n\\n    def fit(self, text, labels):\\n        config = self.config or GenerativeConfig()\\n        if config.variant not in {c.variant for c in GENERATIVE_CONFIGS}:\\n            raise ValueError(f\\"Unknown generative variant: {config.variant}\\")\\n        if config.alpha <= 0:\\n            raise ValueError(\\"Smoothing must be strictly positive.\\")\\n        labels = np.asarray(labels)\\n        self.classes_ = np.unique(labels)\\n        if len(self.classes_) < 2:\\n            raise ValueError(\\"At least two labels are required.\\")\\n        self.vectorizer_ = CountVectorizer(\\n            analyzer=\\"char\\",\\n            ngram_range=(3, 5) if config.variant == \\"char_nbsvm\\" else (2, 5),\\n            lowercase=False, strip_accents=None, binary=True,\\n            min_df=config.min_df, max_features=config.max_features,\\n            dtype=np.float64,\\n        )\\n        counts = self.vectorizer_.fit_transform(text)\\n        self.variant_ = config.variant\\n        self.solver_iterations_ = []\\n        if config.variant == \\"char_mnb\\":\\n            self.classifier_ = MultinomialNB(alpha=config.alpha, fit_prior=False)\\n            self.classifier_.fit(counts, labels)\\n        elif config.variant == \\"char_cnb\\":\\n            self.classifier_ = ComplementNB(\\n                alpha=config.alpha, fit_prior=False, norm=True,\\n            )\\n            self.classifier_.fit(counts, labels)\\n        else:\\n            total = np.asarray(counts.sum(axis=0)).ravel()\\n            self.coef_ = np.empty((len(self.classes_), counts.shape[1]))\\n            self.intercept_ = np.empty(len(self.classes_))\\n            for index, label in enumerate(self.classes_):\\n                target = labels == label\\n                positive_count = np.asarray(counts[target].sum(axis=0)).ravel()\\n                positive = positive_count + config.alpha\\n                negative = total - positive_count + config.alpha\\n                ratio = np.log(positive / positive.sum()) - np.log(\\n                    negative / negative.sum()\\n                )\\n                weighted = counts.multiply(ratio).tocsr()\\n                classifier = LinearSVC(\\n                    C=config.c, dual=True, random_state=config.seed,\\n                    tol=config.tolerance, max_iter=config.max_iter,\\n                )\\n                with warnings.catch_warnings(record=True) as caught:\\n                    warnings.simplefilter(\\"always\\")\\n                    classifier.fit(weighted, target)\\n                for warning in caught:\\n                    LOGGER.warning(\\"%s label=%s %s: %s\\", config.variant, label,\\n                                   warning.category.__name__, warning.message)\\n                if any(issubclass(w.category, ConvergenceWarning) for w in caught):\\n                    raise RuntimeError(f\\"NB-SVM did not converge for {label}.\\")\\n                # Fold the NB weighting into coefficients for cheap inference.\\n                self.coef_[index] = classifier.coef_[0] * ratio\\n                self.intercept_[index] = classifier.intercept_[0]\\n                self.solver_iterations_.append(int(classifier.n_iter_))\\n        self.n_features_in_ = counts.shape[1]\\n        return self\\n\\n    def decision_function(self, text):\\n        check_is_fitted(self, [\\"classes_\\", \\"vectorizer_\\", \\"variant_\\"])\\n        counts = self.vectorizer_.transform(text)\\n        if self.variant_ == \\"char_nbsvm\\":\\n            scores = np.asarray(counts @ self.coef_.T) + self.intercept_\\n        else:\\n            scores = np.asarray(counts @ self.classifier_.feature_log_prob_.T)\\n            if self.variant_ == \\"char_mnb\\":\\n                scores += self.classifier_.class_log_prior_\\n            denominator = np.maximum(\\n                np.asarray(counts.sum(axis=1)).ravel(), 1.0,\\n            )\\n            scores /= denominator[:, None]\\n        if not np.isfinite(scores).all():\\n            raise ValueError(\\"Model produced non-finite scores.\\")\\n        return scores\\n\\n    def predict(self, text):\\n        return self.classes_[self.decision_function(text).argmax(axis=1)]\\n\\n\\ndef build_generative(config: GenerativeConfig) -> GenerativeAttribution:\\n    return GenerativeAttribution(config)\\n", "kernel_v3.py": "\\"\\"\\"Two fixed nonlinear style-kernel hypotheses for the v3 second stage.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, ClassifierMixin\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.metrics.pairwise import euclidean_distances\\nfrom sklearn.multiclass import OneVsRestClassifier\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import SVC\\nfrom sklearn.utils.validation import check_is_fitted\\n\\nfrom name_that_llm.stylometry import StyleFeatures\\n\\n\\nKERNEL_CANDIDATES = {\\n    \\"style_rbf_c4\\": {\\n        \\"C\\": 4.0, \\"char_weight\\": 0.0, \\"style_weight\\": 1.0,\\n        \\"features\\": \\"465 fixed style features -> training-only StandardScaler\\",\\n        \\"gamma\\": \\"inverse median positive squared training style distance\\",\\n        \\"bandwidth_sample_size\\": 256, \\"selection\\": \\"lowest SHA256(text) values\\",\\n        \\"classifier\\": \\"OneVsRestClassifier(SVC(kernel=precomputed)), n_jobs=1\\",\\n    },\\n    \\"additive_char_style_rbf_c4\\": {\\n        \\"C\\": 4.0, \\"char_weight\\": 0.5, \\"style_weight\\": 0.5,\\n        \\"features\\": \\"char3-5 TFIDF cosine + RBF standardized fixed style features\\",\\n        \\"gamma\\": \\"inverse median positive squared training style distance\\",\\n        \\"bandwidth_sample_size\\": 256, \\"selection\\": \\"lowest SHA256(text) values\\",\\n        \\"classifier\\": \\"OneVsRestClassifier(SVC(kernel=precomputed)), n_jobs=1\\",\\n    },\\n}\\n\\n\\ndef bandwidth_sample_indices(texts, max_samples=256):\\n    \\"\\"\\"Choose training passages reproducibly and independently of row order.\\"\\"\\"\\n    return np.asarray(sorted(\\n        range(len(texts)),\\n        key=lambda index: (hashlib.sha256(texts[index].encode(\\"utf-8\\")).digest(), texts[index]),\\n    )[:max_samples], dtype=int)\\n\\n\\ndef median_style_gamma(styles, indices):\\n    subset = np.asarray(styles, dtype=np.float64)[indices]\\n    squared = euclidean_distances(subset, squared=True)\\n    distances = squared[np.triu_indices(len(subset), k=1)]\\n    positive = distances[distances > 1e-12]\\n    median = float(np.median(positive)) if len(positive) else 1.0\\n    return 1.0 / median\\n\\n\\ndef combine_style_char_kernel(styles_left, styles_right, *, gamma,\\n                              char_left=None, char_right=None, char_weight=0.0):\\n    \\"\\"\\"RBF style similarity plus optional normalized character inner product.\\"\\"\\"\\n    if not np.isfinite(gamma) or gamma <= 0 or not 0 <= char_weight <= 1:\\n        raise ValueError(\\"Invalid kernel bandwidth or mixture weight.\\")\\n    squared = euclidean_distances(styles_left, styles_right, squared=True)\\n    kernel = np.exp(-gamma * squared)\\n    if char_weight:\\n        if char_left is None or char_right is None:\\n            raise ValueError(\\"Character features are required for an additive kernel.\\")\\n        inner = char_left @ char_right.T\\n        if hasattr(inner, \\"toarray\\"):\\n            inner = inner.toarray()\\n        kernel = (1.0 - char_weight) * kernel + char_weight * np.asarray(inner)\\n    if not np.isfinite(kernel).all():\\n        raise ValueError(\\"Non-finite kernel values.\\")\\n    return np.ascontiguousarray(kernel, dtype=np.float64)\\n\\n\\nclass StyleKernelClassifier(ClassifierMixin, BaseEstimator):\\n    \\"\\"\\"Train-only kernel state; score each text against stored training features.\\"\\"\\"\\n\\n    def __init__(self, C=4.0, char_weight=0.0, bandwidth_sample_size=256, batch_size=256):\\n        self.C = C\\n        self.char_weight = char_weight\\n        self.bandwidth_sample_size = bandwidth_sample_size\\n        self.batch_size = batch_size\\n\\n    def fit(self, X, y):\\n        texts, labels = list(X), np.asarray(y)\\n        if self.C <= 0 or not 0 <= self.char_weight <= 1:\\n            raise ValueError(\\"Invalid classifier configuration.\\")\\n        if self.bandwidth_sample_size < 2 or self.batch_size < 1:\\n            raise ValueError(\\"Invalid bandwidth sample or prediction batch size.\\")\\n        if labels.ndim != 1 or len(texts) != len(labels) or len(texts) < 2:\\n            raise ValueError(\\"Expected aligned texts and labels.\\")\\n        self.extractor_ = StyleFeatures()\\n        self.scaler_ = StandardScaler()\\n        self.training_styles_ = self.scaler_.fit_transform(self.extractor_.transform(texts))\\n        sample = bandwidth_sample_indices(texts, self.bandwidth_sample_size)\\n        self.gamma_ = median_style_gamma(self.training_styles_, sample)\\n        self.n_bandwidth_samples_ = len(sample)\\n        self.training_chars_ = None\\n        if self.char_weight:\\n            self.vectorizer_ = TfidfVectorizer(\\n                analyzer=\\"char\\", ngram_range=(3, 5), min_df=2,\\n                max_features=250_000, lowercase=False, strip_accents=None,\\n                sublinear_tf=True, norm=\\"l2\\", dtype=np.float64,\\n            )\\n            self.training_chars_ = self.vectorizer_.fit_transform(texts)\\n        gram = combine_style_char_kernel(\\n            self.training_styles_, self.training_styles_, gamma=self.gamma_,\\n            char_left=self.training_chars_, char_right=self.training_chars_,\\n            char_weight=self.char_weight,\\n        )\\n        self.classifier_ = OneVsRestClassifier(\\n            SVC(C=self.C, kernel=\\"precomputed\\", tol=1e-3, cache_size=256,\\n                probability=False, max_iter=-1, random_state=42),\\n            n_jobs=1,\\n        )\\n        self.classifier_.fit(gram, labels)\\n        self.classes_ = self.classifier_.classes_\\n        if len(self.classes_) < 2:\\n            raise ValueError(\\"Need at least two classes.\\")\\n        return self\\n\\n    def _kernel(self, texts):\\n        styles = self.scaler_.transform(self.extractor_.transform(texts))\\n        chars = self.vectorizer_.transform(texts) if self.char_weight else None\\n        return combine_style_char_kernel(\\n            styles, self.training_styles_, gamma=self.gamma_,\\n            char_left=chars, char_right=self.training_chars_,\\n            char_weight=self.char_weight,\\n        )\\n\\n    def decision_function(self, X):\\n        check_is_fitted(self, [\\"classifier_\\", \\"classes_\\", \\"gamma_\\"])\\n        if isinstance(X, str):\\n            raise ValueError(\\"Expected a sequence of passages.\\")\\n        texts = list(X)\\n        if not texts:\\n            return np.empty((0, len(self.classes_)), dtype=np.float64)\\n        batches = []\\n        for start in range(0, len(texts), self.batch_size):\\n            scores = self.classifier_.decision_function(self._kernel(texts[start:start + self.batch_size]))\\n            if scores.ndim == 1:\\n                scores = np.column_stack([-scores, scores])\\n            batches.append(scores)\\n        return np.concatenate(batches)\\n\\n    def predict(self, X):\\n        return self.classes_[self.decision_function(X).argmax(axis=1)]\\n\\n\\ndef build_kernel_pipeline(name):\\n    if name == \\"style_rbf_c4\\":\\n        return StyleKernelClassifier(C=4.0, char_weight=0.0)\\n    if name == \\"additive_char_style_rbf_c4\\":\\n        return StyleKernelClassifier(C=4.0, char_weight=0.5)\\n    raise ValueError(f\\"Unknown kernel candidate: {name}\\")\\n", "linear_v3.py": "\\"\\"\\"Joint multinomial text/style models for the frozen v3 second stage.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import Normalizer, StandardScaler\\n\\nfrom name_that_llm.stylometry import StyleFeatures\\n\\n\\nLINEAR_CANDIDATES = {\\n    \\"joint_char_style_c4\\": {\\"C\\": 4.},\\n    \\"joint_char_style_c16\\": {\\"C\\": 16.},\\n}\\n\\nLINEAR_COMMON = {\\n    \\"char_features\\": {\\n        \\"analyzer\\": \\"char\\", \\"ngram_range\\": [1, 5], \\"min_df\\": 2,\\n        \\"max_features\\": 300000, \\"lowercase\\": False, \\"sublinear_tf\\": True,\\n        \\"dtype\\": \\"float64\\",\\n    },\\n    \\"style_features\\": \\"465 StyleFeatures -> StandardScaler -> L2 Normalizer\\",\\n    \\"union_weights\\": {\\"char\\": 1., \\"style\\": .5},\\n    \\"union_postprocess\\": \\"L2 Normalizer\\",\\n    \\"classifier\\": {\\n        \\"solver\\": \\"lbfgs\\", \\"max_iter\\": 1500, \\"tol\\": 1e-4,\\n        \\"loss\\": \\"joint multinomial negative log likelihood\\",\\n        \\"penalty\\": \\"default L2\\", \\"fit_intercept\\": True,\\n    },\\n}\\n\\n\\ndef build_linear_pipeline(name):\\n    \\"\\"\\"Fit character vocabulary, style scaler and joint classifier on fit rows.\\"\\"\\"\\n    if name not in LINEAR_CANDIDATES:\\n        raise ValueError(f\\"Unknown linear recipe: {name}\\")\\n    return Pipeline([\\n        (\\"features\\", FeatureUnion([\\n            (\\"char\\", TfidfVectorizer(\\n                analyzer=\\"char\\", ngram_range=(1, 5), min_df=2,\\n                max_features=300000, lowercase=False, strip_accents=None,\\n                sublinear_tf=True, dtype=np.float64,\\n            )),\\n            (\\"style\\", Pipeline([\\n                (\\"extract\\", StyleFeatures()),\\n                (\\"scale\\", StandardScaler()),\\n                (\\"normalize\\", Normalizer()),\\n            ])),\\n        ], transformer_weights={\\"char\\": 1., \\"style\\": .5})),\\n        (\\"normalize\\", Normalizer()),\\n        (\\"classifier\\", LogisticRegression(\\n            C=LINEAR_CANDIDATES[name][\\"C\\"],\\n            solver=\\"lbfgs\\", max_iter=1500, tol=1e-4,\\n        )),\\n    ])\\n", "sparse_v2.py": "\\"\\"\\"Predeclared sparse-feature variants for the second experiment.\\"\\"\\"\\n\\nfrom dataclasses import asdict, dataclass\\n\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.svm import LinearSVC\\n\\nimport numpy as np\\n\\n\\n@dataclass(frozen=True)\\nclass SparseConfig:\\n    name: str\\n    ngram_min: int\\n    ngram_max: int\\n    c: float\\n    min_df: int = 2\\n    max_features: int = 300_000\\n    seed: int = 42\\n\\n\\nSPARSE_CANDIDATES = (\\n    SparseConfig(\\"char15_c4\\", 1, 5, 4.0),\\n    SparseConfig(\\"char26_c4\\", 2, 6, 4.0),\\n    SparseConfig(\\"char35_c4\\", 3, 5, 4.0),\\n    SparseConfig(\\"char15_c16\\", 1, 5, 16.0),\\n)\\n\\n\\ndef build_sparse_pipeline(config: SparseConfig) -> Pipeline:\\n    return Pipeline([\\n        (\\"features\\", TfidfVectorizer(\\n            analyzer=\\"char\\", ngram_range=(config.ngram_min, config.ngram_max),\\n            lowercase=False, strip_accents=None, min_df=config.min_df,\\n            max_features=config.max_features, sublinear_tf=True, dtype=np.float32,\\n        )),\\n        (\\"classifier\\", LinearSVC(\\n            C=config.c, dual=True, random_state=config.seed,\\n            max_iter=10_000, tol=1e-4,\\n        )),\\n    ])\\n", "stacking_v4.py": "\\"\\"\\"Prompt-group cross-fitted stacking with an explicit audited OOF boundary.\\n\\nThe evaluation driver uses double exclusions for its inner predictions.\\nThis estimator generates ordinary group OOF only inside its own fit dataset.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nimport hashlib\\nimport json\\nimport logging\\nfrom pathlib import Path\\nimport warnings\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, ClassifierMixin\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.model_selection import GroupKFold\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.utils.validation import check_is_fitted\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom name_that_llm.baseline import LABELS, file_sha256\\nfrom name_that_llm.ensemble_v2 import normalize_evidence\\nfrom name_that_llm.ensemble_v3 import make_v3_component\\n\\nLOGGER = logging.getLogger(\\"nested_v4\\")\\nCOMPONENTS = (\\n    \\"sparse:char15_c4\\",\\n    \\"style:style_hist\\",\\n    \\"style:char_style_svm\\",\\n    \\"linear:joint_char_style_c16\\",\\n)\\nMETA_PARAMETERS = {\\"C\\": .05, \\"solver\\": \\"lbfgs\\", \\"max_iter\\": 2000, \\"tol\\": 1e-4}\\nSOURCE_FILES = (\\n    \\"baseline.py\\", \\"sparse_v2.py\\", \\"stylometry.py\\", \\"linear_v3.py\\",\\n    \\"ensemble_v2.py\\", \\"ensemble_v3.py\\", \\"stacking_v4.py\\",\\n)\\n\\n\\ndef json_hash(value):\\n    return hashlib.sha256(\\n        json.dumps(\\n            value, sort_keys=True, ensure_ascii=False, separators=(\\",\\", \\":\\"),\\n            default=lambda item: item.item() if isinstance(item, np.generic) else item.tolist(),\\n        ).encode()\\n    ).hexdigest()\\n\\n\\ndef array_hash(array):\\n    values = np.ascontiguousarray(array)\\n    return hashlib.sha256(\\n        str(values.dtype).encode() + json.dumps(list(values.shape)).encode() + values.tobytes()\\n    ).hexdigest()\\n\\n\\ndef source_hashes():\\n    root = Path(__file__).resolve().parent\\n    return {name: file_sha256(root / name) for name in SOURCE_FILES}\\n\\n\\ndef recipe_definition():\\n    # Include every effective sklearn parameter, including estimator defaults.\\n    return {\\n        \\"components\\": list(COMPONENTS),\\n        \\"estimator_parameters\\": {\\n            name: {\\n                key: repr(value) for key, value in make_v3_component(name).get_params(deep=True).items()\\n            }\\n            for name in COMPONENTS\\n        },\\n        \\"normalization\\": \\"normalize_evidence per row, log clipped probabilities\\",\\n        \\"meta\\": META_PARAMETERS,\\n        \\"meta_preprocessing\\": \\"StandardScaler fitted on meta training rows only\\",\\n    }\\n\\n\\ndef training_fingerprint(X, y, groups):\\n    return json_hash([list(X), list(y), list(groups)])\\n\\n\\ndef validate_training(X, y, groups):\\n    if isinstance(X, str) or groups is None:\\n        raise ValueError(\\"Pass a sequence of texts and explicit training prompt groups.\\")\\n    texts = np.asarray(X, dtype=object)\\n    targets = np.asarray(y, dtype=object)\\n    prompt_groups = np.asarray(groups, dtype=object)\\n    if any(a.ndim != 1 for a in (texts, targets, prompt_groups)):\\n        raise ValueError(\\"Texts, labels and prompt groups must be one-dimensional.\\")\\n    if not len(texts) or not len(texts) == len(targets) == len(prompt_groups):\\n        raise ValueError(\\"Texts, labels and prompt groups must have equal nonzero length.\\")\\n    if not all(isinstance(t, str) for t in texts):\\n        raise TypeError(\\"All passages must be strings.\\")\\n    if any(g is None or (isinstance(g, float) and np.isnan(g)) for g in prompt_groups):\\n        raise ValueError(\\"Prompt groups cannot be missing.\\")\\n    if set(targets) != set(LABELS):\\n        raise ValueError(\\"All 18 competition classes are required.\\")\\n    return texts, targets, prompt_groups\\n\\n\\ndef validate_fold_assignment(folds, y, groups, *, n_splits=5):\\n    assignment = np.asarray(folds)\\n    targets, prompt_groups = np.asarray(y), np.asarray(groups)\\n    if assignment.ndim != 1 or len(assignment) != len(targets) or len(groups) != len(targets):\\n        raise ValueError(\\"Fold, label and group lengths differ.\\")\\n    if set(assignment) != set(range(n_splits)):\\n        raise ValueError(f\\"Expected folds numbered 0 through {n_splits - 1}.\\")\\n    for group in set(prompt_groups):\\n        if len(set(assignment[prompt_groups == group])) != 1:\\n            raise ValueError(\\"A prompt group crosses folds.\\")\\n    for fold in range(n_splits):\\n        for mask in (assignment == fold, assignment != fold):\\n            if set(targets[mask]) != set(LABELS):\\n                raise ValueError(\\"Every training and held-out fold must contain all classes.\\")\\n    return assignment.astype(np.int64)\\n\\n\\ndef make_group_assignment(X, y, groups, *, n_splits=5, seed=42):\\n    assignment = np.full(len(X), -1, dtype=np.int64)\\n    cv = GroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)\\n    for fold, (_, valid) in enumerate(cv.split(X, y, groups)):\\n        assignment[valid] = fold\\n    return validate_fold_assignment(assignment, y, groups, n_splits=n_splits)\\n\\n\\ndef fit_component(name, X, y):\\n    LOGGER.info(\\"Fit component=%s rows=%d\\", name, len(X))\\n    model = make_v3_component(name)\\n    with warnings.catch_warnings(record=True) as caught:\\n        warnings.simplefilter(\\"always\\")\\n        model.fit(X, y)\\n    for warning in caught:\\n        LOGGER.warning(\\"%s: %s\\", warning.category.__name__, warning.message)\\n    if any(issubclass(w.category, ConvergenceWarning) for w in caught):\\n        raise RuntimeError(f\\"Component failed to converge: {name}\\")\\n    if tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"Base classifier class order mismatch.\\")\\n    return model\\n\\n\\ndef component_evidence(model, X):\\n    if tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"Base classifier class order mismatch.\\")\\n    probabilities = hasattr(model, \\"predict_proba\\")\\n    values = model.predict_proba(X) if probabilities else model.decision_function(X)\\n    return normalize_evidence(values, probabilities=probabilities)\\n\\n\\ndef evidence_features(evidence):\\n    \\"\\"\\"Turn (rows, four components, eighteen classes) into 72 features.\\"\\"\\"\\n    values = np.asarray(evidence, dtype=np.float64)\\n    if values.ndim != 3 or values.shape[1:] != (len(COMPONENTS), len(LABELS)):\\n        raise ValueError(\\"Expected row-by-component-by-class evidence.\\")\\n    if not np.isfinite(values).all():\\n        raise ValueError(\\"Evidence contains non-finite values.\\")\\n    return values.reshape(len(values), -1)\\n\\n\\ndef fit_meta(evidence, y):\\n    features = evidence_features(evidence)\\n    model = Pipeline([\\n        (\\"scale\\", StandardScaler()),\\n        (\\"classifier\\", LogisticRegression(**META_PARAMETERS)),\\n    ])\\n    with warnings.catch_warnings(record=True) as caught:\\n        warnings.simplefilter(\\"always\\")\\n        model.fit(features, y)\\n    for warning in caught:\\n        LOGGER.warning(\\"%s: %s\\", warning.category.__name__, warning.message)\\n    if any(issubclass(w.category, ConvergenceWarning) for w in caught):\\n        raise RuntimeError(\\"Meta classifier failed to converge.\\")\\n    if tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"Meta classifier class order mismatch.\\")\\n    return model\\n\\n\\n@dataclass(frozen=True)\\nclass AuditedOOF:\\n    \\"\\"\\"An OOF bundle bound to exact training rows, folds, source and recipe.\\n\\n    Constructed by the evaluator after auditing producer provenance, or by\\n    generate_fit_oof. A raw array is deliberately not accepted by fit().\\n    \\"\\"\\"\\n\\n    evidence: np.ndarray\\n    folds: np.ndarray\\n    audit: dict\\n\\n    def validate(self, X, y, groups):\\n        evidence_features(self.evidence)\\n        validate_fold_assignment(self.folds, y, groups)\\n        expected = {\\n            \\"training_fingerprint\\": training_fingerprint(X, y, groups),\\n            \\"evidence_sha256\\": array_hash(self.evidence),\\n            \\"fold_assignment_sha256\\": array_hash(self.folds),\\n            \\"source_hashes\\": source_hashes(),\\n            \\"recipe_sha256\\": json_hash(recipe_definition()),\\n            \\"components\\": list(COMPONENTS),\\n            \\"labels\\": list(LABELS),\\n            \\"group_oof_verified\\": True,\\n        }\\n        if self.evidence.shape[0] != len(X):\\n            raise ValueError(\\"OOF evidence row count differs.\\")\\n        for key, value in expected.items():\\n            if self.audit.get(key) != value:\\n                raise ValueError(f\\"OOF audit mismatch: {key}\\")\\n        if not self.audit.get(\\"provenance\\"):\\n            raise ValueError(\\"OOF provenance is required.\\")\\n        return self\\n\\n\\ndef audited_oof_bundle(evidence, folds, X, y, groups, *, provenance):\\n    \\"\\"\\"Seal an OOF matrix after the caller verifies its fold producer records.\\"\\"\\"\\n    values = np.array(evidence, dtype=np.float64, copy=True)\\n    assignment = validate_fold_assignment(folds, y, groups).copy()\\n    values.flags.writeable = False\\n    assignment.flags.writeable = False\\n    audit = {\\n        \\"training_fingerprint\\": training_fingerprint(X, y, groups),\\n        \\"evidence_sha256\\": array_hash(values),\\n        \\"fold_assignment_sha256\\": array_hash(assignment),\\n        \\"source_hashes\\": source_hashes(),\\n        \\"recipe_sha256\\": json_hash(recipe_definition()),\\n        \\"components\\": list(COMPONENTS),\\n        \\"labels\\": list(LABELS),\\n        \\"group_oof_verified\\": True,\\n        \\"provenance\\": provenance,\\n    }\\n    return AuditedOOF(values, assignment, audit).validate(X, y, groups)\\n\\n\\ndef generate_fit_oof(X, y, groups, *, threads=2):\\n    texts, targets, prompt_groups = validate_training(X, y, groups)\\n    folds = make_group_assignment(texts, targets, prompt_groups)\\n    evidence = np.full((len(texts), len(COMPONENTS), len(LABELS)), np.nan)\\n    provenance = []\\n    with threadpool_limits(limits=threads):\\n        for fold in range(5):\\n            fitting, valid = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)\\n            for component, name in enumerate(COMPONENTS):\\n                model = fit_component(name, texts[fitting], targets[fitting])\\n                evidence[valid, component] = component_evidence(model, texts[valid])\\n            provenance.append({\\n                \\"held_out_fold\\": fold, \\"training_indexes_sha256\\": array_hash(fitting),\\n                \\"held_out_indexes_sha256\\": array_hash(valid),\\n            })\\n    return audited_oof_bundle(\\n        evidence, folds, texts, targets, prompt_groups,\\n        provenance={\\"kind\\": \\"runtime_group_oof\\", \\"seed\\": 42, \\"folds\\": provenance},\\n    )\\n\\n\\nclass NestedStacker(ClassifierMixin, BaseEstimator):\\n    \\"\\"\\"Four frozen evidence bases and a group cross-fitted logistic meta model.\\"\\"\\"\\n\\n    requires_groups = True\\n\\n    def __init__(self, threads=2):\\n        self.threads = threads\\n\\n    def fit(self, X, y, groups=None, *, fit_oof=None):\\n        texts, targets, prompt_groups = validate_training(X, y, groups)\\n        if not isinstance(self.threads, int) or not 1 <= self.threads <= 2:\\n            raise ValueError(\\"Use one or two training threads.\\")\\n        if fit_oof is None:\\n            fit_oof = generate_fit_oof(texts, targets, prompt_groups, threads=self.threads)\\n        elif not isinstance(fit_oof, AuditedOOF):\\n            raise TypeError(\\"fit_oof must be an AuditedOOF instance, never raw OOF scores.\\")\\n        fit_oof.validate(texts, targets, prompt_groups)\\n        with threadpool_limits(limits=self.threads):\\n            self.meta_ = fit_meta(fit_oof.evidence, targets)\\n            self.models_ = [fit_component(name, texts, targets) for name in COMPONENTS]\\n        self.classes_ = np.asarray(LABELS)\\n        self.fit_oof_audit_ = fit_oof.audit.copy()\\n        self.n_features_in_ = 1\\n        return self\\n\\n    def base_evidence(self, X):\\n        check_is_fitted(self, [\\"models_\\", \\"meta_\\", \\"classes_\\"])\\n        if isinstance(X, str):\\n            raise ValueError(\\"Expected a sequence of passages.\\")\\n        texts = list(X)\\n        with threadpool_limits(limits=self.threads):\\n            return np.stack([component_evidence(model, texts) for model in self.models_], axis=1)\\n\\n    def predict_proba(self, X):\\n        check_is_fitted(self, [\\"models_\\", \\"meta_\\", \\"classes_\\"])\\n        return self.meta_.predict_proba(evidence_features(self.base_evidence(X)))\\n\\n    def decision_function(self, X):\\n        check_is_fitted(self, [\\"models_\\", \\"meta_\\", \\"classes_\\"])\\n        return self.meta_.decision_function(evidence_features(self.base_evidence(X)))\\n\\n    def predict(self, X):\\n        check_is_fitted(self, [\\"models_\\", \\"meta_\\", \\"classes_\\"])\\n        return self.classes_[self.predict_proba(X).argmax(axis=1)]\\n", "stylometry.py": "\\"\\"\\"Fixed, text-only style features for the separately recorded v2 experiment.\\n\\nFeature families follow the character/word/function-word measurements in\\nhttps://arxiv.org/html/2104.08530v2, extended with Unicode punctuation and\\nposition-specific discourse features. This is not a reproduction of that paper.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nimport re\\nimport string\\nimport unicodedata\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, TransformerMixin\\nfrom sklearn.ensemble import HistGradientBoostingClassifier\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import Normalizer, StandardScaler\\nfrom sklearn.svm import LinearSVC\\n\\n\\n# Fixed before fitting; no corpus vocabulary, labels, IDs, or prompt IDs.\\nFUNCTION_WORDS = tuple(dict.fromkeys(\\"\\"\\"\\na an the and or but if because although though while whereas unless until when\\nwhenever where wherever as than that whether since once before after during\\ndespite through throughout within without with into onto upon from of for to\\nat by in on over under between among against about above below beyond across\\naround along toward towards per via not no nor never neither either both each\\nevery any all some many much few little more most less least several enough\\nother another such same own only even also just still already yet again rather\\nquite very too so then thus hence therefore however instead otherwise indeed\\nperhaps maybe almost always often sometimes usually generally typically\\nespecially particularly simply merely really actually essentially ultimately\\nis are was were be been being am do does did done doing have has had having\\ncan could may might must shall should will would need ought\\ni me my mine myself we us our ours ourselves you your yours yourself yourselves\\nhe him his himself she her hers herself it its itself they them their theirs\\nthemselves one ones someone anyone everyone nobody nothing something everything\\nthis these those there here who whom whose which what why how\\n\\"\\"\\".split()))\\n\\nCONNECTIVES = (\\n    \\"for example\\", \\"for instance\\", \\"in other words\\", \\"in practice\\",\\n    \\"in fact\\", \\"as a result\\", \\"on the other hand\\", \\"at the same time\\",\\n    \\"rather than\\", \\"instead of\\", \\"not only\\", \\"but also\\", \\"even if\\",\\n    \\"even though\\", \\"as long as\\", \\"so that\\", \\"such as\\", \\"in contrast\\",\\n    \\"by contrast\\", \\"in short\\", \\"ultimately\\", \\"however\\", \\"therefore\\",\\n    \\"because\\", \\"although\\", \\"while\\", \\"unless\\", \\"whether\\", \\"if\\",\\n)\\nPUNCTUATION = tuple(dict.fromkeys(string.punctuation + \\"\\u2014\\u2013\\u2018\\u2019\\u201c\\u201d\\u2026\\u2212\\u2022\\\\n\\\\t\\"))\\nWORD_RE = re.compile(r\\"[^\\\\W\\\\d_]+(?:[\'\\u2019][^\\\\W\\\\d_]+)*|\\\\d+(?:\\\\.\\\\d+)?\\", re.UNICODE)\\nSENTENCE_RE = re.compile(r\\"(?<=[.!?])(?:[\\\\\\"\'\\u201d\\u2019)]*)\\\\s+\\")\\n\\n\\nclass StyleFeatures(TransformerMixin, BaseEstimator):\\n    \\"\\"\\"Deterministic measurements; fit deliberately learns nothing.\\"\\"\\"\\n\\n    def fit(self, X, y=None):\\n        return self\\n\\n    def __sklearn_is_fitted__(self):\\n        return True\\n\\n    def get_feature_names_out(self, input_features=None):\\n        return np.asarray(list(self._measure(\\"\\").keys()), dtype=object)\\n\\n    def transform(self, X):\\n        if isinstance(X, str):\\n            raise ValueError(\\"Expected a sequence of passages, not one string.\\")\\n        rows = [list(self._measure(text).values()) for text in X]\\n        if not rows:\\n            return np.empty((0, len(self.get_feature_names_out())), dtype=np.float64)\\n        matrix = np.asarray(rows, dtype=np.float64)\\n        if not np.isfinite(matrix).all():\\n            raise ValueError(\\"Non-finite style feature.\\")\\n        return matrix\\n\\n    @staticmethod\\n    def _measure(text):\\n        if not isinstance(text, str):\\n            raise TypeError(\\"StyleFeatures accepts text strings only.\\")\\n        words = WORD_RE.findall(text.lower())\\n        counts = Counter(words)\\n        n = max(1, len(text))\\n        nw = max(1, len(words))\\n        lengths = np.asarray([len(w) for w in words] or [0], dtype=float)\\n        sentences = [s for s in SENTENCE_RE.split(text) if s.strip()]\\n        sentence_lengths = np.asarray(\\n            [len(WORD_RE.findall(s)) for s in sentences] or [0], dtype=float,\\n        )\\n        ns = max(1, len(sentences))\\n        f = {\\n            \\"log_chars\\": np.log1p(len(text)),\\n            \\"log_words\\": np.log1p(len(words)),\\n            \\"log_sentences\\": np.log1p(len(sentences)),\\n            \\"words_per_sentence\\": len(words) / ns,\\n            \\"chars_per_word\\": len(text) / nw,\\n            \\"type_token_ratio\\": len(counts) / nw,\\n            \\"hapax_ratio\\": sum(v == 1 for v in counts.values()) / nw,\\n            \\"repeated_token_ratio\\": sum(v for v in counts.values() if v > 1) / nw,\\n            \\"digits_rate\\": sum(c.isdigit() for c in text) / n,\\n            \\"uppercase_rate\\": sum(c.isupper() for c in text) / n,\\n            \\"letters_rate\\": sum(c.isalpha() for c in text) / n,\\n            \\"whitespace_rate\\": sum(c.isspace() for c in text) / n,\\n            \\"unicode_rate\\": sum(ord(c) > 127 for c in text) / n,\\n            \\"unicode_punctuation_rate\\": sum(\\n                ord(c) > 127 and unicodedata.category(c).startswith(\\"P\\") for c in text\\n            ) / n,\\n            \\"contraction_rate\\": sum(\\"\'\\" in w or \\"\\u2019\\" in w for w in words) / nw,\\n            \\"short_word_rate\\": float(np.mean(lengths <= 3)),\\n            \\"long_word_rate\\": float(np.mean(lengths >= 8)),\\n            \\"question_sentence_rate\\": sum(\\"?\\" in s for s in sentences) / ns,\\n            \\"exclamation_sentence_rate\\": sum(\\"!\\" in s for s in sentences) / ns,\\n            \\"sentence_length_alternation\\": float(np.abs(np.diff(sentence_lengths)).mean())\\n            if len(sentence_lengths) > 1 else 0.,\\n        }\\n        for prefix, values in ((\\"word_length\\", lengths), (\\"sentence_words\\", sentence_lengths)):\\n            for name, value in (\\n                (\\"mean\\", np.mean(values)), (\\"std\\", np.std(values)),\\n                (\\"min\\", np.min(values)), (\\"max\\", np.max(values)),\\n                (\\"q25\\", np.quantile(values, .25)), (\\"q50\\", np.quantile(values, .5)),\\n                (\\"q75\\", np.quantile(values, .75)),\\n            ):\\n                f[f\\"{prefix}_{name}\\"] = float(value)\\n        for length in range(1, 16):\\n            f[f\\"word_length_bin_{length}\\"] = float(np.mean(lengths == length))\\n        for char in PUNCTUATION:\\n            key = f\\"punct_u{ord(char):04x}\\"\\n            f[f\\"{key}_rate\\"] = text.count(char) / nw\\n            f[f\\"{key}_present\\"] = float(char in text)\\n        for char in string.ascii_lowercase:\\n            f[f\\"letter_{char}_rate\\"] = text.lower().count(char) / n\\n        for word in FUNCTION_WORDS:\\n            f[f\\"fw_{word}\\"] = counts[word] / nw\\n        lower = \\" \\" + \\" \\".join(words) + \\" \\"\\n        first = \\" \\" + \\" \\".join(words[:15]) + \\" \\"\\n        last = \\" \\" + \\" \\".join(words[-15:]) + \\" \\"\\n        for connective in CONNECTIVES:\\n            needle = \\" \\" + connective + \\" \\"\\n            f[f\\"connective_{connective}\\"] = lower.count(needle) / nw\\n            f[f\\"opening_{connective}\\"] = float(needle in first)\\n            f[f\\"ending_{connective}\\"] = float(needle in last)\\n        for position, sentence in (\\n            (\\"opening\\", sentences[0] if sentences else \\"\\"),\\n            (\\"ending\\", sentences[-1] if sentences else \\"\\"),\\n        ):\\n            sw = WORD_RE.findall(sentence.lower())\\n            f[f\\"{position}_length_share\\"] = len(sw) / nw\\n            f[f\\"{position}_question\\"] = float(\\"?\\" in sentence)\\n            f[f\\"{position}_colon\\"] = float(\\":\\" in sentence)\\n            f[f\\"{position}_dash\\"] = float(\\"\\u2014\\" in sentence)\\n            f[f\\"{position}_first_person\\"] = sum(w in {\\"i\\", \\"we\\", \\"our\\", \\"my\\"} for w in sw) / max(1, len(sw))\\n            f[f\\"{position}_second_person\\"] = sum(w in {\\"you\\", \\"your\\"} for w in sw) / max(1, len(sw))\\n        return f\\n\\n\\nCANDIDATES = {\\n    \\"style_logistic\\": {\\n        \\"features\\": \\"fixed StyleFeatures -> StandardScaler\\",\\n        \\"classifier\\": \\"LogisticRegression\\", \\"C\\": .05,\\n        \\"solver\\": \\"lbfgs\\", \\"max_iter\\": 1500,\\n    },\\n    \\"style_hist\\": {\\n        \\"features\\": \\"fixed StyleFeatures\\", \\"classifier\\": \\"HistGradientBoostingClassifier\\",\\n        \\"max_iter\\": 160, \\"max_leaf_nodes\\": 7, \\"learning_rate\\": .07,\\n        \\"l2_regularization\\": 3., \\"min_samples_leaf\\": 25,\\n        \\"early_stopping\\": False, \\"random_state\\": 42,\\n    },\\n    \\"char_style_svm\\": {\\n        \\"features\\": \\"char TFIDF (3,5), min_df2, max_features250000, case-sensitive\\"\\n        \\" + .5 * L2-normalized standardized style; concatenate and L2-normalize\\",\\n        \\"classifier\\": \\"LinearSVC\\", \\"C\\": 1., \\"max_iter\\": 10000,\\n        \\"dual\\": True, \\"random_state\\": 42,\\n    },\\n}\\n\\n\\ndef build_style_pipeline(name):\\n    style = StyleFeatures()\\n    if name == \\"style_logistic\\":\\n        return Pipeline([\\n            (\\"style\\", style), (\\"scale\\", StandardScaler()),\\n            (\\"classifier\\", LogisticRegression(C=.05, solver=\\"lbfgs\\", max_iter=1500)),\\n        ])\\n    if name == \\"style_hist\\":\\n        return Pipeline([\\n            (\\"style\\", style),\\n            (\\"classifier\\", HistGradientBoostingClassifier(\\n                max_iter=160, max_leaf_nodes=7, learning_rate=.07,\\n                l2_regularization=3., min_samples_leaf=25,\\n                early_stopping=False, random_state=42,\\n            )),\\n        ])\\n    if name == \\"char_style_svm\\":\\n        return Pipeline([\\n            (\\"features\\", FeatureUnion([\\n                (\\"char\\", TfidfVectorizer(\\n                    analyzer=\\"char\\", ngram_range=(3, 5), min_df=2,\\n                    max_features=250_000, lowercase=False, strip_accents=None,\\n                    sublinear_tf=True, dtype=np.float32,\\n                )),\\n                (\\"style\\", Pipeline([\\n                    (\\"extract\\", style), (\\"scale\\", StandardScaler()),\\n                    (\\"normalize\\", Normalizer()),\\n                ])),\\n            ], transformer_weights={\\"char\\": 1., \\"style\\": .5})),\\n            (\\"normalize\\", Normalizer()),\\n            (\\"classifier\\", LinearSVC(\\n                C=1., max_iter=10000, dual=True, random_state=42, tol=1e-4,\\n            )),\\n        ])\\n    raise ValueError(name)\\n\\n", "topic_v3.py": "\\"\\"\\"Training-only prompt nuisance removal and fixed content masking for v3.\\n\\nThe masking experiment is motivated by arXiv:2104.08530, not a reproduction.\\nPrompt-centered ridge is a separately derived hypothesis: train within prompts,\\nthen predict individual passages relative to the training global feature mean.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\n\\nimport numpy as np\\nfrom scipy.linalg import solve\\nfrom sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import Normalizer, StandardScaler\\nfrom sklearn.svm import LinearSVC\\nfrom sklearn.utils.validation import check_is_fitted\\n\\nfrom name_that_llm.stylometry import FUNCTION_WORDS, StyleFeatures\\n\\n\\nTOPIC_CANDIDATES = {\\n    \\"centered_ridge_half\\": {\\n        \\"centering\\": 0.5, \\"ridge_alpha\\": 1.0,\\n        \\"features\\": \\"raw char3-5 TFIDF + 0.5 normalized standardized style\\",\\n    },\\n    \\"centered_ridge_full\\": {\\n        \\"centering\\": 1.0, \\"ridge_alpha\\": 1.0,\\n        \\"features\\": \\"raw char3-5 TFIDF + 0.5 normalized standardized style\\",\\n    },\\n    \\"masked_char_style_svm\\": {\\n        \\"mask\\": \\"retain fixed function words; other letters \'*\'; digits \'#\'\\",\\n        \\"C\\": 1.0,\\n        \\"features\\": \\"masked char3-5 TFIDF + 0.5 normalized standardized raw style\\",\\n    },\\n}\\n\\nMASK_TOKEN = re.compile(r\\"[^\\\\W\\\\d_]+(?:[\'\\u2019][^\\\\W\\\\d_]+)*|\\\\d+\\", re.UNICODE)\\nKEEP_WORDS = frozenset(FUNCTION_WORDS)\\n\\n\\nclass MaskContent(TransformerMixin, BaseEstimator):\\n    \\"\\"\\"Preserve punctuation, function words and word lengths, deterministically.\\"\\"\\"\\n\\n    def fit(self, X, y=None):\\n        return self\\n\\n    def __sklearn_is_fitted__(self):\\n        return True\\n\\n    @staticmethod\\n    def mask(text):\\n        if not isinstance(text, str):\\n            raise TypeError(\\"Expected a passage string.\\")\\n\\n        def replacement(match):\\n            word = match.group()\\n            if word.lower() in KEEP_WORDS:\\n                return word\\n            return \\"\\".join(\\"#\\" if c.isdigit() else \\"*\\" if c.isalpha() else c for c in word)\\n\\n        return MASK_TOKEN.sub(replacement, text)\\n\\n    def transform(self, X):\\n        if isinstance(X, str):\\n            raise ValueError(\\"Expected a sequence of passages.\\")\\n        return [self.mask(text) for text in X]\\n\\n\\ndef build_topic_features(masked=False):\\n    char = TfidfVectorizer(\\n        analyzer=\\"char\\", ngram_range=(3, 5), min_df=2, max_features=250_000,\\n        lowercase=False, strip_accents=None, sublinear_tf=True, dtype=np.float64,\\n    )\\n    if masked:\\n        char = Pipeline([(\\"mask\\", MaskContent()), (\\"tfidf\\", char)])\\n    return Pipeline([\\n        (\\"features\\", FeatureUnion([\\n            (\\"char\\", char),\\n            (\\"style\\", Pipeline([\\n                (\\"extract\\", StyleFeatures()), (\\"scale\\", StandardScaler()),\\n                (\\"normalize\\", Normalizer()),\\n            ])),\\n        ], transformer_weights={\\"char\\": 1.0, \\"style\\": 0.5})),\\n        (\\"normalize\\", Normalizer()),\\n    ])\\n\\n\\ndef center_by_training_groups(values, group_indices, strength):\\n    \\"\\"\\"Apply A = I - strength*G - (1-strength)*J to rows.\\n\\n    G averages rows inside each training prompt; J is the training global mean.\\n    These are symmetric projectors and A never uses labels or held-out rows.\\n    \\"\\"\\"\\n    matrix = np.asarray(values, dtype=np.float64)\\n    result = matrix - (1.0 - strength) * matrix.mean(axis=0, keepdims=True)\\n    for indices in group_indices:\\n        result[indices] -= strength * matrix[indices].mean(axis=0, keepdims=True)\\n    return result\\n\\n\\nclass PromptCenteredRidge(ClassifierMixin, BaseEstimator):\\n    \\"\\"\\"Fit ridge on prompt-centered training features; infer with a global mean.\\n\\n    Fit requires training prompt groups. Prediction deliberately cannot accept\\n    groups: a passage is scored without reference to any other test passage.\\n    The Gram implementation avoids densifying the sparse feature matrix.\\n    \\"\\"\\"\\n\\n    requires_groups = True\\n\\n    def __init__(self, strength=1.0, alpha=1.0):\\n        self.strength = strength\\n        self.alpha = alpha\\n\\n    def fit(self, X, y, groups=None):\\n        if not 0.0 <= self.strength <= 1.0 or self.alpha <= 0:\\n            raise ValueError(\\"Invalid centering strength or ridge regularization.\\")\\n        texts, labels = list(X), np.asarray(y)\\n        if groups is None:\\n            raise ValueError(\\"Training prompt groups are required.\\")\\n        groups = np.asarray(groups)\\n        if labels.ndim != 1 or groups.ndim != 1 or not len(texts) == len(labels) == len(groups):\\n            raise ValueError(\\"Texts, labels and groups must be aligned one-dimensional arrays.\\")\\n        self.classes_, label_indices = np.unique(labels, return_inverse=True)\\n        if len(self.classes_) < 2:\\n            raise ValueError(\\"Need at least two classes.\\")\\n        _, group_codes = np.unique(groups, return_inverse=True)\\n        group_indices = [np.flatnonzero(group_codes == code) for code in np.unique(group_codes)]\\n        if len(group_indices) < 2:\\n            raise ValueError(\\"Need at least two training prompt groups.\\")\\n        self.features_ = build_topic_features()\\n        features = self.features_.fit_transform(texts, labels)\\n        targets = np.eye(len(self.classes_))[label_indices]\\n        prior = targets.mean(axis=0)\\n        targets -= prior\\n        gram = (features @ features.T).toarray()\\n        centered = center_by_training_groups(gram, group_indices, self.strength)\\n        centered = center_by_training_groups(centered.T, group_indices, self.strength).T\\n        # Roundoff can leave tiny asymmetry; restoring symmetry precedes the SPD solve.\\n        centered = (centered + centered.T) * 0.5\\n        centered.flat[::len(texts) + 1] += self.alpha\\n        dual = solve(centered, targets, assume_a=\\"pos\\", check_finite=True)\\n        adjusted_dual = center_by_training_groups(dual, group_indices, self.strength)\\n        self.coef_ = np.asarray(features.T @ adjusted_dual).T\\n        feature_mean = np.asarray(features.mean(axis=0)).ravel()\\n        self.intercept_ = prior - self.coef_ @ feature_mean\\n        self.n_training_groups_ = len(group_indices)\\n        if not np.isfinite(self.coef_).all() or not np.isfinite(self.intercept_).all():\\n            raise ValueError(\\"Non-finite fitted parameters.\\")\\n        return self\\n\\n    def decision_function(self, X):\\n        check_is_fitted(self, [\\"features_\\", \\"classes_\\", \\"coef_\\", \\"intercept_\\"])\\n        return np.asarray(self.features_.transform(X) @ self.coef_.T) + self.intercept_\\n\\n    def predict(self, X):\\n        return self.classes_[self.decision_function(X).argmax(axis=1)]\\n\\n\\ndef build_topic_pipeline(name):\\n    if name == \\"centered_ridge_half\\":\\n        return PromptCenteredRidge(strength=0.5, alpha=1.0)\\n    if name == \\"centered_ridge_full\\":\\n        return PromptCenteredRidge(strength=1.0, alpha=1.0)\\n    if name == \\"masked_char_style_svm\\":\\n        return Pipeline([\\n            (\\"features\\", build_topic_features(masked=True)),\\n            (\\"classifier\\", LinearSVC(C=1.0, max_iter=10000, dual=True, random_state=42, tol=1e-4)),\\n        ])\\n    raise ValueError(f\\"Unknown topic candidate: {name}\\")\\n"}')
package=Path.cwd()/'name_that_llm'
package.mkdir(exist_ok=True)
for name, source in sources.items():
    (package/name).write_text(source)
sys.path.insert(0,str(package.parent))
recipe=json.loads('{"embedding_weight": 0.5, "v4_weight": 0.5}')
golden=json.loads('{"texts": ["Hello, world!  Case and spacing stay.\\nA second line.", "A short, deliberately synthetic writing sample with punctuation: yes; no; maybe.", "A longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\nA longer synthetic passage with repeated words and new lines.\\n"], "embeddings": [[-0.0033372528851032257, -0.03664742037653923, 0.020268598571419716, -0.02061406522989273, 0.0005303524667397141, 0.009487897157669067, -0.020066840574145317, -0.06320279091596603, 0.03801847621798515, 0.00264643388800323, -0.015411301515996456, 0.02088073454797268, 0.027871398255228996, 0.015582163818180561, 0.011675352230668068, -0.023232197389006615, 0.026256920769810677, 0.03298993036150932, -0.01663954369723797, -0.007375137880444527, 0.0013586869463324547, 0.023831481114029884, 0.029746802523732185, -0.011294187977910042, 0.010821167379617691, -0.014872398227453232, 0.025242066010832787, 0.0157795250415802, -0.006402650382369757, -0.007728878874331713, -0.03257785737514496, 0.014862391166388988, -0.008727042935788631, -0.037821151316165924, 0.002155625494197011, 0.010896709747612476, -0.01408914104104042, -0.020589232444763184, -0.015150190331041813, 0.023665141314268112, -0.012498387135565281, 0.001402794849127531, 0.042475391179323196, 0.013194648548960686, 0.022981319576501846, 0.0004207843740005046, -0.03628817945718765, -0.000295284204185009, -0.06608136743307114, -0.0029962260741740465, -0.006456723902374506, 0.023863263428211212, -0.008197381161153316, 0.03825484588742256, 0.035707782953977585, 0.0027307928539812565, -0.0031471101101487875, 0.0016180621460080147, -0.013983405195176601, 0.007517282851040363, 0.017367307096719742, 0.007385397329926491, -0.02296038344502449, -0.0037843736354261637, -0.02454826980829239, -0.00499587319791317, -0.0066446647979319096, 0.004010057542473078, -0.025394132360816002, 0.027541767805814743, 0.007398347835987806, 0.040502049028873444, 0.00665071839466691, -0.011095399037003517, -0.03387031331658363, -0.012637827545404434, 0.033720795065164566, -0.12356136739253998, 0.025105994194746017, 0.009047258645296097, -0.046844709664583206, -0.003726110327988863, 0.04994873329997063, 0.009135901927947998, 0.03063235990703106, -0.0020524486899375916, -0.014268496073782444, 0.01541469618678093, 0.014058013446629047, 0.019056133925914764, 0.0465439148247242, -0.0020980241242796183, -0.015926478430628777, 0.04491857811808586, -0.029896505177021027, -0.0046914201229810715, -0.010529532097280025, 0.06466414034366608, 0.015265043824911118, 0.04682662710547447, -0.005414428189396858, 0.01268966868519783, 0.04458776116371155, 0.027178507298231125, 0.02467452362179756, -0.006661121733486652, 0.022179383784532547, 0.02020239643752575, -0.009403996169567108, -0.0017691855318844318, 0.011071898974478245, 0.02055293880403042, 0.0008330137352459133, 0.01243401039391756, 0.020949458703398705, 0.009848076850175858, 0.005213258322328329, -0.02466220036149025, -0.01714359037578106, 0.011434968560934067, 0.011924464255571365, 0.038365185260772705, -0.005521024577319622, 0.01245566550642252, 0.015667658299207687, -0.017532385885715485, 0.004195393528789282, -0.018745198845863342, -0.005440708715468645, -0.030477410182356834, 0.0022652256302535534, -0.016328753903508186, -0.015751993283629417, 0.02574409730732441, 0.012615128420293331, -0.013008435256779194, 0.009072775952517986, -0.04238459840416908, -0.04271036013960838, 0.006955096032470465, 0.04482584074139595, 0.01065998338162899, 0.021799422800540924, 0.024042591452598572, -0.011390862055122852, 0.005717309191823006, -0.005672533065080643, -0.023463692516088486, 0.007560186553746462, 0.02591468207538128, 0.009214845485985279, 0.031014060601592064, -0.008454260416328907, 0.011349556967616081, -0.023374900221824646, -0.04246050864458084, 0.012227834202349186, 0.02030189335346222, -0.020033933222293854, -0.012578390538692474, 0.037521842867136, -0.032649192959070206, 0.005089167505502701, -0.0032904737163335085, 0.01155161764472723, -0.0011054646456614137, 0.005862775258719921, 0.0006195910973474383, -0.005279937293380499, 0.013942287303507328, -0.01395962294191122, 0.00177735835313797, -0.008512980304658413, 0.0028510827105492353, -0.0047394875437021255, 0.027235176414251328, 0.01591724529862404, 0.03576415404677391, 0.016727808862924576, 0.020513715222477913, -0.007009587250649929, 0.010601365938782692, -0.007042372599244118, 0.017290163785219193, -0.0035673014353960752, -0.013737637549638748, 0.02694043144583702, 0.00564575893804431, 0.025693660601973534, -0.0021272171288728714, 0.008332090452313423, 0.0014961111592128873, -0.01540713757276535, 0.00717840064316988, -0.01258019171655178, 0.020433345809578896, 0.015656819567084312, 0.031813934445381165, 0.02307938225567341, 0.041817761957645416, 0.012652423232793808, -0.022700544446706772, 0.03922022134065628, 0.0286631491035223, 0.006721737794578075, -0.016810016706585884, -0.02737104520201683, -0.03414943069219589, 0.05089579150080681, -0.009663932025432587, -0.00438635004684329, 0.01061877142637968, -0.04700577259063721, 0.05761433765292168, -0.02877870202064514, 0.0203259214758873, -0.009235912002623081, -0.04728745296597481, 0.004690500907599926, 0.02047981694340706, 0.003321552649140358, 0.005710179451853037, 0.01674109324812889, 0.018646154552698135, -0.03041950799524784, -0.061739567667245865, -0.01664336584508419, -0.0036047976464033127, 0.0017734284047037363, -0.015925901010632515, -0.0005841617239639163, 0.048961494117975235, -0.008303016424179077, 0.011306309141218662, 0.0021997089497745037, -0.03033447265625, -0.03599853441119194, 0.03749263286590576, 0.024851933121681213, 0.011678067967295647, -0.030438251793384552, -0.007832834497094154, -0.02752656117081642, 0.026135118678212166, -0.018334489315748215, 0.06113950535655022, -0.01073352713137865, 0.035824552178382874, -0.0449296273291111, 0.0016748313792049885, -0.0035084993578493595, 0.013743562623858452, -0.01311334315687418, 0.025569524616003036, -0.0018167501548305154, -0.01583130657672882, 0.021606553345918655, 0.0011318394681438804, -0.012563444674015045, -0.12025496363639832, -0.016331162303686142, -0.008729511871933937, -0.008307373151183128, 0.003852883819490671, -0.014450776390731335, 0.01224935706704855, 0.011993025429546833, -0.033049147576093674, -0.029882920905947685, -0.03278694301843643, -0.02710087038576603, 0.008966206572949886, 0.012037636712193489, -0.006098699755966663, -0.017806587740778923, 0.027822325006127357, 0.016162754967808723, 0.006801053881645203, -0.04034114256501198, -0.011914152652025223, 0.0025985126849263906, 0.014766314066946507, -0.012376301921904087, -0.018330732360482216, -0.028476374223828316, -0.023515058681368828, -0.07743801176548004, -0.010152428410947323, 0.041184451431035995, 0.022200046107172966, 0.009017733857035637, -0.0022070107515901327, 0.013446057215332985, -0.03148166090250015, 0.014302653260529041, -0.028390366584062576, 0.028328262269496918, -0.023021558299660683, 0.06648287922143936, 0.026532303541898727, -0.021209174767136574, 0.011022507213056087, 0.0070998347364366055, -0.019877726212143898, 0.0010242308489978313, -0.0057639749720692635, -0.0044092885218560696, -0.010561663657426834, 0.004782869480550289, -0.0031397026032209396, -0.013461625203490257, -0.04808463528752327, -0.02819783426821232, 0.016151873394846916, -0.011062420904636383, -0.03291885927319527, 0.0025418391451239586, 0.007463700138032436, -0.011283619329333305, -0.02305484190583229, 0.024367930367588997, 0.006042687688022852, -0.03541051596403122, 0.04962322115898132, 0.03742113336920738, 0.03147449344396591, -0.04736924171447754, 0.0016868050443008542, -0.02540087327361107, -0.0035981726832687855, -0.10588400810956955, -0.002658765995875001, 0.02448316663503647, -0.0046117063611745834, 0.020008819177746773, 0.014708415605127811, -0.031146276742219925, 0.013583822175860405, 0.013105069287121296, 0.010783085599541664, -0.030875375494360924, 0.009147445671260357, -0.0318419374525547, -0.0065706269815564156, -0.007941027171909809, -0.018125195056200027, 0.03264202922582626, -0.03510577604174614, -0.010676879435777664, -0.030336014926433563, 0.028140289708971977, -0.027778875082731247, 0.025403160601854324, -0.03109952062368393, -0.012255635112524033, 0.02591472864151001, -0.0004005673108622432, 0.015839530155062675, 0.026843085885047913, 0.026001323014497757, 0.018327323719859123, -0.02165934629738331, -0.0026989872567355633, 0.02532634139060974, 0.029178904369473457, -0.03452987223863602, 0.017339153215289116, 0.022019648924469948, 0.0016166517743840814, -0.008226209320127964, 0.00616003992035985, 0.04088001698255539, -0.022315887734293938, 0.036858730018138885, -0.027113335207104683, 0.009616296738386154, 0.0229559987783432, -0.01097886823117733, -0.011239939369261265, -0.005105677992105484, 0.012563909403979778, -0.017355840653181076, 0.02380450814962387, -0.0031928811222314835, 0.006576227489858866, 0.008042525500059128, 0.01250836718827486, 0.01536414586007595, -0.049286168068647385, 0.026961522176861763, 0.03725053742527962, 0.06558220088481903, -0.008520534262061119, 0.010779356583952904, 0.016895759850740433, 0.008261498063802719, 0.016557643190026283, 0.032659441232681274, 0.007313228212296963, 0.04290919005870819, -0.005549977999180555, -0.014924565330147743, -0.02657824195921421, -0.004816110245883465, -0.008269313722848892, 0.00882440060377121, -0.0172763392329216, -0.012817583978176117, 0.009171281941235065, 0.008829347789287567, 0.0187407024204731, -0.030965710058808327, 0.012697167694568634, -0.002177531598135829, 0.0029185309540480375, 0.01979004591703415, -0.024318337440490723, 0.030793556943535805, 0.029293328523635864, -0.015252471901476383, -0.020517101511359215, -0.020877163857221603, 0.014420759864151478, 0.020212460309267044, 0.010590805672109127, -0.0215359628200531, -0.005716621410101652, -0.0014785045059397817, 0.047897547483444214, 0.013606269843876362, 0.012119176797568798, 0.053097546100616455, 0.0009192042634822428, -0.01960330829024315, 0.00911461841315031, 0.011147507466375828, 0.011237258091568947, -0.05364092066884041, 0.013734353706240654, -0.018079537898302078, -0.002403234364464879, -0.01540342252701521, 0.04736350104212761, 0.0075814672745764256, 0.028037352487444878, 0.006995313335210085, 0.06074124947190285, -0.018292592838406563, 0.009122593328356743, 0.012061145156621933, 0.0502372682094574, -0.011406058445572853, -0.015285821631550789, -0.1653214693069458, 0.026781972497701645, -0.002566693117842078, 0.04078543931245804, -0.019330542534589767, 0.026533225551247597, 0.0007126416312530637, 0.0015881994040682912, -0.017904171720147133, 0.01684276945888996, -0.021641762927174568, 0.026187626644968987, 0.03059028461575508, -0.005990327335894108, 0.03839383274316788, 0.006136905401945114, 0.03838391229510307, 4.8016085202107206e-05, -0.0005676313885487616, -0.0034788413904607296, -0.007461216300725937, -0.011978565715253353, -0.017980467528104782, 0.034934960305690765, 0.020524675026535988, -0.021455911919474602, -0.013109034858644009, 0.011275433003902435, -0.02373424544930458, 0.02662844769656658, 0.009855255484580994, 0.00527789955958724, -0.007170992437750101, -0.016632910817861557, 0.0012217770563438535, -0.031290289014577866, -0.007983900606632233, 0.02775782160460949, 0.015485159121453762, -0.04543454200029373, 0.004107123706489801, -0.038550082594156265, 0.04084477946162224, -0.010373872704803944, -0.008082082495093346, 0.05623722821474075, -0.0007230046321637928, 0.022138476371765137, 0.01974654011428356, -0.012446866370737553, 0.0013708198675885797, 0.03791884332895279, -0.03560331463813782, 0.03692000359296799, -0.018345598131418228, -0.02147839404642582, -0.0007924741366878152, -0.02215338684618473, 0.035707563161849976, 0.060608964413404465, -0.012096178717911243, -0.009996514767408371, -0.012978948652744293, -0.007094013039022684, 0.01485457457602024, 0.00014529710460919887, 0.037075698375701904, -0.011971654370427132, 0.008312850259244442, 0.006647353991866112, -0.037563543766736984, 0.017332352697849274, -0.013463578186929226, -0.026789963245391846, 0.011326958425343037, -0.012995344586670399, 0.02732565440237522, 0.04644005000591278, 0.0188237726688385, 0.02261769212782383, -0.012628014199435711, 0.022810127586126328, 0.031075425446033478, 0.02036636509001255, 0.04260794073343277, 0.019478274509310722, -0.00028059238684363663, -0.00778058310970664, 0.005604315083473921, 0.0067955926060676575, 0.019033988937735558, 0.0069209816865623, 0.028051823377609253, -0.03665691614151001, 0.006466323044151068, 0.01898980140686035, -0.02136022411286831, 0.0001672967482591048, -0.06560474634170532, -0.00044876625179313123, 0.005650175269693136, -0.027552098035812378, -0.02809516340494156, -0.026711823418736458, 0.005846732761710882, 0.014644184149801731, 0.04172474518418312, 0.005401405040174723, -0.06507879495620728, -0.012153000570833683, 0.03953240066766739, 0.005688702687621117, 0.0072928424924612045, 0.0017001995583996177, 0.013152295723557472, 0.003056079614907503, -0.003857659175992012, -0.005175504367798567, -0.01135896984487772, 0.026735110208392143, -0.02906566485762596, 0.001264216029085219, 0.022945759817957878, 0.014309491962194443, 0.03132098168134689, -0.006474901922047138, -0.012222432531416416, 0.00636901892721653, 0.022393004968762398, -0.023474760353565216, -0.0020359845366328955, -0.01928042620420456, -0.0021858676336705685, 0.06096978485584259, 0.05589877441525459, 0.6954898238182068, -0.007939719595015049, -0.0353052131831646, -0.012755273841321468, -0.017772160470485687, -0.028556305915117264, -0.03535334765911102, -0.03774173557758331, -0.005547170527279377, 0.006528689060360193, -0.007274119649082422, -0.020502464845776558, -0.04169883579015732, -0.02106163278222084, 0.017157820984721184, 0.007109328173100948, 0.03118758276104927, -0.01657087355852127, 0.014540696516633034, 0.01873261295258999, 0.0176724661141634, 0.020385878160595894, 0.026531772688031197, 0.08775749802589417, 0.005109505727887154, -0.0214836448431015, 0.03706571087241173, 0.018981559202075005, 0.0010010930709540844, 0.0026645248290151358, 0.011670981533825397, -0.004925837274640799, -0.01195647194981575, 0.06781408935785294, 0.029698586091399193, -0.01089706365019083, 0.00817245151847601, -0.011552584357559681, 0.015025230124592781, 0.004178672097623348, 0.025142183527350426, 0.014747606590390205, 0.004303145222365856, -0.02273603156208992, 0.014844131655991077, 0.01237821951508522, -0.055259786546230316, 0.027210786938667297, -0.009956508874893188, 0.013892484828829765, 0.019122647121548653, -0.011442327871918678, -0.02324853464961052, -0.006690842565149069, -0.013254666700959206, 0.009355001151561737, -0.022342827171087265, 0.03971418738365173, 0.030152926221489906, -0.00821465440094471, 0.03036203607916832, 0.004433655180037022, -0.012322778813540936, -0.005344605538994074, 0.051963530480861664, 0.06813615560531616, -0.003272291738539934, 0.04132243990898132, -0.014514659531414509, 0.0395478792488575, -0.03839587792754173, -0.0070721060037612915, 0.0018009828636422753, -0.01285780593752861, 0.014981914311647415, -0.0029733723495155573, -0.04521525278687477, 0.027000397443771362, -0.034648213535547256, -0.010405883193016052, 0.04263506457209587, 0.02041815221309662, 0.0033295026514679193, -0.0008784888777881861, 0.05046109855175018, 0.01910197176039219, -0.026285367086529732, 0.007747170515358448, -0.042728275060653687, -0.023905634880065918, -0.02776060439646244, -0.055819496512413025, -0.04170513153076172, -0.001966854790225625, 0.019112803041934967, -0.013382325880229473, 0.00852110330015421, -0.00028415475389920175, 0.008395619690418243, 0.02945244312286377, -0.014383779838681221, 0.03463520109653473, -0.02060585469007492, 0.017730755731463432, -0.010002948343753815, 0.023682843893766403, 0.03288668766617775, -0.0230028685182333, 0.019330183044075966, -0.0188978873193264, 0.022678663954138756, 0.04295935481786728, -0.020717261359095573, 0.007731117308139801, 0.00013240677071735263, -0.02842564322054386, -0.01425466313958168, 0.02189101278781891, 0.029676124453544617, 0.007527401205152273, -0.008000820875167847, -0.0002316551690455526, -0.010794602334499359, 0.03833994269371033, 0.03096170164644718, -0.0033422706183046103, -0.0351858027279377, -0.0163923017680645, -0.0007840688340365887, -0.0034591772127896547, -0.01194335613399744, -0.006853234954178333, -0.0016541082877665758, 0.03000977821648121, 0.03389084339141846, 0.008768010884523392, -0.013021709397435188, -0.005526418797671795, 0.017848897725343704, -0.0018453653901815414, 0.03654809668660164, 0.015507036820054054, -0.007917098701000214, 0.0027077984996140003, -0.028103353455662727, -0.02000323310494423, -0.010538086295127869, -0.02312999591231346, -0.0005119094275869429, -0.058034271001815796, 0.011986756697297096, 0.05105509236454964, 0.03148675337433815, -0.05734891816973686, -0.04838507995009422, -0.039955515414476395, -0.00099480664357543, -0.02121833711862564, -0.01973954401910305, 0.029244476929306984, 0.010683718137443066, -0.024229003116488457, 0.006175683345645666, -0.03795195743441582, 0.007482820190489292, 0.04587482288479805, 0.005614218767732382, -0.0019338756101205945, -0.02327352575957775, 0.027943234890699387, 0.006186574697494507, 0.015643926337361336, -0.024333670735359192, -0.0006367704481817782, 0.032866887748241425, -0.012839908711612225, 0.025328747928142548, 0.0469769686460495, -0.02384381741285324, 0.0034487692173570395], [-0.005253931973129511, 0.0022876348812133074, -0.0026871724985539913, -0.027008498087525368, -0.03468671813607216, 0.006137197371572256, -0.014061054214835167, 0.009499108418822289, 0.022442057728767395, 0.006954642944037914, -0.03418723866343498, -0.005339542869478464, 0.020708145573735237, -0.03882419690489769, -0.01280898880213499, 0.02158878557384014, -0.015148545615375042, 0.020527003332972527, 0.019345249980688095, -0.03807951882481575, 0.03730543330311775, -0.027960022911429405, 0.013003572821617126, -0.0022125004325062037, 0.005458510480821133, 0.007756293285638094, 0.025599362328648567, 0.00033600421738810837, 0.020161530002951622, 0.0328560397028923, 0.005967485252767801, 0.01926594041287899, -0.023737071081995964, -0.026853062212467194, 0.03500824794173241, -0.01261197216808796, -0.009527659974992275, -0.0037404154427349567, -0.018619108945131302, 0.007110757753252983, 0.00047431080020032823, 0.010045692324638367, 0.0357535183429718, 0.022825399413704872, -0.016169650480151176, 0.0008656574063934386, -0.042107850313186646, -0.0548597127199173, -0.02471964620053768, 0.003725705435499549, 0.007661669049412012, 0.018211083486676216, 0.011036471463739872, 0.014412208460271358, 0.011582493782043457, -0.016493987292051315, -0.018972989171743393, 0.0013931189896538854, 0.018271328881382942, -0.03429539129137993, 0.003362505929544568, -0.005994666367769241, -0.00959770381450653, 0.0011164003517478704, -0.008555191569030285, 0.0070890821516513824, 0.018306443467736244, 0.010375361889600754, 0.01481429673731327, 0.03355877473950386, -0.022351214662194252, 0.022339550778269768, 0.03466729447245598, -0.005368864629417658, -0.001345735159702599, 0.04017797112464905, 0.00551758473739028, -0.2173909991979599, 0.04581783339381218, 0.0024528622161597013, 0.0022052039857953787, -0.04566375911235809, 0.022220319136977196, 0.026333946734666824, 0.017226774245500565, -0.022979149594902992, 0.016691217198967934, 0.004073971416801214, 0.002814855659380555, 0.016957614570856094, 0.07107777148485184, -0.007352392189204693, -0.038535021245479584, 0.017401786521077156, 0.0007929172134026885, 0.028942929580807686, -0.00036816971260122955, 0.11163684725761414, -0.01857011578977108, 0.06666223704814911, -0.008642716333270073, 0.06408526748418808, 0.04314403980970383, 0.0099580567330122, -0.024060484021902084, 0.0323871485888958, -0.005315718241035938, 0.01568787731230259, 1.024715220410144e-05, 0.032729893922805786, 0.0130087835714221, -0.009930921718478203, -0.01391188707202673, -0.01204149704426527, -0.0003884335746988654, -0.03863895684480667, 0.010849282145500183, -0.021411173045635223, -0.03841531649231911, -0.04881220683455467, 0.019240526482462883, 0.023810870945453644, 0.006863898131996393, 0.0276887696236372, -0.03131144493818283, -0.014472058974206448, 0.018088968470692635, -0.0007272192742675543, 0.02486952766776085, -0.020349860191345215, 0.04172771796584129, -0.03292231634259224, -0.02765854448080063, 0.02097843773663044, 0.03706378862261772, -0.007219500374048948, 0.0269289743155241, -0.01539407018572092, -0.0073793018236756325, -0.03981798142194748, 0.02146507427096367, -0.01352155115455389, 0.033688902854919434, -0.027500513941049576, 0.035745128989219666, 0.006595777813345194, 0.008930190466344357, -0.030803443863987923, -0.039435289800167084, 0.002231724327430129, 0.024395540356636047, 0.030874192714691162, 0.01107067335397005, 0.04071153700351715, 0.037772178649902344, -0.018142150714993477, -0.01915142685174942, 0.02152378484606743, -0.03557462990283966, -0.0076836287043988705, 0.011191720142960548, -0.0075014252215623856, 0.0061457292176783085, 0.004392640199512243, -0.004160741809755564, -0.005154121667146683, 0.007850119844079018, -0.025389060378074646, 0.027233676984906197, 0.018989071249961853, 0.018483715131878853, 0.029961425811052322, -0.01975196972489357, 0.0034955802839249372, 0.005008792504668236, 0.03675723448395729, -0.010142670013010502, 0.021865155547857285, 0.00777911813929677, 0.0028587228152900934, -0.00961610022932291, 0.004507306031882763, 0.007655666209757328, -0.0005477781523950398, -0.008482812903821468, -0.005262786988168955, -0.006313548423349857, 0.0025122882798314095, 0.013479162938892841, 0.0008452540496364236, -0.008268672972917557, -0.008921015076339245, -0.017137451097369194, 0.024556932970881462, -0.007083776406943798, 0.029889194294810295, -0.002155853668227792, 0.03748151659965515, -0.01219448633491993, -0.004321878310292959, -0.0049264151602983475, -0.0284781102091074, 0.04402875527739525, 0.029218927025794983, -0.0003226103144697845, 0.00029066193383187056, -0.010461417958140373, -0.01106881070882082, 0.03662734106183052, -0.030079064890742302, -0.02601679228246212, 0.02968227118253708, -0.0044954498298466206, 0.05667490139603615, 0.038063932210206985, 0.02532348409295082, -0.007635113783180714, -0.054219529032707214, 0.014239142648875713, 0.014415671117603779, 0.011645933613181114, 0.0026698473375290632, -0.018724042922258377, 0.04796440526843071, 0.0395481139421463, -0.02239905297756195, 0.03309261053800583, -0.019595924764871597, -0.015176158398389816, -0.020359372720122337, 0.024894483387470245, 0.048771608620882034, 0.01701756939291954, 0.009572185575962067, -0.00919422972947359, -0.016744127497076988, -0.02716502919793129, -0.01683136634528637, 0.017365729436278343, 0.013156753033399582, 0.003911449108272791, 0.010006570257246494, -0.009358896873891354, 0.012878834269940853, 0.01213813852518797, 0.06368658691644669, -0.043554555624723434, -0.044288270175457, 0.019254181534051895, -0.009986549615859985, 0.02766917645931244, 0.019831286743283272, -0.029171204194426537, -0.002535705454647541, 0.03869200870394707, -0.03844473883509636, -0.00308605725876987, -0.011283130384981632, 0.023060742765665054, -0.048825643956661224, -0.019820701330900192, -0.016311010345816612, 0.0089490981772542, -0.02160174958407879, -0.014793510548770428, 0.03102976270020008, 0.009803942404687405, -0.01743941381573677, -0.011344566009938717, -0.02397971972823143, -0.049813300371170044, -0.018691547214984894, 0.013998016715049744, 0.020314019173383713, -0.0012502047466114163, -0.0019372464157640934, -0.017373068258166313, 0.028032336384058, 0.024279536679387093, 0.0023303960915654898, 0.0020594189409166574, 0.0021133199334144592, 0.017577223479747772, -0.01910991221666336, 0.0020364008378237486, 0.017780354246497154, -0.07590895146131516, 0.023410191759467125, -0.006056459154933691, -0.017602011561393738, -0.0193230751901865, 0.003788337577134371, -0.0037353094667196274, -0.010924827307462692, 0.008800217881798744, -0.03159630671143532, 0.014250904321670532, -0.050038907676935196, 0.056669123470783234, 0.005622922442853451, -0.03370925411581993, -0.04483607038855553, -0.03787362575531006, 0.012450304813683033, -0.038621582090854645, 0.0012779751559719443, 0.012861674651503563, 0.005697268061339855, -0.01457883883267641, -0.0026998398825526237, -0.015384293161332607, -0.011793541721999645, 0.0028597386553883553, 0.010816897265613079, 0.016531474888324738, 0.00667218491435051, -0.004224835429340601, 0.006799850147217512, -0.029899507761001587, -0.025192003697156906, 0.013317358680069447, -0.0048745861276984215, -0.01709320768713951, 0.02421114407479763, 0.04741673544049263, 0.02977724000811577, -0.02186904288828373, -0.00438281474635005, -0.042575009167194366, -0.0429169163107872, -0.13304604589939117, -0.02594129368662834, 0.01572260633111, 0.0005540493875741959, 0.0002271509583806619, -0.010948210954666138, -0.006035648286342621, -0.04751060903072357, -0.017636697739362717, 0.042611368000507355, -0.020851634442806244, -0.0008972818613983691, -0.028208356350660324, -0.020075123757123947, -0.01168630737811327, 0.01829773373901844, 0.025807037949562073, -0.020768893882632256, -0.03445013612508774, -0.02243547886610031, 0.023673459887504578, 0.013543246313929558, 0.039774324744939804, 0.023217804729938507, -0.0017156670801341534, 0.006976937409490347, 0.02246670611202717, -0.003927747718989849, -0.026826949790120125, 0.06176983192563057, 0.04470497742295265, -0.021876979619264603, -0.01671697571873665, 0.03742693364620209, 0.01532019767910242, -0.06566041707992554, 0.016344336792826653, 0.03181944042444229, 0.05589604005217552, 0.015713311731815338, -0.01850336231291294, 0.0007490470306947827, 0.008403348736464977, 0.02733069844543934, -0.009132872335612774, 0.010553482919931412, 0.011145498603582382, -0.00894972588866949, -0.0002521691785659641, 0.005873187445104122, -0.019729817286133766, 0.016233354806900024, 0.03548296168446541, 0.025146478787064552, 0.0484381765127182, 0.002144219120964408, 0.03203701227903366, 0.006064014043658972, -0.0230820644646883, 0.008295091800391674, 0.024506697431206703, 0.044412072747945786, -0.02590726502239704, -0.004491962026804686, -0.00858828704804182, -0.03817484900355339, 0.01896330900490284, 0.016329925507307053, 0.009387639351189137, -0.0008489893516525626, -0.0229690819978714, -0.009065806865692139, 0.0019886582158505917, 0.018256274983286858, -0.005151602905243635, -0.0050378479063510895, -0.02216069959104061, -0.0023346126545220613, -0.039679013192653656, -0.0014204478356987238, -0.0008031260804273188, 0.003599343588575721, 0.0075701754540205, 0.012048834934830666, -0.01714339293539524, 0.028567278757691383, 0.021084096282720566, 0.029491830617189407, 0.03514658659696579, -0.0077706072479486465, 0.009647466242313385, -0.016020555049180984, 0.021023020148277283, 0.02811160869896412, 0.04720115661621094, 0.004733201116323471, -0.012755770236253738, 0.00884833000600338, 0.04124874249100685, 0.04268141835927963, -0.007543929386883974, 0.02284829318523407, -0.0023325993679463863, -0.010560406371951103, 0.03403284773230553, -0.010902280919253826, -0.003812785493209958, -0.02168215438723564, 0.030674874782562256, -0.01477818749845028, -0.014254949986934662, 0.03423457592725754, 0.06843890994787216, 0.006318838335573673, -0.0009734745835885406, -0.026457494124770164, 0.06373847275972366, 0.008520757779479027, -0.01902064122259617, -0.029911629855632782, 0.003961779642850161, -0.01983240619301796, 0.01927884668111801, -0.10859532654285431, 0.040863558650016785, -0.03436049446463585, 0.006976682227104902, -0.020971283316612244, 0.018356742337346077, 0.019181884825229645, 0.0024038669653236866, -0.008379681035876274, -0.01681409776210785, -0.03384656831622124, 0.019680889323353767, 0.0021284676622599363, -0.014127354137599468, 0.010429071262478828, -0.03903995454311371, -0.0023977721575647593, -0.019424572587013245, -0.025685105472803116, -0.018515387549996376, 0.008885552175343037, 0.021396229043602943, -0.03908727318048477, -0.026451053097844124, 0.0410730205476284, -0.023614484816789627, -0.005964791402220726, -0.0023529422469437122, -0.010650510899722576, -0.006606406532227993, 0.022733159363269806, 0.004038406535983086, 0.017530256882309914, 0.002314789919182658, 0.03923353925347328, -0.06088810786604881, -0.00527164526283741, -0.004469104111194611, 0.0017426643753424287, 0.014455708675086498, 0.0028602699749171734, -0.030816761776804924, 0.05748344212770462, -0.014835122972726822, -0.03842378407716751, 0.06279298663139343, 0.015866637229919434, -0.001298937713727355, -0.007616754155606031, -0.01587572693824768, -0.018170693889260292, 0.03295997157692909, 0.007576551754027605, 0.00988085102289915, 0.006035387050360441, 0.007287620101124048, 0.009575815871357918, 0.003029697807505727, 0.023742344230413437, 0.007164673414081335, -0.0032857502810657024, -0.01329439040273428, -0.017378536984324455, -0.026182876899838448, -0.0028808738570660353, 0.004806256387382746, 0.022348526865243912, -0.037018630653619766, 0.006920661777257919, 0.014456314034759998, -0.028866413980722427, -0.0008774391026236117, -0.026845097541809082, 0.012430327013134956, 0.01221863366663456, -0.003693058853968978, 0.03608867526054382, 0.059058211743831635, 0.022841209545731544, 0.013541904278099537, -0.03306673839688301, 0.025505708530545235, 0.036430396139621735, 0.00502338632941246, 0.0318639762699604, 0.0109039805829525, 0.02614949643611908, 0.035377174615859985, 0.017585597932338715, -0.016355842351913452, -0.014506937935948372, 0.028366651386022568, 0.026386085897684097, -0.0003489995142444968, -0.010659347288310528, 0.02508546970784664, -0.05031828582286835, -0.007203773129731417, -0.048896193504333496, 0.0013955372851341963, 0.03834427148103714, -0.009591073729097843, 0.00028664892306551337, 0.02021869458258152, 0.013686546124517918, 0.007293676026165485, -0.003278299467638135, 0.04215719923377037, -0.1032710075378418, 0.037057023495435715, 0.04478183388710022, 0.011888658627867699, 0.022871961817145348, 0.010411831550300121, 0.015734413638710976, -0.029889632016420364, 0.025693746283650398, 0.039458610117435455, -0.03364081308245659, -0.011717185378074646, -0.044194918125867844, -0.002493353560566902, 0.02698463574051857, 0.021751463413238525, 0.036281291395425797, -0.004428103566169739, 0.019937362521886826, 0.01587335392832756, 0.00897094514220953, -0.026231935247778893, -0.022902073338627815, -0.008572335354983807, 0.013034405186772346, 0.022929184138774872, 0.01777179352939129, 0.6693002581596375, -0.003652745857834816, -0.01991419680416584, -0.0031866293866187334, -0.0005242079496383667, 0.0017810059944167733, -0.023754188790917397, -0.03827995806932449, 0.004314340651035309, 0.01066952757537365, -0.01269934605807066, -0.027221912518143654, -0.0272783525288105, -0.0061208950355648994, 0.018172092735767365, 0.016020692884922028, -0.006023005116730928, -0.027113772928714752, -0.02859840914607048, -5.5018743296386674e-05, -0.01629967987537384, 0.029268333688378334, 0.002174271270632744, 0.058258622884750366, 0.009822741150856018, -0.06305354088544846, 0.032676033675670624, -0.006159550976008177, 0.0055778492242097855, -0.01178681943565607, 0.00748282577842474, -0.01873115822672844, 0.027654869481921196, -0.010546481236815453, -0.0035150968469679356, -0.007499105762690306, -0.031570639461278915, -0.01378653571009636, 0.031195808202028275, 0.0012345281429588795, 0.020074285566806793, 0.01601882465183735, -0.007986808195710182, -0.012707298621535301, -0.0005893284105695784, 0.04243215173482895, -0.011412647552788258, -0.010359133593738079, -0.02542262151837349, -0.011125962249934673, 0.009315493516623974, -0.0019672636408358812, 0.010807055048644543, -0.03465425223112106, -0.00838492438197136, -0.01572229340672493, -0.023003999143838882, -0.002225852105766535, 0.013703041709959507, -0.04991844296455383, 0.029770946130156517, -0.044531211256980896, 0.002691968809813261, -0.02040080353617668, 0.009426722303032875, 0.029168277978897095, 0.014672216027975082, 0.008052845485508442, 0.00848458893597126, 0.0360553041100502, -0.025770528241991997, -0.006298775784671307, 0.004566534887999296, 0.03559723496437073, 0.011005285196006298, 0.031520795077085495, -0.02828793227672577, 0.03890659287571907, -0.02122536487877369, -0.017724735662341118, 0.06592966616153717, 0.022148409858345985, 0.013433790765702724, 0.023225883021950722, 0.028540784493088722, 0.014324893243610859, -0.055401381105184555, 0.03768051415681839, -0.052587851881980896, -0.025651533156633377, 0.00044154582428745925, -0.027934597805142403, -0.012787730433046818, -0.019792890176177025, 0.04624786227941513, -0.010774028487503529, -0.017765840515494347, 0.0499267578125, -0.0034083109349012375, 0.011704624630510807, -0.019384846091270447, 0.012339436449110508, -0.02611391246318817, 0.0005685759242624044, -0.003420985536649823, -0.008084241300821304, -0.014753122813999653, -0.05080931633710861, -0.008545137010514736, 0.005460078362375498, 0.009275990538299084, 0.05403915047645569, -0.001209690934047103, 0.04672272130846977, 0.00012586184311658144, 0.003953925799578428, -0.01466074213385582, 0.026786398142576218, -0.009102863259613514, -0.002718305680900812, -0.002638432430103421, 0.001827026717364788, 0.010197574272751808, 0.0074658943340182304, 0.015208588913083076, 0.016280382871627808, -0.00923157762736082, -0.009727700613439083, 0.026002975180745125, -0.018284449353814125, -0.029910961166024208, -0.014810540713369846, -0.005136591382324696, 0.010810852982103825, -0.002605834975838661, 0.010105051100254059, 0.020834658294916153, 0.012113257311284542, -0.008264820091426373, 0.00046946218935772777, 0.024336598813533783, 0.023462997749447823, 0.019252654165029526, -0.021970324218273163, 0.02668192982673645, -0.004289514385163784, -0.03069346956908703, -0.06403359025716782, -0.031993214040994644, -0.03200491890311241, 0.013853468932211399, 0.016223972663283348, -0.01332102157175541, -0.05753146484494209, -0.005578640848398209, -0.012695973739027977, -0.0009861082071438432, -0.0009852611692622304, 0.0026454823091626167, 0.03371210768818855, -0.0069041005335748196, 0.014615408144891262, -0.02215675823390484, -0.0020158838015049696, 0.05466654151678085, 0.027110202237963676, 0.014827394858002663, 0.01913849636912346, -0.00902077741920948, 0.013785517774522305, 0.03781532123684883, 0.026301808655261993, -0.04086333140730858, -0.016011623665690422, 0.041118789464235306, -0.022087831050157547, 0.029033254832029343, 0.02268446609377861, -0.0018600196344777942, -0.04012612625956535], [-0.011708366684615612, 0.022387877106666565, 0.029302241280674934, 0.003075739834457636, -0.01808100752532482, 0.013984802179038525, -0.0018692552112042904, -0.005375845357775688, 0.023150021210312843, -0.0006015350809320807, -0.008914157748222351, 0.01999017223715782, 0.00910997111350298, -0.024182716384530067, -0.007897375151515007, 0.014346113428473473, -0.009786541573703289, 0.013670237734913826, -0.011299188248813152, 0.010791406966745853, -0.003371679689735174, -0.03148619458079338, 0.01686154305934906, -0.020778441801667213, -0.013736252672970295, 0.018769647926092148, 0.024704501032829285, -0.008066877722740173, 0.007516373414546251, 0.005068469326943159, -0.038635123521089554, 0.017549043521285057, 0.002907991176471114, -0.017567036673426628, 0.016627304255962372, 0.0031097109895199537, -0.0017797553446143866, 0.006406736094504595, 0.005042655393481255, -0.01036925334483385, -0.028102880343794823, 0.007349598221480846, 0.034166257828474045, 0.02364983968436718, -0.00327871972694993, -0.0064725689589977264, -0.034676358103752136, -0.02751206047832966, -0.0355302169919014, -0.020035574212670326, -0.002347333123907447, -0.0032711506355553865, -0.0036009422037750483, 0.026440707966685295, 0.0030353786423802376, -0.00033522737794555724, 0.0012485682964324951, -0.012896356172859669, 0.01752820424735546, -0.014104443602263927, 0.00042263412615284324, 0.014614651910960674, 0.004553703591227531, 0.0008401413797400892, -0.00021325489797163755, -0.0075071523897349834, 0.0013449991820380092, 0.05245997756719589, -0.0009081654134206474, 0.038282375782728195, 0.0031545746605843306, 0.010959235019981861, 0.010052969679236412, -0.02262425608932972, -0.011846601963043213, 0.017544355243444443, -0.0067893387749791145, -0.19930076599121094, 0.027636894956231117, -0.012423976324498653, 0.000862634798977524, -0.025977252051234245, 0.07478819787502289, 0.024630550295114517, -0.01204774435609579, -0.02081560157239437, -0.0008081859559752047, 0.015278493985533714, 0.00375936646014452, 0.00807566475123167, 0.020389024168252945, -0.005623648874461651, -0.004993777256458998, 0.015301610343158245, 0.015152913518249989, 0.0011945739388465881, -0.011715360917150974, 0.11906372755765915, -0.007236857898533344, 0.02070850320160389, 0.03352488577365875, 0.014837374910712242, 0.01982891373336315, 0.016209159046411514, 0.004199798684567213, -0.021814415231347084, -0.006458243355154991, 0.01805518940091133, 0.01733538880944252, 0.025896476581692696, -0.00720485532656312, -0.052369289100170135, 0.00154168251901865, -0.020119372755289078, 0.006116746459156275, -0.02470027096569538, -0.008935223333537579, -0.03199779987335205, -0.006722418125718832, -0.03298896551132202, -0.019476471468806267, 0.016281498596072197, -0.019172001630067825, -0.01586439087986946, -0.010055284015834332, -0.008141342550516129, -0.02929331175982952, -0.005673960782587528, 0.014252763241529465, -0.025121605023741722, 0.02649962529540062, -0.05046108737587929, 0.0066482312977313995, 0.015814432874321938, 0.0279172845184803, -0.01966382935643196, 0.004671212285757065, -0.010132923722267151, -0.04294579476118088, 0.0027934880927205086, 0.02258606255054474, 0.01839498057961464, 0.013263611122965813, -0.0019643385894596577, 0.027342135086655617, 0.009076436050236225, -0.013531956821680069, -0.03342253342270851, -0.014980174601078033, 0.02461838722229004, -0.007306748069822788, 0.036945972591638565, 0.027014948427677155, 0.020532697439193726, 0.0188456941395998, -0.005158463027328253, -0.005751190707087517, -0.0027447675820440054, -0.012624663300812244, 0.02256842888891697, -0.003997204825282097, -0.02037714794278145, 0.008407938294112682, -0.014654753729701042, -0.008025108836591244, 0.00996763538569212, -0.009172989055514336, -0.0018568128580227494, 0.001658452209085226, -0.015405390411615372, -0.006078793667256832, 0.0322103425860405, 0.0019860523752868176, -0.018330225721001625, -0.005372101441025734, 0.01648006960749626, -0.02515139989554882, 0.009281603619456291, 0.013803711161017418, 0.0033270190469920635, -0.010573621839284897, -0.019070977345108986, -0.01505898218601942, -0.023280953988432884, -0.0035032599698752165, 0.0071927634999156, 0.00692355539649725, -0.017873486503958702, -0.002697691088542342, 0.005402494687587023, 0.007899132557213306, -0.0011922909179702401, 0.010506282560527325, 0.002617501886561513, 0.0175801832228899, 0.022771475836634636, -0.0005530162015929818, 0.008141931146383286, 0.0027283350937068462, -0.0007069755811244249, 0.0023802085779607296, 0.02990308590233326, 0.031863920390605927, 0.01232903078198433, -0.022556165233254433, -0.028934208676218987, -0.01651860773563385, -0.005071194842457771, 0.009700515307486057, 0.0018161280313506722, -0.0005642733885906637, 0.028295617550611496, -0.008425720036029816, 0.023596249520778656, 0.04529399052262306, 0.02170996367931366, 0.013138039968907833, -0.00697698350995779, 0.001239887671545148, -0.002829235512763262, 0.015198481269180775, -0.02079358883202076, -0.0063134972006082535, 0.031887833029031754, 0.015597520396113396, -0.046774569898843765, 0.02868884615600109, -0.00025339392595924437, -0.016685791313648224, -0.00018638120673131198, 0.017862865701317787, 0.022505739703774452, -0.008402805775403976, -0.00253310170955956, 0.01716648042201996, 0.025218885391950607, -0.030954841524362564, 0.007655076216906309, -0.009697914123535156, 0.005484263878315687, -0.026813067495822906, -0.0014459757367148995, -0.020207371562719345, 0.010053509846329689, -0.02233138307929039, 0.023678941652178764, 0.001258986652828753, -0.05019793286919594, 0.007120287977159023, -0.03617870435118675, 0.011260486207902431, -0.017322562634944916, -0.02501869574189186, 0.0024032723158597946, 0.025125514715909958, -0.01769106276333332, 0.0025619668886065483, -0.0009456084226258099, 0.0466955304145813, -0.027661725878715515, -0.009072473272681236, 0.002240133937448263, -0.01850200816988945, 0.010557284578680992, -0.00453084334731102, 0.05104922875761986, -0.01129324734210968, -0.020108362659811974, -0.012301073409616947, -0.009901306591928005, -0.02813730016350746, -0.003926027100533247, -0.006635364610701799, 0.034713033586740494, 0.00432524224743247, -0.012266195379197598, -0.01291589718312025, 0.0029835375025868416, -0.001020699623040855, -0.007991431280970573, -0.02298222854733467, 0.012586433440446854, -0.0392899364233017, -0.053651466965675354, 0.00037340380367822945, -0.014493177644908428, -0.0728490799665451, 0.041197966784238815, 0.011716330423951149, -0.005781248211860657, -0.0038393517024815083, 0.011724809184670448, 8.275650907307863e-05, -0.02009313553571701, 0.017061056569218636, -0.008853374049067497, -0.0027582149486988783, -0.021939171478152275, 0.03147333487868309, 0.011553313583135605, 0.015441558323800564, 0.00031396246049553156, -0.015528260730206966, -0.01391455065459013, -0.004455894697457552, -0.0018066457705572248, 0.018366266041994095, 0.0044812592677772045, -0.0422334261238575, 0.0029213090892881155, -0.010180559009313583, -0.021112561225891113, -0.012931774370372295, -0.014718365855515003, -0.00926534179598093, 0.036841981112957, 0.005131496582180262, -0.0012369462056085467, -0.008756577968597412, -0.012194198556244373, 0.00930788367986679, 0.022150274366140366, 0.011942516081035137, 0.011709938757121563, 0.011798756197094917, 0.011884909123182297, -0.03358173742890358, -0.005986976437270641, -0.04635334759950638, -0.03786840662360191, -0.05440004542469978, -0.021689526736736298, 0.00820946041494608, 0.00942853931337595, 0.0009964756900444627, -0.01068069413304329, -0.014030594378709793, -0.014989805407822132, -0.018849890679121017, 0.0315357930958271, -0.009135270491242409, -0.016653552651405334, -0.016404053196310997, -0.005939801689237356, -0.01325183641165495, -0.006791919469833374, 0.0158196073025465, 0.00046476273564621806, -0.012343548238277435, -0.031979307532310486, 0.02138928882777691, 0.01567153073847294, 0.0014763331273570657, -0.001182463951408863, -0.006451089400798082, 0.0372655987739563, 0.023286208510398865, 0.007134913932532072, -0.02881496213376522, -0.013837968930602074, 0.04148384556174278, -0.0461650975048542, 7.7138225606177e-05, -0.0016202422557398677, -0.0129264360293746, -0.034747377038002014, -0.0029309848323464394, 0.011914252303540707, 0.003495542798191309, -0.007989158853888512, -0.027842620387673378, -0.020556246861815453, 0.019384974613785744, 0.012153067626059055, -0.006800510920584202, 0.01234907004982233, 0.005930325016379356, -5.405523188528605e-05, 0.016808515414595604, -0.009444795548915863, -0.014692570082843304, 0.007509703282266855, 0.01428715419024229, 0.01857721619307995, 0.013127068057656288, -0.00031497731106355786, 0.010390295647084713, -0.010443578474223614, -0.017062360420823097, 0.014138178899884224, 0.012230590917170048, 0.019562887027859688, -0.003097710432484746, -0.01032849121838808, -0.004183349199593067, -0.029522066935896873, 0.01458542887121439, 0.024088945239782333, 0.0015549040399491787, 0.017608297988772392, -0.008859671652317047, 0.001251323614269495, -0.030960075557231903, 0.0023132450878620148, -0.0002590414951555431, 0.009477944113314152, -0.02346828579902649, 0.012753657065331936, -0.015567229129374027, 0.024356817826628685, -0.0017527819145470858, 0.011928348802030087, 0.010571771301329136, 0.0012174496660009027, -0.009837605059146881, 0.02840208075940609, 0.011216742917895317, -0.009419522248208523, 0.024477414786815643, 0.0036873770877718925, 0.006499492563307285, -0.0016487535322085023, 0.015482136979699135, 0.022472834214568138, 0.009965542703866959, 0.004263249691575766, -0.00580933503806591, 0.010189572349190712, 0.03331499174237251, 0.02113957144320011, -0.01729702763259411, 0.002500548493117094, 0.00558723509311676, 0.01589314453303814, 0.02892744354903698, 0.006961794104427099, -0.0016079734778031707, -0.015606720000505447, -0.004827716387808323, -0.02182740718126297, 0.018226061016321182, -0.023511482402682304, 0.027108915150165558, 0.026558753103017807, 0.01830984838306904, -0.03849072381854057, -0.0020439568907022476, 0.0009843106381595135, -0.020369186997413635, 0.010101881809532642, -0.01768721081316471, 0.011558085680007935, 0.012025246396660805, -0.08971471339464188, 0.02148083597421646, -0.01918279007077217, 0.005906565580517054, 6.699190998915583e-05, -0.008453955873847008, 0.0008124883752316236, -0.0013767984928563237, 0.01900692656636238, 0.004706698004156351, -0.00947963073849678, 0.023972002789378166, 0.008419233374297619, 0.024028686806559563, 0.018831975758075714, -0.07704579830169678, 0.024494269862771034, -0.01564406231045723, -0.0009164020884782076, 0.014321963302791119, 0.013840724714100361, -0.0011588826309889555, -0.030952204018831253, 0.02592037059366703, 0.016268301755189896, 0.0063752480782568455, -0.01075279712677002, -0.017470037564635277, -0.01843332126736641, 0.009277521632611752, 0.017306016758084297, -0.004669926594942808, 0.0038898391649127007, 0.010038644075393677, 0.011002512648701668, -0.04748452454805374, -0.008155006915330887, 0.03431542590260506, -0.010647740215063095, 0.007834171876311302, 0.0010607520816847682, -0.004494208376854658, 0.025186773389577866, -0.043109651654958725, -0.023777559399604797, 0.019018694758415222, 0.03710096329450607, 0.009462478570640087, -0.010974559932947159, -0.02594844438135624, 0.025820782408118248, 0.025066815316677094, -0.002349532674998045, 0.024576690047979355, 0.012333239428699017, 0.007274379022419453, 0.0005117587861604989, 0.020581303164362907, 0.00489760609343648, 0.008028470911085606, -0.00866994634270668, -0.008065257221460342, 0.013609632849693298, -0.017540980130434036, -0.017184432595968246, 0.00434450525790453, 0.02973240427672863, 0.023534554988145828, 0.005186521913856268, 0.01819007843732834, -0.010710753500461578, 0.021665479987859726, -0.01524313259869814, 0.0006704282714053988, -0.014921160414814949, 0.009291494265198708, 0.023389726877212524, 0.02215440385043621, 0.02975967526435852, 0.02027881145477295, -0.01466760877519846, 7.378627196885645e-05, 0.01649867184460163, -0.005010997876524925, 0.01764053665101528, 0.01868453435599804, 0.01907723769545555, -0.014094667509198189, -0.004628760274499655, -0.0230353232473135, -0.015504087321460247, 0.015565275214612484, 0.023764463141560555, 0.00787858571857214, -0.003810495138168335, 0.00854011531919241, 0.015028159134089947, -0.01957574300467968, 0.20457832515239716, -0.014745676890015602, 0.03752356395125389, 0.010581501759588718, 0.004200391471385956, 0.02007373794913292, 0.0016273363726213574, 0.0004860105109401047, 0.010263307951390743, 0.04475152865052223, -0.04132290184497833, 0.01558014377951622, 0.04707862436771393, 0.016982879489660263, 0.026001593098044395, 0.015868164598941803, 0.023243479430675507, -0.0003945626667700708, 0.026042576879262924, 0.1045626625418663, -0.022345775738358498, -0.004123571794480085, -0.06341978162527084, 0.004487365949898958, 0.008811956271529198, 0.011621128767728806, 0.02287803404033184, 0.00963406078517437, -0.00453376118093729, 0.008147550746798515, 0.030866513028740883, -0.021215571090579033, 0.01020791195333004, 0.007466103881597519, 0.018155647441744804, -0.04182927682995796, 0.01110218558460474, 0.7736034989356995, 0.011357770301401615, -0.031440384685993195, -0.022210631519556046, 0.006465508136898279, 0.012154520489275455, -0.023728566244244576, -0.00012879887071903795, -0.0037470231764018536, -0.004097583703696728, 0.00852235034108162, -0.0024437573738396168, -5.078917820355855e-05, -0.012236620299518108, 0.005322948098182678, 0.020378926768898964, 0.015433422289788723, -0.026274872943758965, -0.02967035584151745, -0.0019187394063919783, -0.017542006447911263, -0.013067263178527355, 0.0171294417232275, 0.009891076944768429, -0.009877321310341358, -0.02789066545665264, 0.011738195084035397, -0.04310797154903412, 0.004466617479920387, -0.023888561874628067, 0.0038122094701975584, -0.008513501845300198, 0.013409724459052086, 0.005993639584630728, -0.02326911687850952, -0.025496643036603928, -0.038161735981702805, -0.013159841299057007, 0.010197672992944717, 0.01818447932600975, 0.016974307596683502, -0.00974725279957056, -0.011597046628594398, 0.017558421939611435, 0.000996834016405046, 0.018896039575338364, -0.025311265140771866, -0.0011490717297419906, -0.008428405970335007, -0.020415307953953743, 0.012491576373577118, -0.0020404329989105463, -0.01871304214000702, 0.01301551889628172, 0.015039985068142414, 0.013960015028715134, -0.007980460301041603, 0.01755729876458645, 0.010712435469031334, -0.01937813125550747, 0.009955271147191525, -0.03831744194030762, -0.0023186744656413794, -0.05363984405994415, 0.02076454646885395, 0.0237063467502594, -0.0005572050577029586, 0.04456070810556412, -0.02436034381389618, -0.007785055320709944, -0.012595411390066147, -0.018035953864455223, 0.011664358898997307, 0.039537351578474045, 0.0031091980636119843, 0.008205379359424114, -0.0017539948457852006, 0.03557056188583374, -0.013079717755317688, -0.012722120620310307, 0.0335254967212677, -0.012721344828605652, 0.020625080913305283, 0.01049614418298006, 0.012820924632251263, 0.031439825892448425, -0.007198818493634462, -0.03141685202717781, -0.032844822853803635, -0.02729668654501438, 0.008916818536818027, -0.01904807984828949, -0.016034645959734917, -0.008264787495136261, 0.030157707631587982, -0.0025957219768315554, -0.015436368994414806, 0.020873406901955605, 0.0015289641451090574, -0.004112626891583204, -0.019200891256332397, 0.01193244569003582, -0.023370841518044472, -0.00996678601950407, 0.005520207341760397, 0.013711761683225632, -0.021918227896094322, -0.033774737268686295, -0.0074331192299723625, 0.011981235817074776, 0.01179483626037836, 0.007690640166401863, 0.015365004539489746, 0.014380509965121746, 0.041383516043424606, 0.01048954576253891, -0.0010889980476349592, 0.008605113252997398, 0.007145370356738567, 0.0010289046913385391, 0.010095473378896713, 0.03015178069472313, 0.006090981885790825, 0.022776350378990173, 0.018564926460385323, 0.019851788878440857, 0.0038544354028999805, -0.004153143148869276, 0.02976919524371624, -0.0039059508126229048, -0.02442597784101963, -0.010514767840504646, 0.0069615282118320465, -0.02805984765291214, 0.007213718723505735, 0.02104724571108818, -0.001913278829306364, -0.00655590184032917, -0.015363925136625767, -0.01337362453341484, 0.007585297338664532, -0.014489416964352131, 0.012662157416343689, -0.029033971950411797, -0.0017891282914206386, -0.00647031981498003, -0.037664398550987244, -0.04423869028687477, -0.011321387253701687, -0.00918065756559372, 0.01625269278883934, -0.0012845097808167338, -0.005250066984444857, -0.04179656133055687, 0.008045431226491928, -0.019695399329066277, -0.001238690223544836, -0.0026710100937634706, 0.010878468863666058, 0.023922976106405258, -0.008193875662982464, 0.06116213649511337, 0.0039280857890844345, -0.009057044982910156, 0.011651931330561638, 0.010651974938809872, 0.008115900680422783, 0.011686289682984352, 0.012747031636536121, 0.013562801294028759, 0.005348870996385813, -0.004732422064989805, -0.020919479429721832, 0.006581152323633432, 0.009577600285410881, -0.01464113313704729, -0.010606717318296432, -0.0159520972520113, 0.02584041841328144, -0.03303214907646179]], "provenance": {"model_id": "StyleDistance/styledistance", "revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca", "manifest_sha256": "af95d2984503b01fe61bd40eaa9093cfc596be2c140a708c577a776821f2c3aa", "model_files": [{"path": "1_Pooling/config.json", "size_bytes": 296, "sha256": "1c8bc7bd750c5c20d8707f1c5c578f5d69bb3d1d5ebcf4b2fde5128de154ec1c", "upstream_etag": "9213783a33aa495a3c2d3791c7c5888d90607a4e", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "README.md", "size_bytes": 4713, "sha256": "cc17054b24e25b806c0150908367ba69e9b3ecf0ba0a2cfd0bdca9c594e9a52f", "upstream_etag": "43cae080d26f01cb6a5477ebdc19ac89b6db5732", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "config.json", "size_bytes": 678, "sha256": "b7bcc672f41211775356b1bb7b614453b91edc8425f58eb4ba2c71d6d8b1875a", "upstream_etag": "6a53568996dda58bf45eff852dca45188073cdd8", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "config_sentence_transformers.json", "size_bytes": 171, "sha256": "fe35b3fc2768f364c610e91305d45aa600eda73a2613635a47db835f9cc4fb74", "upstream_etag": "63ffff2feb02c50e2614f75b11aaff00553fc70c", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "datadreamer.json", "size_bytes": 4240, "sha256": "9939b9dc337417817fab8951012bd9c4b8bb58184c29d305ed9d271c18770a5c", "upstream_etag": "d4f4cc069653bc41c230787e59a73161986379dc", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "merges.txt", "size_bytes": 456318, "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5", "upstream_etag": "226b0752cac7789c48f0cb3ec53eda48b7be36cc", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "model.safetensors", "size_bytes": 249313784, "sha256": "6cd0908217a8110f068a1502f7e7157303bffbf965427f4a0b88a48b1b6544b1", "upstream_etag": "6cd0908217a8110f068a1502f7e7157303bffbf965427f4a0b88a48b1b6544b1", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "modules.json", "size_bytes": 229, "sha256": "8f4b264b80206c830bebbdcae377e137925650a433b689343a63bdc9b3145460", "upstream_etag": "f7640f94e81bb7f4f04daf1668850b38763a13d9", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "sentence_bert_config.json", "size_bytes": 53, "sha256": "ec8e29d6dcb61b611b7d3fdd2982c4524e6ad985959fa7194eacfb655a8d0d51", "upstream_etag": "f789d99277496b282d19020415c5ba9ca79ac875", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "special_tokens_map.json", "size_bytes": 280, "sha256": "06e405a36dfe4b9604f484f6a1e619af1a7f7d09e34a8555eb0b77b66318067f", "upstream_etag": "d5698132694f4f1bcff08fa7d937b1701812598e", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "tokenizer.json", "size_bytes": 2108643, "sha256": "6e6a06a662f8bce9823abd4c2cda07dcb7e07a84153a2ed190b8d901516b3162", "upstream_etag": "3f8fcc9a267618736514e5d493e1a6c9b4ff9154", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "tokenizer_config.json", "size_bytes": 1215, "sha256": "56f306b23b9233c94016f1b0b8aea3bc120676b378f70933908c9e62ab2daa36", "upstream_etag": "40ba7e2aca7217b59a4459a5344dce53c636a5b1", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "training_args.json", "size_bytes": 4476, "sha256": "f8526cc82acbfd631b55599e6ae928c2a826f7174b726b1796b518ae002fa2b5", "upstream_etag": "eb8526d396038821c83bc65373a029a4f1a15e5e", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}, {"path": "vocab.json", "size_bytes": 798293, "sha256": "ed19656ea1707df69134c4af35c8ceda2cc9860bf2c3495026153a133670ab5e", "upstream_etag": "4ebe4bb3f3114daf2e4cc349f24873a1175a35d7", "upstream_revision": "b7df5f0b0480773c097ba3121d83ca32b71015ca"}], "runtime": {"python": "3.12.8", "platform": "macOS-26.6.2-arm64-arm-64bit", "torch": "2.13.0", "transformers": "5.9.0", "huggingface_hub": "1.16.4", "tokenizers": "0.22.2", "safetensors": "0.7.0", "numpy": "2.3.2", "parameter_count": 124645632, "parameter_dtype": "torch.float32", "backbone_attention": "sdpa", "torch_num_threads": 2, "settings": {"content_tokens": 510, "sequence_length": 512, "max_batch_windows": 8, "cpu_threads": 2, "device": "cpu", "dtype": "float32"}, "loading_info": {"missing_keys": [], "unexpected_keys": [], "mismatched_keys": [], "error_msgs": []}, "custom_remote_code": false, "pooling_accumulation": "float32 window mean; float64 token-count-weighted passage sum; final float32 unit vector"}, "encoder_source_sha256": "a76bed479b06158d3fd7e8225a1e49be3d8133196733217a81240e81618934d0"}, "cross_runtime_max_abs_tolerance": 0.0001, "note": "Synthetic transfer check only; not competition rows, training features, or model selection."}')
expected_model_manifest_sha256='af95d2984503b01fe61bd40eaa9093cfc596be2c140a708c577a776821f2c3aa'


In [ ]:
import gc
import logging
import time
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from name_that_llm.baseline import LABELS, load_data, validate_submission, runtime_versions
from name_that_llm.encoder_v7 import FrozenStyleDistanceEncoder, sha256_file
from name_that_llm.classifier_v7 import fit_classifier, model_evidence, combine_evidence
from name_that_llm.classifier_v5 import incumbent_evidence
from name_that_llm.stacking_v4 import NestedStacker, evidence_features

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
started=time.perf_counter()
data_paths=[Path('/kaggle/input/competitions/name-that-llm'),Path('/kaggle/input/name-that-llm')]
if os.environ.get('NAME_THAT_LLM_DATA_DIR'):
    data_paths=[Path(os.environ['NAME_THAT_LLM_DATA_DIR'])]
data_dir=next((p for p in data_paths if (p/'train.csv').is_file() and (p/'test.csv').is_file()),None)
if data_dir is None:
    raise FileNotFoundError('Attach the competition data')
manifest_candidates=list(Path('/kaggle/input').rglob('manifest.json'))
if os.environ.get('NAME_THAT_LLM_MODEL_DIR'):
    manifest_candidates=[Path(os.environ['NAME_THAT_LLM_MODEL_DIR'])/'manifest.json']
matching=[p for p in manifest_candidates if sha256_file(p)==expected_model_manifest_sha256]
if len(matching)!=1:
    raise ValueError('Attach exactly one verified StyleDistance model package')
train,test=load_data(data_dir)
encoder=FrozenStyleDistanceEncoder(matching[0].parent)
golden_vectors=encoder.encode(golden['texts'])
golden_error=float(np.max(np.abs(golden_vectors-np.asarray(golden['embeddings']))))
if golden_error>golden['cross_runtime_max_abs_tolerance']:
    raise ValueError(f'Cross-runtime encoder difference exceeds tolerance: {golden_error}')
print(json.dumps({'versions':runtime_versions(),'encoder_runtime':encoder.runtime,
                  'golden_max_abs_difference':golden_error,'train_rows':len(train),'test_rows':len(test)},indent=2))
def progress(record):
    if record['batch']%50==0:
        print(json.dumps(record),flush=True)
train_vectors=encoder.encode(train['text'].tolist(),progress=progress)
train_report=encoder.report
test_vectors=encoder.encode(test['text'].tolist(),progress=progress)
encoder_audit={'provenance':encoder.provenance(),'train':train_report,'test':encoder.report,'golden_error':golden_error}
del encoder
gc.collect()
with threadpool_limits(limits=2):
    linear=fit_classifier(train_vectors,train['model'])
    embedding_scores=model_evidence(linear,test_vectors)
    stacker=None
    old=np.zeros_like(embedding_scores)
    if recipe['v4_weight']:
        stacker=NestedStacker(threads=2).fit(train['text'],train['model'],groups=train['prompt_id'])
        base=stacker.base_evidence(test['text'])
        old=incumbent_evidence(stacker.meta_.predict_proba(evidence_features(base)),base)
    scores=combine_evidence(embedding_scores,old,recipe)
    submission=pd.DataFrame({'id':test['id'].to_numpy(),'model':np.asarray(LABELS)[scores.argmax(axis=1)]})
validate_submission(test,submission)
submission.to_parquet('submission.parquet',index=False)
validate_submission(test,pd.read_parquet('submission.parquet'))
audit={'encoder':encoder_audit,'stacker':stacker.fit_oof_audit_ if stacker is not None else None,'recipe':recipe,
       'runtime_source_sha256':{name:sha256_file(package/name) for name in sources},
       'seconds':time.perf_counter()-started,'submission_rows':len(submission)}
Path('training_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
print(json.dumps({'verified_submission_rows':len(submission),'seconds':audit['seconds']}))

In [ ]:
(package/'assignment_v8.py').write_text('"""Transductive prompt recovery and one-model-per-prompt assignment (v8).\n\nTraining data holds 150 prompts, each answered once by every one of the 18 models.\nV8 assumes the hidden test set shares that structure: it recovers prompt groups from\npassage text alone and assigns models within each group by minimum-cost matching.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom sklearn.cluster import AgglomerativeClustering\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\nGROUP_SIZE = 18\nASSIGNMENT_PARAMETERS = {"tfidf": {"sublinear_tf": True, "stop_words": "english", "min_df": 2},\n                         "linkage": "average", "metric": "cosine", "cap": GROUP_SIZE,\n                         "n_clusters": "round(rows / 18)", "cost": "-log softmax(evidence)"}\n\n\ndef _cluster(distances, n_clusters):\n    if n_clusters < 2:\n        return np.zeros(len(distances), dtype=int)\n    return AgglomerativeClustering(n_clusters=n_clusters, metric="precomputed",\n                                   linkage="average").fit_predict(distances)\n\n\ndef recover_prompts(texts, cap=GROUP_SIZE):\n    """Cluster passages by topic, then split any cluster larger than one full prompt."""\n    texts = list(texts)\n    n_clusters = round(len(texts) / cap)\n    if n_clusters < 2:\n        return None\n    x = TfidfVectorizer(**ASSIGNMENT_PARAMETERS["tfidf"]).fit_transform(texts)\n    distances = np.clip(1.0 - (x @ x.T).toarray(), 0.0, 2.0)\n    np.fill_diagonal(distances, 0.0)\n    labels = _cluster(distances, n_clusters)\n    queue, nxt = list(np.unique(labels)), labels.max() + 1\n    while queue:\n        rows = np.where(labels == queue.pop())[0]\n        if len(rows) <= cap:\n            continue\n        parts = _cluster(distances[np.ix_(rows, rows)], int(np.ceil(len(rows) / cap)))\n        for k in range(1, parts.max() + 1):\n            labels[rows[parts == k]] = nxt\n            queue.append(nxt)\n            nxt += 1\n        queue.append(labels[rows[parts == 0][0]])\n    return labels\n\n\ndef assign_within_prompts(scores, groups):\n    """Return class indices; within each recovered group no class is used twice."""\n    scores = np.asarray(scores, dtype=float)\n    shifted = scores - scores.max(axis=1, keepdims=True)\n    log_p = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))\n    prediction = log_p.argmax(axis=1)\n    if groups is None:\n        return prediction\n    for group in np.unique(groups):\n        rows = np.where(groups == group)[0]\n        if len(rows) > scores.shape[1]:\n            raise ValueError("A recovered prompt group exceeds the number of models")\n        _, cols = linear_sum_assignment(-log_p[rows])\n        prediction[rows] = cols\n    return prediction\n')
from name_that_llm.assignment_v8 import ASSIGNMENT_PARAMETERS, assign_within_prompts, recover_prompts
v8_started=time.perf_counter()
groups=recover_prompts(test['text'].fillna('').astype(str).tolist())
prediction=assign_within_prompts(scores,groups)
v7_prediction=scores.argmax(axis=1)
submission=pd.DataFrame({'id':test['id'].to_numpy(),'model':np.asarray(LABELS)[prediction]})
validate_submission(test,submission)
submission.to_parquet('submission.parquet',index=False)
validate_submission(test,pd.read_parquet('submission.parquet'))
sizes=np.bincount(groups) if groups is not None else np.array([])
v8_audit={'parameters':ASSIGNMENT_PARAMETERS,'test_rows':len(test),'groups':int(len(sizes)),
          'group_size_counts':{int(k):int(v) for k,v in zip(*np.unique(sizes,return_counts=True))},
          'changed_from_v7':int((prediction!=v7_prediction).sum()),
          'assignment_sha256':sha256_file(package/'assignment_v8.py'),
          'seconds':time.perf_counter()-v8_started,
          'label_counts':submission['model'].value_counts().to_dict()}
audit['v8']=v8_audit
Path('training_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
print(json.dumps(v8_audit,indent=2),flush=True)